# SignSage — Full iSign Sign→English Training Notebook — Kaggle-ready final

In Kaggle full mode this notebook uses the complete cleaned iSign v1.1 corpus: 123,779 usable rows, split leakage-free by `video_id`. It does not silently fall back to a 10k training subset.

The remote 158+ GiB pose archive is handled with ZIP-member locality batching, bounded HTTP range caches, UID-only Dataset access, a bounded scratch cache, AMP, and memory-safe evaluation.

A real throughput benchmark and GPU benchmark determine whether at least one complete full-corpus epoch fits the declared session budget. A mandatory tiny-overfit test runs before full training.

This notebook cannot guarantee model quality in advance; it guarantees that the experiment either runs under the measured budget or stops before wasting the expensive full-corpus training phase.

## 0. Prerequisites

### Kaggle mode

The notebook is optimized for Kaggle GPU execution.

The large iSign pose archive is **not fully extracted into Kaggle storage**. Instead, the notebook uses the Hugging Face-backed multipart ZIP64 reader already implemented below and fetches only the pose members required by the selected samples.

Expected remote resources:

- `iSign_v1.1.csv`
- multipart iSign pose archive:
  - `iSign-poses_v1.1_part_aa`
  - `iSign-poses_v1.1_part_ab`
  - `iSign-poses_v1.1_part_ac`
  - `iSign-poses_v1.1_part_ad`
- Kaggle Secret: `HF_TOKEN`, with read access to `Exploration-Lab/iSign`

### Local mode

The same notebook can still run on the local storage machine when `SIGNSAGE_DATA_DIR` points to the iSign root directory containing:

```text
iSign/
├── iSign_v1.1.csv
└── poses/
    └── iSign-poses_v1.1/
        └── *.pose
```

### Execution order

Run the notebook **top to bottom**. The notebook is deliberately organized as:

```text
Environment
    ↓
Dataset verification
    ↓
Data audit
    ↓
Leak-safe video-level split
    ↓
Tokenizer / vocabulary
    ↓
Pose normalization + cache
    ↓
DataLoader smoke check
    ↓
Sign → Text models
    ↓
Sign → Text evaluation
    ↓
Text → Sign baselines / generator
    ↓
Text → Sign evaluation
    ↓
Visualization + round-trip
    ↓
Latency
    ↓
Knowledge repository finalization
    ↓
Research GO / NO-GO report
    ↓
Export manifest
```


## 0a. Performance & memory strategy

The Kaggle primary run uses all 123,779 cleaned iSign rows. There is no 10,000-row training cap. The large pose archive remains remote.

Training, validation and test batches are grouped by ZIP member header offset so samples inside a batch are physically local in the remote archive. Training shuffles the order of those local batches; evaluation keeps the local batches sequential.

Kaggle does **not** build a multi-GB normalized-pose disk cache for the full corpus. The remote locality optimization is the primary I/O strategy; only a bounded per-worker in-memory LRU is used. Local mode may use the regenerable disk cache. This avoids the previous host-disk/cache growth failure.

RAM is constrained with two workers, no persistent workers, prefetch=1, a bounded in-process cache, and single-worker evaluation. AMP and non-blocking transfers are used for GPU execution.

The notebook measures actual DataLoader and GPU time before full training. Epoch count is resolved from those measurements and the declared Kaggle session budget. If even one complete full-corpus epoch is projected not to fit, training stops before wasting the session.

In [1]:
# ============================================================
# SignSage configuration + environment detection
# ============================================================
import os, gc, io, json, random, time, warnings, zipfile, math, shutil
from pathlib import Path
from collections import Counter
from functools import lru_cache
from contextlib import nullcontext
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler, Subset

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type=="cuda": torch.cuda.manual_seed_all(SEED)
IS_KAGGLE=Path("/kaggle").exists() or bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
DATA_MODE="kaggle" if IS_KAGGLE else "local"
RUN_MODE="full" if IS_KAGGLE else "smoke"

RUN_PRIMARY_S2T=True
RUN_GRU_BASELINE=False
RUN_NOFACE_ABLATION=False
RUN_TEXT_TO_POSE=False
RUN_RETRIEVAL_BASELINE=True

if RUN_MODE=="smoke":
    TRAIN_LIMIT,VAL_LIMIT,TEST_LIMIT=300,60,60
else:
    TRAIN_LIMIT=None
    VAL_LIMIT=None
    TEST_LIMIT=None

N_EPOCHS_TRANSFORMER=None
MIN_EPOCHS_TRANSFORMER=3  # RESTORED: 1 reproduces the diagnosed collapse config
MAX_EPOCHS_TRANSFORMER=4  # RESTORED: was silently dropped to 2
TRAIN_TIME_BUDGET_HOURS=9.0 if RUN_MODE=="full" else 0.25
SESSION_SAFETY_MARGIN_MINUTES=90 if RUN_MODE=="full" else 10

RESUME_FROM_RECOVERY=False
RUN_TINY_OVERFIT_TEST=True
TINY_OVERFIT_N=64
TINY_OVERFIT_STEPS=200
TINY_OVERFIT_MIN_EXACT_MATCH=0.50
TINY_OVERFIT_MAX_LOSS=1.50

TRANSFORMER_PEAK_LR=2e-4
TRANSFORMER_WARMUP_FRACTION=0.06
TRANSFORMER_MIN_LR_RATIO=0.1
N_EPOCHS_GRU=0
N_EPOCHS_POSE=0

MAX_TOKENS=96
TARGET_VOCAB_SIZE=16000
MAX_FRAMES=160
POSE_OUTPUT_FRAMES=60

BATCH_SIZE=64 if DEVICE.type=="cuda" else 4
NUM_WORKERS=2 if IS_KAGGLE else 0
CACHE_SIZE=64
PIN_MEMORY=DEVICE.type=="cuda"
PERSISTENT_WORKERS=False
PREFETCH_FACTOR=1
GRAD_ACCUM_STEPS=1 if DEVICE.type=="cuda" else 4
USE_AMP=DEVICE.type=="cuda"
USE_BF16_AUTOCAST_CPU=False

REMOTE_BATCH_LOCALITY=True
REMOTE_CHUNK_SIZE_MB=32
REMOTE_CACHE_CHUNKS=4
PERF_PREFLIGHT_BATCHES=8
PERF_WARMUP_BATCHES=2
PERF_MAX_SECONDS_PER_BATCH=8.0
GPU_BENCHMARK_BATCHES=5

USE_POSE_DISK_CACHE=(False if IS_KAGGLE else True)
PREWARM_CACHE=False
USE_LLM_GLOSS=False
DTW_EVAL_N=200
RETRIEVAL_MAX_SAMPLES=3000
BEAM_WIDTH=4
REPETITION_PENALTY=1.15
NO_REPEAT_NGRAM_SIZE=3
MIN_OUTPUT_TOKENS=1
MAX_DECODE_TOKENS=MAX_TOKENS-2
EVAL_NUM_WORKERS=0

def normalize_text(text): return " ".join(str(text).lower().strip().split())

def autocast_ctx():
    if USE_AMP: return torch.autocast(device_type="cuda",dtype=torch.float16)
    if USE_BF16_AUTOCAST_CPU and DEVICE.type=="cpu":
        try: return torch.autocast(device_type="cpu",dtype=torch.bfloat16)
        except Exception: pass
    return nullcontext()
try: scaler=torch.amp.GradScaler("cuda",enabled=USE_AMP)
except TypeError: scaler=torch.cuda.amp.GradScaler(enabled=USE_AMP)

def check_memory():
    try:
        import psutil
        mem=psutil.virtual_memory(); sw=psutil.swap_memory()
        print(f"RAM: {mem.used/1024**3:.1f}/{mem.total/1024**3:.1f} GB ({mem.percent}%)")
        print(f"Swap: {sw.used/1024**3:.1f}/{sw.total/1024**3:.1f} GB ({sw.percent}%)")
        if DEVICE.type=="cuda":
            for i in range(torch.cuda.device_count()):
                p=torch.cuda.get_device_properties(i)
                print(f"GPU{i}: alloc={torch.cuda.memory_allocated(i)/1024**3:.2f} GB reserved={torch.cuda.memory_reserved(i)/1024**3:.2f} GB total={p.total_memory/1024**3:.2f} GB")
    except Exception as exc: print("Memory check unavailable:",type(exc).__name__)

def maybe_check_memory(): check_memory()

print("=== SignSage configuration ===")
print("Environment:",DATA_MODE,"| device:",DEVICE)
print("FULL-CORPUS REQUEST:",TRAIN_LIMIT is None and VAL_LIMIT is None and TEST_LIMIT is None)
print("Batch:",BATCH_SIZE,"workers:",NUM_WORKERS,"persistent:",PERSISTENT_WORKERS)
print("Archive locality:",REMOTE_BATCH_LOCALITY,"chunk MB:",REMOTE_CHUNK_SIZE_MB)
print("Training budget:",TRAIN_TIME_BUDGET_HOURS,"h")


=== SignSage configuration ===
Environment: kaggle | device: cuda
FULL-CORPUS REQUEST: True
Batch: 64 workers: 2 persistent: False
Archive locality: True chunk MB: 32
Training budget: 9.0 h


In [2]:
# 0b. Project knowledge repository (SQLite + JSON)
#
# Kaggle is the experiment workspace. This repository is the structured record
# that will later be copied into the main SignSage project for backend use.
#
# SQLite = queryable historical/research record
# JSON    = portable run manifest / configuration snapshot

import sqlite3
from pathlib import Path
from datetime import datetime, timezone

OUTPUT_DIR = Path("/kaggle/working/signsage_outputs") if DATA_MODE == "kaggle" else Path.cwd() / "signsage_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

REPORT_DIR = OUTPUT_DIR / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

KNOWLEDGE_DB_PATH = OUTPUT_DIR / "signsage_knowledge.db"
KNOWLEDGE_JSON_PATH = OUTPUT_DIR / "signsage_knowledge_manifest.json"

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_STARTED_AT = datetime.now(timezone.utc).isoformat()

def _sql_connect():
    conn = sqlite3.connect(KNOWLEDGE_DB_PATH)
    conn.execute("PRAGMA foreign_keys = ON")
    return conn

def init_knowledge_repository():
    conn = _sql_connect()
    conn.executescript("""
    CREATE TABLE IF NOT EXISTS runs (
        run_id TEXT PRIMARY KEY,
        started_at TEXT NOT NULL,
        completed_at TEXT,
        data_mode TEXT NOT NULL,
        run_mode TEXT NOT NULL,
        device TEXT NOT NULL,
        gpu_count INTEGER NOT NULL,
        seed INTEGER NOT NULL
    );

    CREATE TABLE IF NOT EXISTS dataset (
        run_id TEXT PRIMARY KEY,
        dataset_name TEXT NOT NULL,
        csv_path TEXT,
        csv_rows INTEGER,
        unique_video_ids INTEGER,
        pose_source TEXT,
        selected_rows INTEGER,
        FOREIGN KEY(run_id) REFERENCES runs(run_id)
    );

    CREATE TABLE IF NOT EXISTS splits (
        run_id TEXT PRIMARY KEY,
        train_rows INTEGER,
        val_rows INTEGER,
        test_rows INTEGER,
        train_videos INTEGER,
        val_videos INTEGER,
        test_videos INTEGER,
        leakage_free INTEGER,
        FOREIGN KEY(run_id) REFERENCES runs(run_id)
    );

    CREATE TABLE IF NOT EXISTS experiments (
        experiment_id INTEGER PRIMARY KEY AUTOINCREMENT,
        run_id TEXT NOT NULL,
        name TEXT NOT NULL,
        family TEXT NOT NULL,
        input_features TEXT,
        epochs INTEGER,
        batch_size INTEGER,
        learning_rate REAL,
        weight_decay REAL,
        checkpoint_path TEXT,
        best_validation_loss REAL,
        FOREIGN KEY(run_id) REFERENCES runs(run_id)
    );

    CREATE TABLE IF NOT EXISTS metrics (
        metric_id INTEGER PRIMARY KEY AUTOINCREMENT,
        run_id TEXT NOT NULL,
        experiment_name TEXT NOT NULL,
        metric_name TEXT NOT NULL,
        metric_value REAL,
        split_name TEXT,
        FOREIGN KEY(run_id) REFERENCES runs(run_id)
    );

    CREATE TABLE IF NOT EXISTS artifacts (
        artifact_id INTEGER PRIMARY KEY AUTOINCREMENT,
        run_id TEXT NOT NULL,
        artifact_name TEXT NOT NULL,
        artifact_type TEXT NOT NULL,
        path TEXT NOT NULL,
        description TEXT,
        FOREIGN KEY(run_id) REFERENCES runs(run_id)
    );

    CREATE TABLE IF NOT EXISTS observations (
        observation_id INTEGER PRIMARY KEY AUTOINCREMENT,
        run_id TEXT NOT NULL,
        key TEXT NOT NULL,
        value TEXT,
        FOREIGN KEY(run_id) REFERENCES runs(run_id)
    );
    """)
    conn.execute(
        """INSERT OR REPLACE INTO runs
           (run_id, started_at, completed_at, data_mode, run_mode, device, gpu_count, seed)
           VALUES (?, ?, ?, ?, ?, ?, ?, ?)""",
        (RUN_ID, RUN_STARTED_AT, None, DATA_MODE, RUN_MODE, str(DEVICE),
         int(torch.cuda.device_count()), int(SEED)),
    )
    conn.commit()
    conn.close()

def knowledge_insert(table, columns, values, delete_keys=None):
    """Insert one row, optionally deleting any existing row(s) that match
    `delete_keys` first. Tables whose primary key is `run_id` (dataset,
    splits) don't need this -- INSERT OR REPLACE already keys off it -- but
    tables with an autoincrement id (experiments) have no natural
    uniqueness, so without an explicit delete, re-running a training cell
    silently appends a duplicate row instead of replacing it."""
    conn = _sql_connect()
    if delete_keys:
        where_clause = " AND ".join(f"{col} IS ?" for col in delete_keys)
        conn.execute(f"DELETE FROM {table} WHERE {where_clause}", tuple(delete_keys.values()))
    placeholders = ", ".join(["?"] * len(values))
    conn.execute(
        f"INSERT OR REPLACE INTO {table} ({', '.join(columns)}) VALUES ({placeholders})",
        tuple(values),
    )
    conn.commit()
    conn.close()

def knowledge_metric(experiment_name, metric_name, value, split_name=None):
    conn = _sql_connect()
    # Idempotent: replace the matching (run, experiment, metric, split) row
    # instead of appending a duplicate on re-run. `IS ?` (not `= ?`) so this
    # also matches correctly when split_name is NULL.
    conn.execute(
        """DELETE FROM metrics
           WHERE run_id = ? AND experiment_name = ? AND metric_name = ? AND split_name IS ?""",
        (RUN_ID, experiment_name, metric_name, split_name),
    )
    conn.execute(
        """INSERT INTO metrics
           (run_id, experiment_name, metric_name, metric_value, split_name)
           VALUES (?, ?, ?, ?, ?)""",
        (RUN_ID, experiment_name, metric_name, float(value), split_name),
    )
    conn.commit()
    conn.close()

def knowledge_artifact(name, artifact_type, path, description=""):
    conn = _sql_connect()
    # Idempotent: one row per (run_id, artifact_name); re-running a cell
    # updates the existing artifact record instead of duplicating it.
    conn.execute(
        "DELETE FROM artifacts WHERE run_id = ? AND artifact_name = ?",
        (RUN_ID, name),
    )
    conn.execute(
        """INSERT INTO artifacts
           (run_id, artifact_name, artifact_type, path, description)
           VALUES (?, ?, ?, ?, ?)""",
        (RUN_ID, name, artifact_type, str(path), description),
    )
    conn.commit()
    conn.close()

def knowledge_observation(key, value):
    # Deliberately append-only (unlike the writers above): observations such
    # as "go_no_go" are log-style notes, not a single fact per key, so a
    # re-run recording another entry is the correct behavior here.
    conn = _sql_connect()
    conn.execute(
        "INSERT INTO observations (run_id, key, value) VALUES (?, ?, ?)",
        (RUN_ID, str(key), json.dumps(value, default=str)),
    )
    conn.commit()
    conn.close()

init_knowledge_repository()

print("Knowledge DB :", KNOWLEDGE_DB_PATH)
print("Knowledge JSON:", KNOWLEDGE_JSON_PATH)
print("Run ID       :", RUN_ID)


Knowledge DB : /kaggle/working/signsage_outputs/signsage_knowledge.db
Knowledge JSON: /kaggle/working/signsage_outputs/signsage_knowledge_manifest.json
Run ID       : 20260926T040300Z


In [3]:
# Kaggle-only dependency installation.
# Kaggle's default GPU image does not include pose-format, jiwer, sacrebleu,
# or rouge-score, and its bundled huggingface_hub can be too old for
# `hf_hub_download(..., local_dir=...)` to behave reliably. Installing these
# here -- before anything below imports them -- is what actually lets the
# rest of this notebook run top-to-bottom on a fresh Kaggle kernel; without
# it, the very next cell's `from pose_format import Pose` raises
# ModuleNotFoundError before the dataset is even touched.
#
# Local-machine venvs are expected to already have these installed (see
# Section 0 / the commented-out install line in the original local notebook),
# so this deliberately does nothing outside Kaggle mode.
if IS_KAGGLE:
    import subprocess
    import sys as _sys

    _KAGGLE_ONLY_PACKAGES = [
        "pose-format",
        "jiwer",
        "sacrebleu",
        "rouge-score",
        "sentencepiece",
        "huggingface_hub>=0.24.0",
        "requests",  # used directly by the remote ZIP64 backend (cell below);
                     # declared explicitly rather than assumed from the base image.
    ]
    print("Kaggle mode: installing packages not present in the default image:")
    print(" ", ", ".join(_KAGGLE_ONLY_PACKAGES))
    subprocess.run(
        [_sys.executable, "-m", "pip", "install", "-q", "--no-input", *_KAGGLE_ONLY_PACKAGES],
        check=True,
    )
    print("Done. (Safe to re-run this cell; pip install is idempotent.)")
else:
    print("Local mode: skipping Kaggle-only package installation "
          "(the local venv is expected to already have these -- see Section 0).")


Kaggle mode: installing packages not present in the default image:
  pose-format, jiwer, sacrebleu, rouge-score, sentencepiece, huggingface_hub>=0.24.0, requests
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 58.7 MB/s eta 0:00:00
Done. (Safe to re-run this cell; pip install is idempotent.)


In [4]:
# ============================================================
# 1. Verify and clean the iSign dataset before training
# ============================================================

if "DATA_MODE" not in globals():
    raise RuntimeError(
        "Run the configuration cell first."
    )

# ------------------------------------------------------------
# Resolve a stable dataset root.
#
# Kaggle:
#   /kaggle/working is used only for the downloaded CSV and
#   small metadata artifacts.
#
# The large pose archive is NOT extracted here. Individual
# .pose members are streamed from the remote multipart ZIP64
# source later.
#
# Local:
#   Respect DATA_ROOT/DATA_DIR if already defined.
# ------------------------------------------------------------

if DATA_MODE == "kaggle":

    if "DATA_ROOT" not in globals():
        DATA_ROOT = Path("/kaggle/working/isign")

    DATA_ROOT = Path(DATA_ROOT)
    DATA_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

else:

    if "DATA_ROOT" not in globals():

        if "DATA_DIR" in globals():
            DATA_ROOT = Path(DATA_DIR)
        else:
            DATA_ROOT = Path("./data")

    DATA_ROOT = Path(DATA_ROOT)
    DATA_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

# Keep DATA_DIR as a compatibility alias because older notebook
# cells may still reference it.
DATA_DIR = DATA_ROOT

print(f"Dataset root: {DATA_ROOT}")
print(f"Data mode: {DATA_MODE}")


# ------------------------------------------------------------
# Locate / download the CSV
# ------------------------------------------------------------

if DATA_MODE == "local":

    if "CSV_PATH" not in globals():
        raise RuntimeError(
            "Local mode requires CSV_PATH to be defined "
            "by the configuration cell."
        )

    CSV_PATH = Path(CSV_PATH)

    if not CSV_PATH.is_file():
        raise FileNotFoundError(
            f"Missing local iSign CSV: {CSV_PATH}"
        )

    if "POSE_DIR" not in globals():
        raise RuntimeError(
            "Local mode requires POSE_DIR to be defined "
            "by the configuration cell."
        )

    POSE_DIR = Path(POSE_DIR)

    if not POSE_DIR.is_dir():
        raise FileNotFoundError(
            f"Missing local pose directory: {POSE_DIR}"
        )

    pose_files = sorted(
        POSE_DIR.glob("*.pose")
    )

else:

    # --------------------------------------------------------
    # Kaggle / remote iSign access
    # --------------------------------------------------------

    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import hf_hub_download

    try:
        HF_TOKEN = (
            os.environ.get("HF_TOKEN")
            or UserSecretsClient().get_secret("HF_TOKEN")
        )
    except Exception as exc:
        raise RuntimeError(
            "Add HF_TOKEN as a Kaggle Secret."
        ) from exc

    if not HF_TOKEN:
        raise RuntimeError(
            "HF_TOKEN is empty."
        )

    print(
        "Downloading/verifying iSign CSV from Hugging Face..."
    )

    CSV_PATH = Path(
        hf_hub_download(
            repo_id="Exploration-Lab/iSign",
            filename="iSign_v1.1.csv",
            repo_type="dataset",
            token=HF_TOKEN,
            local_dir=str(DATA_ROOT),
        )
    )

    if not CSV_PATH.is_file():
        raise FileNotFoundError(
            f"Hugging Face download did not produce the expected CSV: "
            f"{CSV_PATH}"
        )

    # The large pose archive is intentionally NOT extracted.
    # Individual UID-level .pose members are streamed later.
    pose_files = []


# ------------------------------------------------------------
# Read CSV
# ------------------------------------------------------------

df = pd.read_csv(CSV_PATH)

RAW_CSV_ROWS = len(df)

required = {
    "uid",
    "text",
}

missing = required - set(df.columns)

if missing:
    raise RuntimeError(
        f"Missing required iSign CSV columns: {sorted(missing)}"
    )


# ------------------------------------------------------------
# Normalize basic fields
# ------------------------------------------------------------

df["uid"] = (
    df["uid"]
    .astype(str)
    .str.strip()
)

df["text"] = (
    df["text"]
    .fillna("")
    .astype(str)
    .str.replace(
        r"\s+",
        " ",
        regex=True,
    )
    .str.strip()
)


# ------------------------------------------------------------
# Derive video_id + sequence from UID
# ------------------------------------------------------------

df[["video_id", "seq"]] = (
    df["uid"]
    .str.rsplit(
        "-",
        n=1,
        expand=True,
    )
)

bad_video_id = (
    df["video_id"].isna()
    | (
        df["video_id"]
        .astype(str)
        .str.strip()
        .eq("")
    )
)

bad_seq = (
    df["seq"].isna()
    | (
        df["seq"]
        .astype(str)
        .str.strip()
        .eq("")
    )
)

malformed_uid_mask = (
    bad_video_id
    | bad_seq
)

malformed_uid = int(
    malformed_uid_mask.sum()
)

if malformed_uid:

    print(
        f"Removing {malformed_uid:,} rows "
        "with malformed UID/video/sequence fields."
    )

    df = df.loc[
        ~malformed_uid_mask
    ].copy()

df["video_id"] = (
    df["video_id"]
    .astype(str)
    .str.strip()
)

df["seq"] = (
    df["seq"]
    .astype(str)
    .str.strip()
)

df["n_words"] = (
    df["text"]
    .str.split()
    .str.len()
)


# ------------------------------------------------------------
# Remove only demonstrably invalid caption artifacts
#
# Legitimate one-word English targets such as:
#   hello
#   thanks
#   i
#
# remain in the dataset.
# ------------------------------------------------------------

before_text_cleaning = len(df)

text_norm = df["text"].map(
    normalize_text
)

known_junk = (
    text_norm.eq("")
    | text_norm.str.fullmatch(
        r"\d+"
    )
    | text_norm.str.fullmatch(
        r"[a-z]"
    )
    | text_norm.str.fullmatch(
        r"\[[^\]]+\]"
    )
    | text_norm.str.fullmatch(
        r"[^a-z0-9]+"
    )
)

# "i" is a legitimate English pronoun.
known_junk &= ~text_norm.eq("i")

# Empty UID is also invalid.
known_junk |= df["uid"].eq("")


junk_counts = {
    "empty_text": int(
        text_norm.eq("").sum()
    ),

    "numeric_only": int(
        text_norm.str.fullmatch(
            r"\d+"
        ).sum()
    ),

    "single_letter_non_i": int(
        (
            text_norm.str.fullmatch(
                r"[a-z]"
            )
            & ~text_norm.eq("i")
        ).sum()
    ),

    "bracketed_caption": int(
        text_norm.str.fullmatch(
            r"\[[^\]]+\]"
        ).sum()
    ),

    "punctuation_only": int(
        text_norm.str.fullmatch(
            r"[^a-z0-9]+"
        ).sum()
    ),
}

df = df.loc[
    ~known_junk
].copy()

after_text_cleaning = len(df)

text_cleaned_rows = (
    before_text_cleaning
    - after_text_cleaning
)


# ------------------------------------------------------------
# Duplicate UID safety check
# ------------------------------------------------------------

duplicate_uid = int(
    df["uid"]
    .duplicated()
    .sum()
)

if duplicate_uid:
    raise RuntimeError(
        f"Found {duplicate_uid:,} duplicate UIDs after cleaning. "
        "Refusing to train until resolved."
    )


# ------------------------------------------------------------
# Target-length audit
#
# Do NOT silently remove long targets here.
# They are reported and handled later by tokenization/truncation
# according to the declared MAX_TOKENS policy.
# ------------------------------------------------------------

too_long = int(
    (
        df["n_words"]
        > MAX_TOKENS - 2
    ).sum()
)


# ------------------------------------------------------------
# Final dataset report
# ------------------------------------------------------------

print()
print("=" * 70)
print("iSIGN DATASET VERIFICATION")
print("=" * 70)

print(
    f"Raw CSV rows: "
    f"{RAW_CSV_ROWS:,}"
)

print(
    f"Usable rows after cleaning: "
    f"{len(df):,}"
)

print(
    f"Rows removed by text/caption cleaning: "
    f"{text_cleaned_rows:,}"
)

print(
    f"Malformed UID rows removed: "
    f"{malformed_uid:,}"
)

print(
    "Known junk/caption rows removed "
    "(overlapping categories may be counted once in total): "
    f"{text_cleaned_rows:,}"
)

print(
    "Junk categories:",
    junk_counts,
)

print(
    f"Rows longer than {MAX_TOKENS - 2} words: "
    f"{too_long:,}"
)

print(
    f"Unique video IDs: "
    f"{df['video_id'].nunique():,}"
)


# ------------------------------------------------------------
# Pose source report
# ------------------------------------------------------------

if DATA_MODE == "local":

    print(
        f"Local pose files: "
        f"{len(pose_files):,}"
    )

else:

    print(
        "Pose source: remote multipart ZIP64; "
        "individual UID members will be streamed."
    )


# ------------------------------------------------------------
# Final safety checks
# ------------------------------------------------------------

if len(df) == 0:
    raise RuntimeError(
        "No usable iSign rows remain after cleaning."
    )

if df["uid"].isna().any():
    raise RuntimeError(
        "NaN UID values remain after cleaning."
    )

if df["video_id"].isna().any():
    raise RuntimeError(
        "NaN video_id values remain after cleaning."
    )

if df["text"].isna().any():
    raise RuntimeError(
        "NaN text values remain after cleaning."
    )

print()
print(
    f"Dataset verification PASSED: "
    f"{len(df):,} usable rows ready for splitting."
)

Dataset root: /kaggle/working/isign
Data mode: kaggle
Downloading/verifying iSign CSV from Hugging Face...


iSign_v1.1.csv: 0.00B [00:00, ?B/s]

Removing 2,291 rows with malformed UID/video/sequence fields.

iSIGN DATASET VERIFICATION
Raw CSV rows: 127,237
Usable rows after cleaning: 123,779
Rows removed by text/caption cleaning: 1,167
Malformed UID rows removed: 2,291
Known junk/caption rows removed (overlapping categories may be counted once in total): 1,167
Junk categories: {'empty_text': 391, 'numeric_only': 559, 'single_letter_non_i': 177, 'bracketed_caption': 31, 'punctuation_only': 9}
Rows longer than 94 words: 0
Unique video IDs: 3,797
Pose source: remote multipart ZIP64; individual UID members will be streamed.

Dataset verification PASSED: 123,779 usable rows ready for splitting.


In [5]:
# 1a. Record dataset facts in the knowledge repository.
pose_source = (
    "local UID-level .pose files"
    if DATA_MODE == "local"
    else "remote multipart ZIP64 archive on Hugging Face; lazy member reads"
)
selected_rows = len(df)

knowledge_insert(
    "dataset",
    ["run_id", "dataset_name", "csv_path", "csv_rows", "unique_video_ids", "pose_source", "selected_rows"],
    [RUN_ID, "iSign v1.1", str(CSV_PATH), int(len(df)), int(df["video_id"].nunique()), pose_source, int(selected_rows)],
)
knowledge_observation("dataset_columns", df.columns.tolist())
knowledge_observation("text_empty_rows", int((df["text"].str.strip() == "").sum()))

print("Dataset metadata recorded in SQLite.")


Dataset metadata recorded in SQLite.


In [6]:
# Kaggle remote ZIP64 backend
# The four iSign pose files are consecutive byte ranges of ONE ZIP archive.
# This backend presents them as one seekable virtual file so Python zipfile can
# read the ZIP64 central directory and then fetch only requested members.
# Reads are explicitly bounded to both part and HTTP-chunk boundaries so no bytes
# are skipped when zipfile asks for a range that crosses a cached chunk.

if DATA_MODE == "kaggle":
    import re
    import time
    import requests
    from huggingface_hub import hf_hub_url

    REMOTE_PARTS = [
        "iSign-poses_v1.1_part_aa",
        "iSign-poses_v1.1_part_ab",
        "iSign-poses_v1.1_part_ac",
        "iSign-poses_v1.1_part_ad",
    ]
    REMOTE_CHUNK_SIZE = REMOTE_CHUNK_SIZE_MB * 1024 * 1024
    REMOTE_CONNECT_TIMEOUT = 20
    REMOTE_READ_TIMEOUT = 180
    REMOTE_RETRIES = 5
    REMOTE_BACKOFF_SECONDS = 1.5

    class RemoteConcatFile(io.RawIOBase):
        def __init__(self, urls, sizes, token, chunk_size=REMOTE_CHUNK_SIZE):
            super().__init__()
            self.urls = urls
            self.sizes = sizes
            self.starts = np.cumsum([0] + sizes[:-1]).tolist()
            self.total_size = int(sum(sizes))
            self.pos = 0
            self.token = token
            self.chunk_size = int(chunk_size)
            self.session = requests.Session()
            self.session.headers.update({
                "Authorization": f"Bearer {token}",
                "User-Agent": "SignSage-Kaggle/1.0",
            })
            from collections import OrderedDict
            self.cache = OrderedDict()
            # Caches the final (post-redirect) CDN URL per part so repeated chunk
            # fetches skip the huggingface.co auth+redirect hop. Cleared and
            # re-resolved automatically if a cached link ever expires (see _fetch).
            self.resolved = {}

        def readable(self):
            return True

        def seekable(self):
            return True

        def tell(self):
            return self.pos

        def seek(self, offset, whence=os.SEEK_SET):
            if whence == os.SEEK_SET:
                new = offset
            elif whence == os.SEEK_CUR:
                new = self.pos + offset
            elif whence == os.SEEK_END:
                new = self.total_size + offset
            else:
                raise ValueError("invalid whence")
            if new < 0:
                raise ValueError("negative seek position")
            self.pos = min(int(new), self.total_size)
            return self.pos

        def _part_for(self, pos):
            idx = int(np.searchsorted(self.starts, pos, side="right") - 1)
            idx = max(0, min(idx, len(self.urls) - 1))
            return idx, pos - self.starts[idx]

        def _fetch(self, part_idx, local_start, n):
            key = (part_idx, (local_start // self.chunk_size) * self.chunk_size)
            if key in self.cache:
                self.cache.move_to_end(key)
                returned_start, content = self.cache[key]
                offset = local_start - returned_start
                piece = content[offset:offset + n]
                if len(piece) == n:
                    return piece
                # A corrupted/incomplete cache entry must never be served.
                self.cache.pop(key, None)

            chunk_start = key[1]
            chunk_end = min(
                self.sizes[part_idx] - 1,
                chunk_start + self.chunk_size - 1,
            )
            headers = {"Range": f"bytes={chunk_start}-{chunk_end}"}

            last_error = None
            for attempt in range(1, REMOTE_RETRIES + 1):
                try:
                    # Resolve the origin URL once, then reuse the CDN URL.
                    request_url = self.resolved.get(part_idx, self.urls[part_idx])
                    r = self.session.get(
                        request_url,
                        headers=headers,
                        timeout=(REMOTE_CONNECT_TIMEOUT, REMOTE_READ_TIMEOUT),
                        allow_redirects=True,
                        stream=False,
                    )

                    if r.status_code != 206 and request_url != self.urls[part_idx]:
                        # Cached CDN URL may have expired. Re-resolve once.
                        self.resolved.pop(part_idx, None)
                        r.close()
                        r = self.session.get(
                            self.urls[part_idx],
                            headers=headers,
                            timeout=(REMOTE_CONNECT_TIMEOUT, REMOTE_READ_TIMEOUT),
                            allow_redirects=True,
                            stream=False,
                        )

                    if r.status_code != 206:
                        raise RuntimeError(
                            f"Range request failed for part {part_idx}: HTTP {r.status_code}"
                        )

                    cr = r.headers.get("Content-Range", "")
                    match = re.fullmatch(r"bytes (\d+)-(\d+)/(\d+)", cr.strip())
                    if not match:
                        raise RuntimeError(
                            f"Invalid Content-Range from HF response: {cr!r}"
                        )

                    returned_start, returned_end, returned_total = map(int, match.groups())
                    if returned_total != self.sizes[part_idx]:
                        raise RuntimeError(
                            f"HF reported part {part_idx} total size {returned_total}, "
                            f"but the size probe reported {self.sizes[part_idx]}."
                        )
                    if not (returned_start <= chunk_start and chunk_end <= returned_end):
                        raise RuntimeError(
                            f"HF returned byte range {returned_start}-{returned_end}; "
                            f"requested {chunk_start}-{chunk_end}."
                        )

                    content = r.content
                    expected_response_len = returned_end - returned_start + 1
                    if len(content) != expected_response_len:
                        raise RuntimeError(
                            f"Incomplete range response for part {part_idx}: "
                            f"Content-Range says {expected_response_len} bytes, "
                            f"but received {len(content)}."
                        )

                    self.resolved[part_idx] = r.url
                    self.cache[key] = (returned_start, content)
                    self.cache.move_to_end(key)
                    while len(self.cache) > REMOTE_CACHE_CHUNKS:
                        self.cache.popitem(last=False)

                    offset = local_start - returned_start
                    piece = content[offset:offset + n]
                    if len(piece) != n:
                        raise RuntimeError(
                            f"Remote range backend returned {len(piece)} bytes; expected {n}."
                        )
                    return piece

                except (requests.exceptions.Timeout,
                        requests.exceptions.ConnectionError,
                        requests.exceptions.ChunkedEncodingError) as exc:
                    last_error = exc
                    self.resolved.pop(part_idx, None)
                    try:
                        r.close()
                    except Exception:
                        pass
                    if attempt < REMOTE_RETRIES:
                        delay = REMOTE_BACKOFF_SECONDS * (2 ** (attempt - 1))
                        print(
                            f"[iSign remote] transient network error on part {part_idx}, "
                            f"range {chunk_start}-{chunk_end}; retry "
                            f"{attempt}/{REMOTE_RETRIES} after {delay:.1f}s"
                        )
                        time.sleep(delay)
                    continue
                except Exception:
                    try:
                        r.close()
                    except Exception:
                        pass
                    raise

            raise ConnectionError(
                f"Unable to fetch iSign ZIP range {chunk_start}-{chunk_end} "
                f"from part {part_idx} after {REMOTE_RETRIES} attempts."
            ) from last_error

        def read(self, n=-1):
            if self.pos >= self.total_size:
                return b""
            if n is None or n < 0:
                n = self.total_size - self.pos
            n = min(int(n), self.total_size - self.pos)
            out = bytearray()
            remaining = n
            while remaining:
                part_idx, local = self._part_for(self.pos)
                bytes_to_part_end = self.sizes[part_idx] - local
                bytes_to_chunk_end = self.chunk_size - (local % self.chunk_size)
                take = min(remaining, bytes_to_part_end, bytes_to_chunk_end)
                piece = self._fetch(part_idx, local, take)
                out.extend(piece)
                self.pos += take
                remaining -= take
            return bytes(out)

    def _remote_size(url):
        # A one-byte Range request gives the authoritative total size via Content-Range
        # without downloading a huge file.
        r = requests.get(
            url,
            headers={
                "Authorization": f"Bearer {HF_TOKEN}",
                "Range": "bytes=0-0",
                "User-Agent": "SignSage-Kaggle/1.0",
            },
            timeout=(REMOTE_CONNECT_TIMEOUT, REMOTE_READ_TIMEOUT),
            allow_redirects=True,
        )
        if r.status_code != 206:
            raise RuntimeError(f"Could not probe remote part size: HTTP {r.status_code}")
        cr = r.headers.get("Content-Range", "")
        total = cr.rsplit("/", 1)[-1] if "/" in cr else ""
        if not total.isdigit():
            raise RuntimeError(f"Could not parse total size from Content-Range: {cr!r}")
        return int(total)

    remote_urls = [
        hf_hub_url("Exploration-Lab/iSign", f, repo_type="dataset")
        for f in REMOTE_PARTS
    ]
    remote_sizes = []
    for name, url in zip(REMOTE_PARTS, remote_urls):
        size = _remote_size(url)
        remote_sizes.append(size)
        print(f"{name}: {size / 1024**3:.2f} GiB")

    remote_file = RemoteConcatFile(remote_urls, remote_sizes, HF_TOKEN)
    remote_zip = zipfile.ZipFile(remote_file, mode="r", allowZip64=True)
    remote_names = set(remote_zip.namelist())
    print(f"Remote ZIP opened: {len(remote_names):,} members")

    # The member-path layout inside this archive is an assumption, not a verified
    # fact: RemotePoseStore below only tries "iSign-poses_v1.1/{uid}.pose" and
    # "{uid}.pose". Printing a few real member names here means a different prefix
    # or nesting depth is visible immediately, on the very first Kaggle run --
    # instead of surfacing later as a FileNotFoundError on every single UID, after
    # the split/tokenizer/cache cells have already run.
    print("Sample members:", sorted(remote_names)[:5])

    class RemotePoseStore:
        def __init__(self, zf, names):
            self.zf = zf
            self.names = names

        def _candidate(self, uid):
            candidates = (
                f"iSign-poses_v1.1/{uid}.pose",
                f"{uid}.pose",
            )
            for name in candidates:
                if name in self.names:
                    return name
            raise FileNotFoundError(
                f"UID not found in remote iSign pose archive: {uid}"
            )

        def info_for_uid(self, uid):
            """Return ZIP metadata without downloading the pose payload."""
            return self.zf.getinfo(self._candidate(uid))

        def read_member_bytes(self, uid):
            with self.zf.open(self._candidate(uid), "r") as f:
                return f.read()

    remote_pose_store = RemotePoseStore(remote_zip, remote_names)
    print("Remote pose backend ready — individual .pose members will be fetched lazily.")

    # Verify that every CSV UID has a corresponding .pose member using the ZIP
    # central directory only. This does NOT download all pose files.
    def _find_remote_member_name(uid):
        for candidate in (f"iSign-poses_v1.1/{uid}.pose", f"{uid}.pose"):
            if candidate in remote_names:
                return candidate
        return None

    _remote_uid_missing = []
    for uid in df["uid"].astype(str):
        if _find_remote_member_name(uid) is None:
            _remote_uid_missing.append(uid)
    print(f"Remote pose coverage: {len(df)-len(_remote_uid_missing):,}/{len(df):,} CSV UIDs")
    if _remote_uid_missing:
        print("First missing UIDs:", _remote_uid_missing[:10])
        raise RuntimeError(
            f"{len(_remote_uid_missing):,} CSV UIDs have no matching .pose member. "
            "Refusing to start full training with incomplete pose coverage."
        )
else:
    remote_pose_store = None

# Backward-compatible alias for any legacy helper; new network code uses the explicit connect/read timeouts above.
REMOTE_TIMEOUT = (REMOTE_CONNECT_TIMEOUT, REMOTE_READ_TIMEOUT)


iSign-poses_v1.1_part_aa: 45.00 GiB
iSign-poses_v1.1_part_ab: 45.00 GiB
iSign-poses_v1.1_part_ac: 45.00 GiB
iSign-poses_v1.1_part_ad: 23.54 GiB
Remote ZIP opened: 127,238 members
Sample members: ['iSign-poses_v1.1/', 'iSign-poses_v1.1/-02o_0vVwzI--0.pose', 'iSign-poses_v1.1/-02o_0vVwzI--1.pose', 'iSign-poses_v1.1/-02o_0vVwzI--10.pose', 'iSign-poses_v1.1/-02o_0vVwzI--11.pose']
Remote pose backend ready — individual .pose members will be fetched lazily.
Remote pose coverage: 123,779/123,779 CSV UIDs


## 2. Data audit

The audit is deliberately before training. A model cannot repair systematic text/pose misalignment.

The local dataset inspection already established that the `.pose` archive uses **row/UID-level files**, not `<video_id>.pose`. Therefore every CSV row is resolved through its complete UID.

In [7]:
from pose_format import Pose

_worker_remote_pose_store = None

def _get_active_remote_pose_store():
    """Return a process-local ZIP reader; never share a seekable ZIP across workers."""
    global _worker_remote_pose_store
    if DATA_MODE != "kaggle":
        return remote_pose_store
    from torch.utils.data import get_worker_info
    if get_worker_info() is None:
        return remote_pose_store
    if _worker_remote_pose_store is None:
        assert REMOTE_CHUNK_SIZE > 0
        worker_file = RemoteConcatFile(remote_urls, remote_sizes, HF_TOKEN, REMOTE_CHUNK_SIZE)
        worker_zip = zipfile.ZipFile(worker_file, mode="r", allowZip64=True)
        _worker_remote_pose_store = RemotePoseStore(worker_zip, set(worker_zip.namelist()))
    return _worker_remote_pose_store

def read_pose_bytes_for_uid(uid):
    if DATA_MODE == "local":
        path = POSE_DIR / f"{uid}.pose"
        if not path.is_file():
            raise FileNotFoundError(path)
        return path.read_bytes()
    return _get_active_remote_pose_store().read_member_bytes(uid)

# Inspect a few actual pose files through the active backend.
sample_uids = df.head(5)["uid"].tolist()
for uid in sample_uids:
    p = Pose.read(read_pose_bytes_for_uid(uid))
    data = np.asarray(np.ma.filled(p.body.data, 0.0))
    conf = np.asarray(np.ma.filled(p.body.confidence, 0.0))
    print(f"{uid}.pose: shape={data.shape}, mean_confidence={conf.mean():.3f}")

example_pose = Pose.read(read_pose_bytes_for_uid(sample_uids[0]))
example_data = np.asarray(np.ma.filled(example_pose.body.data, 0.0))
TOTAL_KEYPOINTS = example_data.shape[2]
print("\nVerified pose shape:", example_data.shape)
print("Keypoints:", TOTAL_KEYPOINTS)


1782bea75c7d-1.pose: shape=(113, 1, 576, 3), mean_confidence=0.965
1782bea75c7d-2.pose: shape=(104, 1, 576, 3), mean_confidence=0.960
1782bea75c7d-3.pose: shape=(196, 1, 576, 3), mean_confidence=0.961
1782bea75c7d-4.pose: shape=(78, 1, 576, 3), mean_confidence=0.965
1782bea75c7d-5.pose: shape=(124, 1, 576, 3), mean_confidence=0.963

Verified pose shape: (113, 1, 576, 3)
Keypoints: 576


In [8]:
## 2. Data integrity audit before training

# Dataset-use policy:
# - Primary training/evaluation: complete cleaned iSign v1.1 table and matching UID-level .pose members.
# - Excluded: only empty text and demonstrable non-sentence/caption artifacts.
# - Retained: legitimate one-word English targets.
# - INCLUDE is not merged into Sign→English training because it is an isolated-sign
#   dataset with a different task structure; it remains a separate reference/baseline dataset.

# These checks are intentionally cheap (CSV-only) and run over the entire usable iSign table.
# They answer whether the "large dataset" is actually being used and whether labels are
# duplicated/conflicting before any GPU time is spent.
print("=== FULL DATA AUDIT ===")
print("Rows available after non-empty-text cleaning:", f"{len(df):,}")
print("Unique UIDs:", f"{df.uid.nunique():,}")
print("Unique videos:", f"{df.video_id.nunique():,}")
print("Unique normalized English targets:", f"{df.text.map(normalize_text).nunique():,}")
print("Rows with 1 word:", int((df.n_words==1).sum()))
print("Rows > max target word length:", int((df.n_words>MAX_TOKENS-2).sum()))
spm_estimated_truncation_note = "SentencePiece token length is audited after tokenizer training."

# Duplicate UID must never map to multiple pose files.
assert df.uid.is_unique, "UID duplication detected."

# Report repeated target sentences; repetition is not itself an error.
target_counts=df.text.map(normalize_text).value_counts()
print("Most frequent targets:")
for text,count in target_counts.head(10).items():
    print(f"  {count:5d} × {text}")

# Check for exact duplicate UID/text rows before split.
dupes=int(df.duplicated(subset=["uid","text"]).sum())
print("Exact duplicate UID/text rows:",dupes)

knowledge_observation("full_dataset_audit",{
    "usable_rows":int(len(df)),
    "unique_uids":int(df.uid.nunique()),
    "unique_videos":int(df.video_id.nunique()),
    "unique_targets":int(df.text.map(normalize_text).nunique()),
    "one_word_rows":int((df.n_words==1).sum()),
    "too_long_rows":int((df.n_words>MAX_TOKENS-2).sum()),
    "exact_uid_text_duplicates":dupes,
})


# Full-run coverage statement: this is a row-level experiment over every cleaned
# CSV UID in the selected split; pose members are streamed lazily rather than
# downloaded/extracted as a monolithic archive.
print("Training policy: every row in the selected training split is eligible and visited once per epoch.")
if RUN_MODE != "full":
    print("This run is bounded; results must not be described as full-dataset training.")


=== FULL DATA AUDIT ===
Rows available after non-empty-text cleaning: 123,779
Unique UIDs: 123,779
Unique videos: 3,797
Unique normalized English targets: 120,221
Rows with 1 word: 2663
Rows > max target word length: 0
Most frequent targets:
     66 × let me tell you about it.
     52 × blank
     45 × dash.
     38 × have a look at the video.
     33 × dash
     32 × for the teacher.
     31 × blank.
     29 × let me tell you what happened.
     29 × reading is fun.
     27 × new words
Exact duplicate UID/text rows: 0
Training policy: every row in the selected training split is eligible and visited once per epoch.


In [9]:
# 3. Leak-safe split by video_id.
video_ids = np.array(df["video_id"].dropna().unique(), dtype=object)
rng = np.random.default_rng(SEED); rng.shuffle(video_ids)

n_videos = len(video_ids)
train_cut, val_cut = int(.80*n_videos), int(.90*n_videos)
train_ids = set(video_ids[:train_cut]); val_ids = set(video_ids[train_cut:val_cut]); test_ids = set(video_ids[val_cut:])

df_train_all = df[df.video_id.isin(train_ids)].reset_index(drop=True)
df_val_all = df[df.video_id.isin(val_ids)].reset_index(drop=True)
df_test_all = df[df.video_id.isin(test_ids)].reset_index(drop=True)

# Do not drop long but otherwise valid samples. The tokenizer truncates targets only
# when they exceed MAX_TOKENS-2, and the truncation count is reported in the audit.
# This keeps every non-empty iSign row represented in the split.
df_train_all = df_train_all.reset_index(drop=True)
df_val_all = df_val_all.reset_index(drop=True)
df_test_all = df_test_all.reset_index(drop=True)

def limit_frame(frame, limit):
    if limit is None or limit >= len(frame):
        return frame.copy().reset_index(drop=True)
    return frame.sample(int(limit), random_state=SEED).reset_index(drop=True)

df_train = limit_frame(df_train_all, TRAIN_LIMIT)
df_val = limit_frame(df_val_all, VAL_LIMIT)
df_test = limit_frame(df_test_all, TEST_LIMIT)

print(f"Usable split rows: train={len(df_train_all):,}, val={len(df_val_all):,}, test={len(df_test_all):,}")
print(f"Rows used this run: train={len(df_train):,}, val={len(df_val):,}, test={len(df_test):,}")
# The *_all frames must account for every cleaned row. The *_run frames may
# be bounded in smoke/scaled modes; in full mode they must equal *_all.
assert len(df_train_all) + len(df_val_all) + len(df_test_all) == len(df), (
    "Leak-safe full split does not account for every cleaned row."
)
assert len(df_train) <= len(df_train_all) and len(df_val) <= len(df_val_all) and len(df_test) <= len(df_test_all)
if TRAIN_LIMIT is None and VAL_LIMIT is None and TEST_LIMIT is None:
    assert len(df_train) == len(df_train_all) and len(df_val) == len(df_val_all) and len(df_test) == len(df_test_all)
print(f"All cleaned rows assigned to exactly one full split: {len(df_train_all)+len(df_val_all)+len(df_test_all):,}/{len(df):,}")
assert set(df_train.video_id).isdisjoint(df_val.video_id)
assert set(df_train.video_id).isdisjoint(df_test.video_id)
assert set(df_val.video_id).isdisjoint(df_test.video_id)
print("No video_id leakage across train/val/test.")


Usable split rows: train=99,699, val=11,927, test=12,153
Rows used this run: train=99,699, val=11,927, test=12,153
All cleaned rows assigned to exactly one full split: 123,779/123,779
No video_id leakage across train/val/test.


In [10]:
# 3b. Explicit dataset-usage report
cleaned_rows = len(df)
split_rows = len(df_train_all) + len(df_val_all) + len(df_test_all)
assert split_rows == cleaned_rows
print("=== DATA USED BY PRIMARY SIGN→ENGLISH EXPERIMENT ===")
print(f"Original CSV rows:       {RAW_CSV_ROWS:,}")
print(f"Cleaned usable rows:     {cleaned_rows:,}")
print(f"Rows excluded by cleaning: {RAW_CSV_ROWS-cleaned_rows:,}")
print(f"Train rows:              {len(df_train):,} ({len(df_train)/cleaned_rows:.2%})")
print(f"Validation rows:         {len(df_val):,} ({len(df_val)/cleaned_rows:.2%})")
print(f"Test rows:               {len(df_test):,} ({len(df_test)/cleaned_rows:.2%})")
print(f"Rows used in this run:   {len(df_train)+len(df_val)+len(df_test):,}")
print("Run mode:", RUN_MODE)
print("Full-data mode:", TRAIN_LIMIT is None and VAL_LIMIT is None and TEST_LIMIT is None)
print("NOTE: the raw CSV is not identical to the training set: only explicitly invalid/"
      "empty/caption-artifact rows are removed. INCLUDE is not merged because it is an "
      "isolated-sign dataset and would change the task definition.")


=== DATA USED BY PRIMARY SIGN→ENGLISH EXPERIMENT ===
Original CSV rows:       127,237
Cleaned usable rows:     123,779
Rows excluded by cleaning: 3,458
Train rows:              99,699 (80.55%)
Validation rows:         11,927 (9.64%)
Test rows:               12,153 (9.82%)
Rows used in this run:   123,779
Run mode: full
Full-data mode: True
NOTE: the raw CSV is not identical to the training set: only explicitly invalid/empty/caption-artifact rows are removed. INCLUDE is not merged because it is an isolated-sign dataset and would change the task definition.


In [11]:
# 3a. Record the video-level split in the knowledge repository.
train_videos = int(df_train["video_id"].nunique())
val_videos = int(df_val["video_id"].nunique())
test_videos = int(df_test["video_id"].nunique())

overlap_train_val = len(set(df_train["video_id"]) & set(df_val["video_id"]))
overlap_train_test = len(set(df_train["video_id"]) & set(df_test["video_id"]))
overlap_val_test = len(set(df_val["video_id"]) & set(df_test["video_id"]))
leakage_free = (overlap_train_val == overlap_train_test == overlap_val_test == 0)

knowledge_insert(
    "splits",
    ["run_id", "train_rows", "val_rows", "test_rows", "train_videos", "val_videos", "test_videos", "leakage_free"],
    [RUN_ID, len(df_train), len(df_val), len(df_test), train_videos, val_videos, test_videos, int(leakage_free)],
)
knowledge_observation("split_video_overlap", {
    "train_val": overlap_train_val,
    "train_test": overlap_train_test,
    "val_test": overlap_val_test,
})
print("Split metadata recorded. Leakage-free:", leakage_free)


Split metadata recorded. Leakage-free: True


## 4. Tokenizer

A simple whitespace tokenizer is retained for the core experiment so the notebook does not hide the translation problem behind a large pretrained language model.

The vocabulary is built from **training text only**. Unknown validation/test words map to `<unk>`.

In [12]:
## 4. SentencePiece subword tokenizer

# The previous run exposed a fundamental problem with a one-token-per-word vocabulary:
# 59,742 output classes for 99,699 training examples. That makes the decoder's
# classification problem unnecessarily sparse and encourages generic-output collapse.
#
# Keep the complete usable iSign dataset, but represent English with a compact
# subword vocabulary. SentencePiece is trained ONLY on the training split, then
# frozen and exported as an inference artifact. This preserves the research
# experiment while making rare words compositional instead of automatic <unk>.
import sentencepiece as spm

TOKENIZER_DIR = REPORT_DIR / "tokenizer"
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_PREFIX = TOKENIZER_DIR / "signsage_en_sp"
TOKENIZER_MODEL = TOKENIZER_PREFIX.with_suffix(".model")
TOKENIZER_VOCAB = TOKENIZER_PREFIX.with_suffix(".vocab")
TOKENIZER_CORPUS = TOKENIZER_DIR / "train_text.txt"

# Deterministic training corpus: one normalized target per line.
with open(TOKENIZER_CORPUS, "w", encoding="utf-8") as f:
    for text in df_train["text"].map(normalize_text):
        f.write(text.replace("\n", " ") + "\n")

if not TOKENIZER_MODEL.exists():
    spm.SentencePieceTrainer.train(
        input=str(TOKENIZER_CORPUS),
        model_prefix=str(TOKENIZER_PREFIX),
        vocab_size=TARGET_VOCAB_SIZE,
        model_type="unigram",
        character_coverage=1.0,
        normalization_rule_name="nmt_nfkc_cf",
        pad_id=0, unk_id=1, bos_id=2, eos_id=3,
        pad_piece="<pad>", unk_piece="<unk>",
        bos_piece="<bos>", eos_piece="<eos>",
        hard_vocab_limit=False,
        shuffle_input_sentence=False,
        input_sentence_size=0,
        seed_sentencepiece_size=1000000,
        num_threads=max(1, min(8, os.cpu_count() or 1)),
    )

sp = spm.SentencePieceProcessor(model_file=str(TOKENIZER_MODEL))
PAD_IDX, UNK_IDX, BOS_IDX, EOS_IDX = sp.pad_id(), sp.unk_id(), sp.bos_id(), sp.eos_id()
SPECIALS = ["<pad>", "<bos>", "<eos>", "<unk>"]

# Keep a simple token_to_id dictionary for the existing experiment database/export code.
vocab = {sp.id_to_piece(i): i for i in range(sp.get_piece_size())}
inv_vocab = {i: sp.id_to_piece(i) for i in range(sp.get_piece_size())}

def encode_text(text, max_len=MAX_TOKENS):
    ids = sp.encode(normalize_text(text), out_type=int, add_bos=True, add_eos=True)
    if len(ids) > max_len:
        ids = ids[:max_len-1] + [EOS_IDX]
    return ids

def decode_ids(ids):
    clean=[]
    for idx in ids:
        idx=int(idx)
        if idx == EOS_IDX:
            break
        if idx not in (BOS_IDX, PAD_IDX):
            clean.append(idx)
    if not clean:
        return ""
    return normalize_text(sp.decode(clean))

def _unk_rate(frame):
    total=0
    unk=0
    for text in frame.text:
        ids=sp.encode(normalize_text(text), out_type=int)
        total += len(ids)
        unk += sum(int(x)==UNK_IDX for x in ids)
    return unk/max(1,total)

val_unk_rate, test_unk_rate = _unk_rate(df_val), _unk_rate(df_test)
print("SentencePiece vocabulary size:", sp.get_piece_size())
print(f"Validation <unk> piece rate: {val_unk_rate:.4%}")
print(f"Test <unk> piece rate: {test_unk_rate:.4%}")
print(f"Tokenizer model: {TOKENIZER_MODEL}")
assert PAD_IDX == 0 and UNK_IDX == 1 and BOS_IDX == 2 and EOS_IDX == 3


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: /kaggle/working/signsage_outputs/reports/tokenizer/train_text.txt
  input_format: 
  model_prefix: /kaggle/working/signsage_outputs/reports/tokenizer/signsage_en_sp
  model_type: UNIGRAM
  vocab_size: 16000
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 0
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 4
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 0
  use_all_vocab: 0
  unk_id: 1
  bos_id: 2
  eos_id: 3
  pad_id: 0
  unk_piece: <unk>
  bos_pie

SentencePiece vocabulary size: 16000
Validation <unk> piece rate: 0.0006%
Test <unk> piece rate: 0.0025%
Tokenizer model: /kaggle/working/signsage_outputs/reports/tokenizer/signsage_en_sp.model


In [13]:
# 4a. SentencePiece target-length audit
_train_token_lengths = np.array([len(sp.encode(normalize_text(t), out_type=int)) + 2 for t in df_train["text"]], dtype=np.int32)
_val_token_lengths = np.array([len(sp.encode(normalize_text(t), out_type=int)) + 2 for t in df_val["text"]], dtype=np.int32)
_test_token_lengths = np.array([len(sp.encode(normalize_text(t), out_type=int)) + 2 for t in df_test["text"]], dtype=np.int32)
print("SentencePiece token length percentiles (train):",
      {p:int(np.percentile(_train_token_lengths,p)) for p in [50,90,95,99,100]})
print("Targets truncated by MAX_TOKENS:",
      int((_train_token_lengths > MAX_TOKENS).sum()),
      f"/ {len(_train_token_lengths):,} training rows")
assert (_train_token_lengths > MAX_TOKENS).sum() <= max(10, int(0.001 * len(_train_token_lengths))), \
    "Too many targets would be truncated; increase MAX_TOKENS before training."


SentencePiece token length percentiles (train): {50: 15, 90: 23, 95: 26, 99: 31, 100: 87}
Targets truncated by MAX_TOKENS: 0 / 99,699 training rows


## 5. Pose representation and face ablation

The actual pose header contains:

- `POSE_LANDMARKS`: 33
- `FACE_LANDMARKS`: 468
- `LEFT_HAND_LANDMARKS`: 21
- `RIGHT_HAND_LANDMARKS`: 21
- `POSE_WORLD_LANDMARKS`: 33

For SignSage's intended MediaPipe-style input, the main experiment uses the first **543 image-space landmarks** (pose + face + hands) and deliberately excludes the duplicated `POSE_WORLD_LANDMARKS` block.

The no-face ablation retains pose + both hands.

Sequences are normalized by a sequence-level body center and scale. This reduces dependence on camera translation/size while preserving temporal motion.

In [14]:
# Confirm component ranges from the real file rather than hardcoding the face indices blindly.
def get_component_ranges(pose):
    ranges = {}
    idx = 0
    for component in pose.header.components:
        n_points = len(component.points)
        ranges[str(component.name)] = (idx, idx + n_points)
        idx += n_points
    return ranges

component_ranges = get_component_ranges(example_pose)
print(component_ranges)

IMAGE_KEYPOINTS = list(range(543))
face_range = component_ranges.get("FACE_LANDMARKS")
assert face_range is not None, "FACE_LANDMARKS was not found; do not run the ablation until the pose header is understood."

keep_noface = [
    i for i in IMAGE_KEYPOINTS
    if not (face_range[0] <= i < face_range[1])
]

FULL_FEATURE_MASK = np.array([3*i + j for i in IMAGE_KEYPOINTS for j in range(3)], dtype=np.int64)
NOFACE_FEATURE_MASK = np.array([3*i + j for i in keep_noface for j in range(3)], dtype=np.int64)

print("Full keypoints/features:", len(IMAGE_KEYPOINTS), len(FULL_FEATURE_MASK))
print("No-face keypoints/features:", len(keep_noface), len(NOFACE_FEATURE_MASK))


{'POSE_LANDMARKS': (0, 33), 'FACE_LANDMARKS': (33, 501), 'LEFT_HAND_LANDMARKS': (501, 522), 'RIGHT_HAND_LANDMARKS': (522, 543), 'POSE_WORLD_LANDMARKS': (543, 576)}
Full keypoints/features: 543 1629
No-face keypoints/features: 75 225


### 5b. Landmark/feature-order contract export (backend-critical)


In [15]:
## 5b. Export the authoritative landmark/feature-order contract (BLOCKING FIX #1)
#
# component_ranges was previously computed correctly from the real pose header
# but only ever printed -- never saved. The backend has no way to know how the
# 543 landmarks are ordered without this file. Everything below is derived
# directly from component_ranges / FULL_FEATURE_MASK (the actual training-time
# values), never manually typed.

def _find_component_range(ranges, keywords):
    """Best-effort identification of a named component range by keyword match.
    Returns None (never a guess) if no component name confidently matches --
    the authoritative ground truth is always component_ranges/component_order
    below, regardless of whether this convenience lookup succeeds."""
    for name, rng in ranges.items():
        lname = str(name).lower()
        if all(k in lname for k in keywords):
            return {"component_name": name, "start": int(rng[0]), "end": int(rng[1])}
    return None

component_order = list(component_ranges.keys())
pose_range = _find_component_range(component_ranges, ["pose"])
face_range_named = _find_component_range(component_ranges, ["face"])
left_hand_range = _find_component_range(component_ranges, ["left"]) or _find_component_range(component_ranges, ["left", "hand"])
right_hand_range = _find_component_range(component_ranges, ["right"]) or _find_component_range(component_ranges, ["right", "hand"])

# Sanity check: the hip indices normalize_pose_sequence() hardcodes (23, 24)
# must actually fall inside the identified pose range, or the exported
# contract and the real normalization would silently disagree.
if pose_range is not None:
    assert pose_range["start"] <= 23 < pose_range["end"] and pose_range["start"] <= 24 < pose_range["end"], (
        "Hip indices (23,24) used by normalize_pose_sequence() fall outside the "
        f"identified pose component range {pose_range}. Do NOT export this contract "
        "until this is resolved -- the backend's normalization would not match training."
    )
    print(f"Verified: hip indices (23,24) fall inside pose component range {pose_range}.")
else:
    print("WARNING: could not confidently identify a 'pose' component by name; "
          "component_ranges/component_order below is still the full authoritative "
          "ordering regardless.")

# Verify (not assume) the expected 4-component prefix and confirm anything
# beyond the first 543 global indices (e.g. POSE_WORLD_LANDMARKS, if this
# pose format's header includes it) is genuinely excluded from
# FULL_FEATURE_MASK. This is a real runtime check against the actual header,
# not a documentation claim -- if pose_format's component order ever changes,
# this fails loudly here rather than shipping a silently-wrong contract.
_expected_component_prefix = ["POSE_LANDMARKS", "FACE_LANDMARKS", "LEFT_HAND_LANDMARKS", "RIGHT_HAND_LANDMARKS"]
_actual_component_prefix = component_order[:4]
_prefix_matches_expected = (_actual_component_prefix == _expected_component_prefix)
if _prefix_matches_expected:
    print(f"Verified: the first four pose-header components are exactly "
          f"{_expected_component_prefix}, matching the layout FULL_FEATURE_MASK assumes.")
else:
    print(f"NOTE: pose-header component order is {_actual_component_prefix}, which does not "
          f"name-match the conventional {_expected_component_prefix} ordering. "
          "component_ranges/component_order below are still the exact, authoritative "
          "training-time layout regardless of naming -- use them, not this convenience check.")

_excluded_components = [str(name) for name, rng in component_ranges.items() if int(rng[0]) >= 543]
if _excluded_components:
    print(f"Components excluded from the 1629-D feature vector (start index >= 543): {_excluded_components}")
else:
    print("No pose-header components fall outside the first 543 global indices "
          "(e.g. no separate POSE_WORLD_LANDMARKS component is present, or it is not reached).")

landmark_contract = {
    "contract_version": "landmark_contract_v1",
    "total_landmark_count": 543,
    "raw_feature_dim": int(len(FULL_FEATURE_MASK)),
    "coordinates_per_landmark": 3,
    "coordinate_order": ["x", "y", "z"],
    # Authoritative ground truth: derived directly from the real .pose file's
    # header at training time, in file order. Not a manual approximation.
    # NOTE: component_order lists COMPONENT NAMES (each a named landmark group
    # from the pose file header), NOT landmark indices. Global landmark
    # indices 0..542 are assigned by walking component_order in this exact
    # order and concatenating each component's points in sequence;
    # component_ranges gives the resulting half-open [start,end) index range
    # per component. Any component whose range starts at or beyond global
    # index 543 (see excluded_components below) is outside FULL_FEATURE_MASK's
    # range(543) and is excluded from the model's raw 1629-D feature vector.
    "component_order": component_order,
    "component_order_note": (
        "component_order lists COMPONENT NAMES, not landmark indices. Global landmark "
        "indices 0..542 are assigned by walking this list in order and concatenating each "
        "component's points; see component_ranges for the resulting index range per component."
    ),
    "component_ranges": {str(k): [int(v[0]), int(v[1])] for k, v in component_ranges.items()},
    "global_index_layout": {
        "expected_component_prefix": _expected_component_prefix,
        "actual_component_prefix": _actual_component_prefix,
        "matches_expected_prefix": bool(_prefix_matches_expected),
        "excluded_components": _excluded_components,
        "excluded_components_note": (
            "Any component whose range starts at or after global index 543 (e.g. "
            "POSE_WORLD_LANDMARKS, if present in this pose file's header) falls outside "
            "FULL_FEATURE_MASK's range(543) and is excluded from the model's raw 1629-D "
            "feature vector entirely -- verified above against the real header, not assumed."
        ),
    },
    # Convenience lookups (best-effort name matching over component_ranges above;
    # null means "not confidently identified by name" -- component_ranges is
    # still the ground truth either way).
    "pose_landmark_range": pose_range,
    "face_landmark_range": face_range_named,
    "left_hand_landmark_range": left_hand_range,
    "right_hand_landmark_range": right_hand_range,
    "normalization_reference_points": {
        "hip_indices": [23, 24],
        "hip_indices_description": "MediaPipe pose landmarks: 23 = left hip, 24 = right hip (global indices).",
        "body_landmark_count_for_scale": 33,
        "body_landmark_description": "First 33 global landmark indices (the pose component) are used to estimate per-frame scale.",
    },
    "feature_flattening_order": (
        "For each frame: for landmark index i in component_order (global indices 0..542), "
        "emit [x_i, y_i, z_i]. Frame feature vector = concat over i of [x_i,y_i,z_i], length "
        "543*3=1629. Frames are stacked as rows: final raw array shape (num_frames, 1629)."
    ),
    "full_feature_mask": FULL_FEATURE_MASK.tolist(),
    "normalization_version": "pose_normalize_v1",
    "normalization_description": (
        "Per-sequence: root-center every frame on the mean of hip landmarks (23,24) "
        "(falling back to the mean of the first 33 body landmarks if the hip mean is "
        "non-finite); divide by the median-over-frames standard deviation of the first "
        "33 body landmarks (falling back to 1.0 if degenerate/non-finite); replace any "
        "remaining NaN/Inf with 0.0. Implemented verbatim in "
        "signsage_inference/preprocessing.py:normalize_pose_sequence()."
    ),
    "temporal_delta": {
        "computed_by": "model (SignTransformer.FrameProjection), not by preprocessing",
        "description": (
            "The backend must send plain normalized 1629-D frames (after "
            "prepare_live_landmarks), NOT pre-computed deltas. The model internally "
            "computes delta[t] = x[t]-x[t-1] (delta[0]=0) and concatenates "
            "[x, delta] -> 3258-D before its first linear layer."
        ),
    },
    "max_frames": int(MAX_FRAMES),
    # LANDMARK_DIM is defined later (Section 6); computed directly here from
    # FULL_FEATURE_MASK (the same underlying value) so this cell does not need
    # to run after that definition.
    "final_model_input_dim": int(len(FULL_FEATURE_MASK) * 2),
}

with open(REPORT_DIR / "landmark_contract.json", "w", encoding="utf-8") as f:
    json.dump(landmark_contract, f, indent=2, ensure_ascii=False)

print("Landmark contract written:", REPORT_DIR / "landmark_contract.json")
print(json.dumps({k: v for k, v in landmark_contract.items() if k not in ("full_feature_mask",)}, indent=2)[:1500])

knowledge_artifact("landmark_contract", "json", REPORT_DIR / "landmark_contract.json",
                   "Authoritative 543-landmark ordering/feature-flattening contract for backend MediaPipe integration.")


Verified: hip indices (23,24) fall inside pose component range {'component_name': 'POSE_LANDMARKS', 'start': 0, 'end': 33}.
Verified: the first four pose-header components are exactly ['POSE_LANDMARKS', 'FACE_LANDMARKS', 'LEFT_HAND_LANDMARKS', 'RIGHT_HAND_LANDMARKS'], matching the layout FULL_FEATURE_MASK assumes.
Components excluded from the 1629-D feature vector (start index >= 543): ['POSE_WORLD_LANDMARKS']
Landmark contract written: /kaggle/working/signsage_outputs/reports/landmark_contract.json
{
  "contract_version": "landmark_contract_v1",
  "total_landmark_count": 543,
  "raw_feature_dim": 1629,
  "coordinates_per_landmark": 3,
  "coordinate_order": [
    "x",
    "y",
    "z"
  ],
  "component_order": [
    "POSE_LANDMARKS",
    "FACE_LANDMARKS",
    "LEFT_HAND_LANDMARKS",
    "RIGHT_HAND_LANDMARKS",
    "POSE_WORLD_LANDMARKS"
  ],
  "component_order_note": "component_order lists COMPONENT NAMES, not landmark indices. Global landmark indices 0..542 are assigned by walking this

### 5a. Disk-backed normalized-pose cache

The cache stores normalized pose arrays locally after their first access. In **Kaggle mode**, this is a small working-disk cache over a remote source; it is intentionally populated lazily rather than by scanning the entire archive.


In [16]:
# ============================================================================
# POSE LOADING + NORMALIZATION + DISK CACHE
# ============================================================================
# Design goals:
#   1. Kaggle: normalized pose cache lives only in /kaggle/temp (scratch).
#   2. Local: cache persists under DATA_DIR/cache.
#   3. Training hot path operates on UID strings, not Pandas rows.
#   4. Disk cache is fp16 to reduce I/O/storage; arrays are returned as fp32.
#   5. In-memory LRU cache is bounded by CACHE_SIZE.
#   6. Raw pose data is normalized once and then reused.
#   7. Backend/live inference must continue using the same raw 1629-D
#      feature representation before any model-side delta features are added.
# ============================================================================

# Kaggle scratch is intentionally used for regenerable cache data.
# Do NOT put the multi-GB normalized cache under /kaggle/working.
CACHE_ROOT = (
    Path("/kaggle/temp/signsage_cache")
    if DATA_MODE == "kaggle"
    else DATA_DIR / "cache"
)

POSE_CACHE_DIR = CACHE_ROOT / "normalized_v1"
POSE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Pose cache dir:", POSE_CACHE_DIR)


# ----------------------------------------------------------------------------
# Local-mode UID -> .pose path
# ----------------------------------------------------------------------------

def pose_path_for_uid(uid):
    """Return the local .pose file corresponding to a UID."""
    if DATA_MODE != "local":
        raise RuntimeError(
            "pose_path_for_uid() is only valid in local mode; "
            "use read_pose_bytes_for_uid() in Kaggle mode."
        )

    path = POSE_DIR / f"{uid}.pose"

    if not path.is_file():
        raise FileNotFoundError(
            f"Missing UID-level pose file: {path}"
        )

    return path


# ----------------------------------------------------------------------------
# Cache path
# ----------------------------------------------------------------------------

def _cache_path_for_uid(uid):
    """Return the normalized-pose cache path for one UID."""
    return POSE_CACHE_DIR / f"{uid}.npy"


# ----------------------------------------------------------------------------
# Normalization diagnostics
# ----------------------------------------------------------------------------

POSE_NORMALIZATION_STATS = {
    "total_sequences": 0,
    "bad_center_sequences": 0,
    "degenerate_scale_sequences": 0,
}


# ----------------------------------------------------------------------------
# Pose normalization
# ----------------------------------------------------------------------------

def normalize_pose_sequence(flat):
    """
    Root-center and scale one pose sequence.

    Input:
        flat: [frames, 543 * 3]

    Output:
        float32 array with shape [frames, 543 * 3]

    The normalization preserves temporal motion while removing
    translation/scale variation.
    """
    POSE_NORMALIZATION_STATS["total_sequences"] += 1

    x = flat.reshape(flat.shape[0], 543, 3).astype(np.float32)

    # MediaPipe pose landmarks:
    # 23 = left hip
    # 24 = right hip
    hip = x[:, [23, 24], :]

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Mean of empty slice")
        center = np.nanmean(hip, axis=1, keepdims=True)

    bad_center = ~np.isfinite(center).all(axis=(1, 2))

    if bad_center.any():
        POSE_NORMALIZATION_STATS["bad_center_sequences"] += 1

        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message="Mean of empty slice")
            fallback_center = np.nanmean(
                x[bad_center, :33, :],
                axis=1,
                keepdims=True,
            )

        center[bad_center] = fallback_center

    x = x - center

    # Use body landmarks to estimate sequence scale.
    body = x[:, :33, :]

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Degrees of freedom")
        scale_per_frame = np.nanstd(
            body.reshape(body.shape[0], -1),
            axis=1,
        )

    valid_scale = scale_per_frame[
        np.isfinite(scale_per_frame)
        & (scale_per_frame > 1e-6)
    ]

    if valid_scale.size == 0:
        POSE_NORMALIZATION_STATS["degenerate_scale_sequences"] += 1
        scale = 1.0
    else:
        scale = np.nanmedian(valid_scale)

        if not np.isfinite(scale) or scale <= 1e-6:
            POSE_NORMALIZATION_STATS["degenerate_scale_sequences"] += 1
            scale = 1.0

    x = x / float(scale)

    # Final numerical safety.
    x = np.nan_to_num(
        x,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    return x.reshape(x.shape[0], -1).astype(np.float32)


# ----------------------------------------------------------------------------
# Temporal resizing
# ----------------------------------------------------------------------------

def resize_sequence(seq, target_len):
    """
    Resize/pad a sequence to target_len.

    Long sequences:
        uniformly sampled down to target_len.

    Short sequences:
        zero-padded at the end.
    """
    src_len = seq.shape[0]

    if src_len <= 0:
        raise ValueError("Empty pose sequence")

    if src_len == target_len:
        return seq.astype(np.float32, copy=False)

    if src_len > target_len:
        idx = np.linspace(
            0,
            src_len - 1,
            target_len,
        )

        return seq[
            np.round(idx).astype(np.int64)
        ].astype(np.float32, copy=False)

    out = np.zeros(
        (target_len, seq.shape[1]),
        dtype=np.float32,
    )

    out[:src_len] = seq

    return out


# ----------------------------------------------------------------------------
# Raw UID -> normalized pose
# ----------------------------------------------------------------------------

def _read_and_normalize_raw(uid):
    """
    Read one UID's raw .pose payload and return normalized fp32 landmarks.

    Output:
        [frames, 1629]
    """
    raw_bytes = read_pose_bytes_for_uid(uid)

    pose = Pose.read(raw_bytes)

    data = np.asarray(
        np.ma.filled(pose.body.data, 0.0),
        dtype=np.float32,
    )

    if data.ndim != 4 or data.shape[1] < 1:
        raise ValueError(
            f"Unexpected pose tensor {data.shape} for {uid}"
        )

    # Use the first detected person.
    data = data[:, 0, :, :]

    # Canonical feature contract:
    # pose 33 + face 468 + left hand 21 + right hand 21 = 543 landmarks
    data = data[:, :543, :]

    flat = data.reshape(
        data.shape[0],
        -1,
    ).astype(np.float32)

    return normalize_pose_sequence(flat)


# ----------------------------------------------------------------------------
# Cached UID loader
# ----------------------------------------------------------------------------

@lru_cache(maxsize=CACHE_SIZE)
def _read_pose_file(uid):
    """
    Load normalized pose for one UID.

    Lookup order:
        1. bounded in-memory LRU
        2. fp16 disk cache
        3. remote/local raw .pose source

    Always returns fp32.
    """

    uid = str(uid)

    # --------------------------------------------------------
    # 1. Disk cache
    # --------------------------------------------------------
    if USE_POSE_DISK_CACHE:
        cpath = _cache_path_for_uid(uid)

        if cpath.is_file():
            try:
                cached = np.load(
                    cpath,
                    allow_pickle=False,
                )

                # Ensure callers always receive fp32.
                return np.asarray(
                    cached,
                    dtype=np.float32,
                )

            except Exception:
                # Corrupt/incomplete cache entry.
                cpath.unlink(missing_ok=True)

    # --------------------------------------------------------
    # 2. Raw source
    # --------------------------------------------------------
    arr = _read_and_normalize_raw(uid)

    # --------------------------------------------------------
    # 3. Write regenerable fp16 cache
    # --------------------------------------------------------
    if USE_POSE_DISK_CACHE:
        cpath = _cache_path_for_uid(uid)
        tmp_path = cpath.with_name(
            cpath.name + ".tmp"
        )

        try:
            with open(tmp_path, "wb") as f:
                np.save(
                    f,
                    arr.astype(np.float16),
                    allow_pickle=False,
                )

            # Atomic replacement prevents partially written .npy files
            # from being treated as valid cache entries.
            os.replace(tmp_path, cpath)

        except Exception:
            # Cache failure must never break model training.
            try:
                tmp_path.unlink(missing_ok=True)
            except Exception:
                pass

    return arr


# ----------------------------------------------------------------------------
# Public training loader
# ----------------------------------------------------------------------------

def load_pose(uid, feature_mask=FULL_FEATURE_MASK, max_frames=MAX_FRAMES):
    """Load one UID using the canonical normalized landmark representation."""
    uid=str(uid)
    raw=_read_pose_file(uid)
    processed_raw=raw[:,feature_mask]
    true_len=min(len(processed_raw),max_frames)
    processed=resize_sequence(processed_raw,max_frames)
    return processed,true_len



Pose cache dir: /kaggle/temp/signsage_cache/normalized_v1


**Before running a full pass**, a rough sanity check on free disk space for the new
cache (skip if `USE_POSE_DISK_CACHE = False`):


In [17]:
# Kaggle full-corpus mode deliberately disables the multi-GB disk cache.
if USE_POSE_DISK_CACHE:
    free_bytes=shutil.disk_usage(POSE_CACHE_DIR).free
    print(f'Free space at {POSE_CACHE_DIR}: {free_bytes/1024**3:.1f} GB')
    print('Disk cache is enabled only for local mode; Kaggle full-corpus uses bounded worker RAM + archive locality.')
else:
    print('Kaggle full-corpus: normalized pose disk cache disabled to prevent multi-GB scratch growth.')


Kaggle full-corpus: normalized pose disk cache disabled to prevent multi-GB scratch growth.


### 5c. Pose cache warm-up

**Disabled by default.** The previous version prewarmed every selected UID before training. That is reasonable only when the data is already local. In Kaggle mode, prewarming thousands of remote ZIP members just moves the same work earlier and makes the notebook appear stuck.

Training now fetches pose members lazily and shows progress. Set `PREWARM_CACHE = True` only if you deliberately want to populate the selected sample before training.


In [18]:
if PREWARM_CACHE and USE_POSE_DISK_CACHE:
    _prewarm_uids = pd.concat([df_train["uid"], df_val["uid"], df_test["uid"]]).unique()
    _already_cached = sum(1 for u in _prewarm_uids if _cache_path_for_uid(u).is_file())
    print(f"{len(_prewarm_uids):,} unique selected rows; {_already_cached:,} already cached.")
    for uid in tqdm(_prewarm_uids, desc="Prewarming selected pose cache"):
        _read_pose_file(uid)
    print("Selected-cache prewarm complete.")
else:
    print("Lazy pose loading enabled — no full prewarm will run.")


Lazy pose loading enabled — no full prewarm will run.


In [19]:
# ============================================================
# 6. Optimized Dataset / DataLoader
# ============================================================
class SignToTextDataset(Dataset):
    """UID-only hot path; no Pandas access during training."""
    def __init__(self,frame,feature_mask=FULL_FEATURE_MASK):
        frame=frame.reset_index(drop=True)
        self.feature_mask=feature_mask
        self.uids=frame["uid"].astype(str).to_numpy(copy=True)
        self.texts=frame["text"].astype(str).to_numpy(copy=True)
        print(f"Pre-tokenizing {len(self.texts):,} captions...")
        self.token_ids=[np.asarray(encode_text(t),dtype=np.int64) for t in tqdm(self.texts,desc="Tokenizing",leave=False)]
        self.archive_offsets=None
        if DATA_MODE=="kaggle" and REMOTE_BATCH_LOCALITY:
            print("Building ZIP-member locality index from the central directory...")
            offsets=np.empty(len(self.uids),dtype=np.int64)
            store=_get_active_remote_pose_store()
            for i,uid in enumerate(tqdm(self.uids,desc="Indexing pose offsets",leave=False)):
                offsets[i]=int(store.info_for_uid(uid).header_offset)
            self.archive_offsets=offsets
        print(f"Dataset ready: {len(self.uids):,} samples")
    def __len__(self): return len(self.uids)
    def __getitem__(self,idx):
        uid=self.uids[idx]
        frames,length=load_pose(uid,self.feature_mask)
        tokens=self.token_ids[idx]
        return torch.from_numpy(np.ascontiguousarray(frames)),torch.tensor(length,dtype=torch.long),torch.from_numpy(tokens.copy()).long()

def collate_sign_text(batch):
    frames,lengths,tokens=zip(*batch)
    frames=torch.stack(frames); lengths=torch.stack(lengths)
    max_tok=max(len(t) for t in tokens)
    tok_pad=torch.full((len(tokens),max_tok),PAD_IDX,dtype=torch.long)
    for i,t in enumerate(tokens): tok_pad[i,:len(t)]=t
    return frames,lengths,tok_pad[:,:-1],tok_pad[:,1:]

class ArchiveLocalBatchSampler(Sampler):
    """Shuffle batch order while preserving ZIP-local sample order inside each batch."""
    def __init__(self,dataset,batch_size,shuffle_batches=True,seed=SEED,drop_last=False):
        self.dataset=dataset; self.batch_size=int(batch_size); self.shuffle_batches=shuffle_batches; self.seed=int(seed); self.drop_last=drop_last; self.epoch=0
        order=np.argsort(dataset.archive_offsets,kind="stable") if dataset.archive_offsets is not None else np.arange(len(dataset))
        self.batches=[]
        for start in range(0,len(order),self.batch_size):
            b=order[start:start+self.batch_size].tolist()
            if len(b)<self.batch_size and drop_last: continue
            self.batches.append(b)
    def set_epoch(self,epoch): self.epoch=int(epoch)
    def __iter__(self):
        ids=np.arange(len(self.batches))
        if self.shuffle_batches: np.random.default_rng(self.seed+self.epoch).shuffle(ids)
        for i in ids: yield self.batches[int(i)]
    def __len__(self): return len(self.batches)

def make_train_loader(ds):
    kwargs=dict(num_workers=NUM_WORKERS,pin_memory=PIN_MEMORY and DEVICE.type=="cuda")
    if NUM_WORKERS>0:
        kwargs.update(persistent_workers=PERSISTENT_WORKERS,prefetch_factor=PREFETCH_FACTOR)
    if DATA_MODE=="kaggle" and REMOTE_BATCH_LOCALITY:
        return DataLoader(ds,batch_sampler=ArchiveLocalBatchSampler(ds,BATCH_SIZE,True),collate_fn=collate_sign_text,**kwargs)
    return DataLoader(ds,batch_size=BATCH_SIZE,shuffle=True,collate_fn=collate_sign_text,**kwargs)

def make_eval_loader(ds):
    kwargs=dict(num_workers=EVAL_NUM_WORKERS,pin_memory=PIN_MEMORY and DEVICE.type=="cuda")
    if DATA_MODE=="kaggle" and REMOTE_BATCH_LOCALITY:
        return DataLoader(ds,batch_sampler=ArchiveLocalBatchSampler(ds,BATCH_SIZE,shuffle_batches=False),collate_fn=collate_sign_text,**kwargs)
    return DataLoader(ds,batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate_sign_text,**kwargs)

train_ds_full=SignToTextDataset(df_train,FULL_FEATURE_MASK)
val_ds_full=SignToTextDataset(df_val,FULL_FEATURE_MASK)
test_ds_full=SignToTextDataset(df_test,FULL_FEATURE_MASK)
train_loader_full=make_train_loader(train_ds_full)
val_loader_full=make_eval_loader(val_ds_full)
test_loader_full=make_eval_loader(test_ds_full)

if RUN_NOFACE_ABLATION:
    train_ds_noface=SignToTextDataset(df_train,NOFACE_FEATURE_MASK); val_ds_noface=SignToTextDataset(df_val,NOFACE_FEATURE_MASK); test_ds_noface=SignToTextDataset(df_test,NOFACE_FEATURE_MASK)
    train_loader_noface=make_train_loader(train_ds_noface); val_loader_noface=make_eval_loader(val_ds_noface); test_loader_noface=make_eval_loader(test_ds_noface)
else:
    train_ds_noface=val_ds_noface=test_ds_noface=None; train_loader_noface=val_loader_noface=test_loader_noface=None

LANDMARK_DIM=len(FULL_FEATURE_MASK); NOFACE_DIM=len(NOFACE_FEATURE_MASK)
print("LANDMARK_DIM:",LANDMARK_DIM,"NOFACE_DIM:",NOFACE_DIM)
print("BATCH_SIZE:",BATCH_SIZE,"NUM_WORKERS:",NUM_WORKERS,"REMOTE_BATCH_LOCALITY:",REMOTE_BATCH_LOCALITY)


Pre-tokenizing 99,699 captions...


Tokenizing:   0%|          | 0/99699 [00:00<?, ?it/s]

Building ZIP-member locality index from the central directory...


Indexing pose offsets:   0%|          | 0/99699 [00:00<?, ?it/s]

Dataset ready: 99,699 samples
Pre-tokenizing 11,927 captions...


Tokenizing:   0%|          | 0/11927 [00:00<?, ?it/s]

... (3 outputs left out)

Tokenizing:   0%|          | 0/12153 [00:00<?, ?it/s]

Building ZIP-member locality index from the central directory...


Indexing pose offsets:   0%|          | 0/12153 [00:00<?, ?it/s]

Dataset ready: 12,153 samples
LANDMARK_DIM: 1629 NOFACE_DIM: 225
BATCH_SIZE: 64 NUM_WORKERS: 2 REMOTE_BATCH_LOCALITY: True


In [20]:
# ============================================================
# Section 4a — Reproducibility / experiment configuration
# ============================================================
config_observation={
    "seed":SEED,"run_mode":RUN_MODE,"data_mode":DATA_MODE,"device":str(DEVICE),"is_kaggle":IS_KAGGLE,
    "primary_sign_to_text":RUN_PRIMARY_S2T,
    "full_usable_dataset_requested":TRAIN_LIMIT is None and VAL_LIMIT is None and TEST_LIMIT is None,
    "train_limit":TRAIN_LIMIT,"val_limit":VAL_LIMIT,"test_limit":TEST_LIMIT,
    "epochs_transformer_requested":N_EPOCHS_TRANSFORMER,"min_epochs_transformer":MIN_EPOCHS_TRANSFORMER,
    "max_epochs_transformer":MAX_EPOCHS_TRANSFORMER,"train_time_budget_hours":TRAIN_TIME_BUDGET_HOURS,
    "session_safety_margin_minutes":SESSION_SAFETY_MARGIN_MINUTES,
    "transformer_peak_lr":TRANSFORMER_PEAK_LR,"transformer_warmup_fraction":TRANSFORMER_WARMUP_FRACTION,
    "transformer_min_lr_ratio":TRANSFORMER_MIN_LR_RATIO,"max_tokens":MAX_TOKENS,"max_frames":MAX_FRAMES,
    "target_vocab_size":TARGET_VOCAB_SIZE,"batch_size":BATCH_SIZE,"grad_accum_steps":GRAD_ACCUM_STEPS,
    "num_workers":NUM_WORKERS,"prefetch_factor":PREFETCH_FACTOR,"persistent_workers":PERSISTENT_WORKERS,
    "pin_memory":PIN_MEMORY,"remote_batch_locality":REMOTE_BATCH_LOCALITY,"remote_chunk_size_mb":REMOTE_CHUNK_SIZE_MB,
    "remote_cache_chunks":REMOTE_CACHE_CHUNKS,"cache_size":CACHE_SIZE,"use_pose_disk_cache":USE_POSE_DISK_CACHE,
    "prewarm_cache":PREWARM_CACHE,"eval_num_workers":EVAL_NUM_WORKERS,"use_amp":USE_AMP,
    "run_gru_baseline":RUN_GRU_BASELINE,"run_noface_ablation":RUN_NOFACE_ABLATION,
    "run_text_to_pose":RUN_TEXT_TO_POSE,"run_retrieval_baseline":RUN_RETRIEVAL_BASELINE,
    "beam_width":BEAM_WIDTH,"repetition_penalty":REPETITION_PENALTY,"no_repeat_ngram_size":NO_REPEAT_NGRAM_SIZE,
    "max_decode_tokens":MAX_DECODE_TOKENS,"resume_from_recovery":RESUME_FROM_RECOVERY,
    "run_tiny_overfit_test":RUN_TINY_OVERFIT_TEST,"tiny_overfit_n":TINY_OVERFIT_N,"tiny_overfit_steps":TINY_OVERFIT_STEPS,
}
knowledge_observation("configuration",config_observation)
token_vocabulary_path=REPORT_DIR/"token_vocabulary.json"
with open(token_vocabulary_path,"w",encoding="utf-8") as f:
    json.dump({"token_to_id":vocab,"special_tokens":SPECIALS},f,indent=2,ensure_ascii=False)
knowledge_artifact("token_vocabulary","json",token_vocabulary_path,"Sign-to-text token mapping used by the primary experiment.")
print("Configuration recorded. Full corpus requested:",config_observation["full_usable_dataset_requested"])


Configuration recorded. Full corpus requested: True


## 6a. One-batch smoke check

This catches the exact class of failure that previously appeared as a delayed `FileNotFoundError` or undefined dimension.

In [21]:
batch = next(iter(train_loader_full))
frames_b, lengths_b, tok_in_b, tok_out_b = batch
print("frames:", tuple(frames_b.shape))
print("lengths:", lengths_b.tolist())
print("tok_in:", tuple(tok_in_b.shape))
print("tok_out:", tuple(tok_out_b.shape))
assert frames_b.shape[-1] == LANDMARK_DIM
assert torch.isfinite(frames_b).all()
print("DataLoader smoke check passed.")


frames: (64, 160, 1629)
lengths: [106, 157, 93, 121, 45, 160, 160, 160, 160, 64, 160, 56, 132, 160, 160, 160, 71, 158, 160, 95, 91, 84, 160, 148, 160, 160, 160, 160, 50, 145, 55, 101, 101, 160, 160, 89, 76, 160, 117, 160, 160, 160, 118, 160, 153, 160, 160, 160, 74, 90, 160, 160, 160, 136, 160, 41, 157, 148, 46, 160, 160, 160, 83, 160]
tok_in: (64, 35)
tok_out: (64, 35)
DataLoader smoke check passed.


In [22]:
# ============================================================
# 6a. Training pipeline throughput gate
# ============================================================
def benchmark_loader(loader,n_batches=PERF_PREFLIGHT_BATCHES,warmup_batches=PERF_WARMUP_BATCHES):
    global DATA_LOADER_MEAN_SEC_PER_BATCH,DATA_LOADER_P90_SEC_PER_BATCH,DATA_LOADER_SAMPLES_PER_SEC,DATA_LOADER_EPOCH_HOURS
    print("="*70); print("DATA PIPELINE THROUGHPUT BENCHMARK"); print("="*70)
    print(f"samples={len(loader.dataset):,} batches={len(loader):,} batch_size={BATCH_SIZE} workers={NUM_WORKERS} locality={REMOTE_BATCH_LOCALITY}")
    it=iter(loader)
    for _ in range(min(warmup_batches,len(loader))):
        try:
            _warm_batch=next(it); del _warm_batch
        except StopIteration: break
    times=[]; samples=0
    for i in range(min(n_batches,max(1,len(loader)-warmup_batches))):
        t0=time.perf_counter()
        try: batch=next(it)
        except StopIteration: break
        elapsed=time.perf_counter()-t0
        frames,lengths,tok_in,tok_out=batch; bs=int(frames.shape[0])
        times.append(elapsed); samples+=bs
        print(f"batch {i+1:02d}: {elapsed:.3f}s | samples={bs} | frames={tuple(frames.shape)}")
        del batch,frames,lengths,tok_in,tok_out
    if not times: raise RuntimeError("No batches measured in throughput benchmark.")
    DATA_LOADER_MEAN_SEC_PER_BATCH=float(np.mean(times)); DATA_LOADER_P90_SEC_PER_BATCH=float(np.percentile(times,90))
    DATA_LOADER_SAMPLES_PER_SEC=float(samples/max(1e-9,sum(times))); DATA_LOADER_EPOCH_HOURS=float(len(loader)*DATA_LOADER_MEAN_SEC_PER_BATCH/3600.0)
    print(f"Mean={DATA_LOADER_MEAN_SEC_PER_BATCH:.3f}s P90={DATA_LOADER_P90_SEC_PER_BATCH:.3f}s samples/s={DATA_LOADER_SAMPLES_PER_SEC:.2f} loader-floor={DATA_LOADER_EPOCH_HOURS:.2f}h")
    if DATA_LOADER_MEAN_SEC_PER_BATCH>PERF_MAX_SECONDS_PER_BATCH: print("WARNING: above reference; locality optimization remains active.")
    del it; gc.collect()
    if DEVICE.type=="cuda": torch.cuda.empty_cache()
    return DATA_LOADER_MEAN_SEC_PER_BATCH
benchmark_loader(train_loader_full)


DATA PIPELINE THROUGHPUT BENCHMARK
samples=99,699 batches=1,558 batch_size=64 workers=2 locality=True
batch 01: 8.705s | samples=64 | frames=(64, 160, 1629)
batch 02: 6.424s | samples=64 | frames=(64, 160, 1629)
batch 03: 6.596s | samples=64 | frames=(64, 160, 1629)
batch 04: 0.000s | samples=64 | frames=(64, 160, 1629)
batch 05: 3.755s | samples=64 | frames=(64, 160, 1629)
batch 06: 1.223s | samples=64 | frames=(64, 160, 1629)
batch 07: 8.020s | samples=64 | frames=(64, 160, 1629)
batch 08: 0.000s | samples=64 | frames=(64, 160, 1629)
Mean=4.341s P90=8.226s samples/s=14.74 loader-floor=1.88h


4.34051907837501

In [23]:
# ============================================================
# 6b. GPU training-step benchmark helper
# ============================================================
def benchmark_training_step(model,loader,n_batches=GPU_BENCHMARK_BATCHES):
    global GPU_MEAN_SEC_PER_BATCH
    model.train(); optimizer=torch.optim.AdamW(model.parameters(),lr=TRANSFORMER_PEAK_LR,weight_decay=1e-4); criterion=nn.CrossEntropyLoss(ignore_index=PAD_IDX)
    it=iter(loader); times=[]
    # warmup
    frames,lengths,tok_in,tok_out=next(it)
    frames=frames.to(DEVICE,non_blocking=True); lengths=lengths.to(DEVICE,non_blocking=True); tok_in=tok_in.to(DEVICE,non_blocking=True); tok_out=tok_out.to(DEVICE,non_blocking=True)
    optimizer.zero_grad(set_to_none=True)
    with autocast_ctx(): logits=model(frames,padding_mask(lengths,frames.size(1)),tok_in,causal_mask(tok_in.size(1))); loss=criterion(logits.reshape(-1,logits.size(-1)),tok_out.reshape(-1))
    scaler.scale(loss).backward(); scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(optimizer); scaler.update()
    if DEVICE.type=="cuda": torch.cuda.synchronize()
    del frames,lengths,tok_in,tok_out,logits,loss
    for i in range(n_batches):
        try: frames,lengths,tok_in,tok_out=next(it)
        except StopIteration: break
        frames=frames.to(DEVICE,non_blocking=True); lengths=lengths.to(DEVICE,non_blocking=True); tok_in=tok_in.to(DEVICE,non_blocking=True); tok_out=tok_out.to(DEVICE,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if DEVICE.type=="cuda": torch.cuda.synchronize()
        t0=time.perf_counter()
        with autocast_ctx(): logits=model(frames,padding_mask(lengths,frames.size(1)),tok_in,causal_mask(tok_in.size(1))); loss=criterion(logits.reshape(-1,logits.size(-1)),tok_out.reshape(-1))
        scaler.scale(loss).backward(); scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(optimizer); scaler.update()
        if DEVICE.type=="cuda": torch.cuda.synchronize()
        times.append(time.perf_counter()-t0); print(f"GPU batch {i+1:02d}: {times[-1]:.3f}s")
        del frames,lengths,tok_in,tok_out,logits,loss
    if not times: raise RuntimeError("GPU benchmark measured no batches.")
    GPU_MEAN_SEC_PER_BATCH=float(np.mean(times)); print(f"Mean GPU compute={GPU_MEAN_SEC_PER_BATCH:.3f}s/batch")
    del it,optimizer; gc.collect()
    if DEVICE.type=="cuda": torch.cuda.empty_cache()
    return GPU_MEAN_SEC_PER_BATCH


## 7. Sign → Text: GRU baseline

This is intentionally simple. It is a sanity baseline, not the final model.

The important criterion is not whether it reaches a strong score; it is whether the training signal is learnable without immediately overfitting into nonsense.

In [24]:
## 7. Optional GRU baseline
# Disabled by default for the full run so Kaggle compute is spent on the actual research target.
if RUN_GRU_BASELINE:
    class GRUEncoder(nn.Module):
        def __init__(self,input_dim,hidden_dim=192):
            super().__init__(); self.gru=nn.GRU(input_dim,hidden_dim,batch_first=True,bidirectional=True); self.proj=nn.Linear(hidden_dim*2,hidden_dim)
        def forward(self,x,lengths):
            packed=nn.utils.rnn.pack_padded_sequence(x,lengths.cpu().clamp(min=1),batch_first=True,enforce_sorted=False)
            _,h=self.gru(packed); h=torch.cat([h[0],h[1]],dim=-1); return self.proj(h).unsqueeze(0)
    class GRUDecoder(nn.Module):
        def __init__(self,vocab_size,hidden_dim=192):
            super().__init__(); self.embed=nn.Embedding(vocab_size,hidden_dim,padding_idx=PAD_IDX); self.gru=nn.GRU(hidden_dim,hidden_dim,batch_first=True); self.out=nn.Linear(hidden_dim,vocab_size)
        def forward(self,tokens,hidden):
            out,hidden=self.gru(self.embed(tokens),hidden); return self.out(out),hidden
    class Seq2SeqGRU(nn.Module):
        def __init__(self,input_dim,vocab_size,hidden_dim=192):
            super().__init__(); self.encoder=GRUEncoder(input_dim,hidden_dim); self.decoder=GRUDecoder(vocab_size,hidden_dim)
        def forward(self,frames,lengths,target_in):
            return self.decoder(target_in,self.encoder(frames,lengths))[0]
    @torch.no_grad()
    def eval_loss_gru(model,loader,criterion):
        model.eval(); total=0.0
        for frames,lengths,tin,tout in tqdm(loader,desc="GRU eval",leave=False):
            frames,lengths,tin,tout=(frames.to(DEVICE,non_blocking=True),
                                     lengths.to(DEVICE,non_blocking=True),
                                     tin.to(DEVICE,non_blocking=True),
                                     tout.to(DEVICE,non_blocking=True))
            with autocast_ctx():
                logits=model(frames,lengths,tin)
                total += criterion(logits.reshape(-1,len(vocab)),tout.reshape(-1)).item()
        return total/max(1,len(loader))

    def train_epoch_gru(model,loader,opt,criterion):
        model.train(); total=0
        for frames,lengths,tin,tout in tqdm(loader,desc="GRU train",leave=False):
            frames,lengths,tin,tout=frames.to(DEVICE),lengths.to(DEVICE),tin.to(DEVICE),tout.to(DEVICE)
            with autocast_ctx(): loss=criterion(model(frames,lengths,tin).reshape(-1,len(vocab)),tout.reshape(-1))
            scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); total+=loss.item()
        return total/max(1,len(loader))
    @torch.no_grad()
    def batch_greedy_gru(model,loader):
        model.eval(); refs=[]; hyps=[]; confs=[]
        for frames,lengths,tin,tout in tqdm(loader,desc="GRU test",leave=False):
            frames,lengths=frames.to(DEVICE,non_blocking=True),lengths.to(DEVICE,non_blocking=True)
            memory=model.encoder(frames,lengths)
            ids=torch.full((frames.size(0),1),BOS_IDX,dtype=torch.long,device=DEVICE)
            finished=torch.zeros(frames.size(0),dtype=torch.bool,device=DEVICE)
            sum_logp=torch.zeros(frames.size(0),device=DEVICE); steps=torch.zeros(frames.size(0),device=DEVICE)
            generated=[]
            hidden=memory
            for step in range(MAX_DECODE_TOKENS):
                logits,hidden=model.decoder(ids[:,-1:],hidden)
                logits=logits[:,-1].float()
                logits[:,PAD_IDX]=-torch.inf; logits[:,BOS_IDX]=-torch.inf; logits[:,UNK_IDX]=-torch.inf
                if step < MIN_OUTPUT_TOKENS: logits[:,EOS_IDX]=-torch.inf
                lp=torch.log_softmax(logits,dim=-1); prob,nxt=torch.max(lp,dim=-1)
                active=~finished; sum_logp += torch.where(active,prob,torch.zeros_like(prob)); steps += active
                nxt=torch.where(finished,torch.full_like(nxt,EOS_IDX),nxt)
                generated.append(nxt.cpu().numpy()); finished |= nxt.eq(EOS_IDX)
                ids=torch.cat([ids,nxt[:,None]],dim=1)
                if finished.all(): break
            gen=np.stack(generated,axis=1)
            conf=torch.exp(sum_logp/steps.clamp(min=1)).cpu().numpy()
            for r,row in enumerate(tout.cpu().tolist()):
                refs.append(decode_ids(row)); hyps.append(decode_ids(gen[r].tolist())); confs.append(float(conf[r]))
        return refs,hyps,confs

    gru_model=Seq2SeqGRU(LANDMARK_DIM,len(vocab)).to(DEVICE)
    gru_opt=torch.optim.AdamW(gru_model.parameters(),lr=1e-3,weight_decay=1e-4)
    criterion_gru=nn.CrossEntropyLoss(ignore_index=PAD_IDX)
    gru_history=[]; best=float("inf")
    for epoch in range(1,max(1,N_EPOCHS_GRU)+1):
        tr=train_epoch_gru(gru_model,train_loader_full,gru_opt,criterion_gru)
        va=eval_loss_gru(gru_model,val_loader_full,criterion_gru)
        gru_history.append((epoch,tr,va)); print(f"[GRU] {epoch}: train={tr:.4f} val={va:.4f}")
        if va<best: best=va; torch.save(gru_model.state_dict(),CHECKPOINT_DIR/"signsage_gru_best.pt")
    gru_model.load_state_dict(torch.load(CHECKPOINT_DIR/"signsage_gru_best.pt",map_location=DEVICE))
else:
    gru_model=None; gru_history=[]
    print("GRU baseline skipped; primary compute is reserved for Sign→Text Transformer.")


GRU baseline skipped; primary compute is reserved for Sign→Text Transformer.


In [25]:
# 7a. Record optional GRU experiment.
if RUN_GRU_BASELINE:
    _GRU_NAME="GRU baseline"; gru_ckpt=CHECKPOINT_DIR/"signsage_gru_best.pt"
    gru_best_val=float(min(v for _,_,v in gru_history))
    knowledge_insert("experiments",
        ["run_id","name","family","input_features","epochs","batch_size","learning_rate","weight_decay","checkpoint_path","best_validation_loss"],
        [RUN_ID,_GRU_NAME,"Sign→Text","full_pose",N_EPOCHS_GRU,BATCH_SIZE,1e-3,1e-4,str(gru_ckpt),gru_best_val],
        delete_keys={"run_id":RUN_ID,"name":_GRU_NAME})
    for epoch,tr_loss,val_loss in gru_history:
        knowledge_metric(_GRU_NAME,f"train_loss_epoch_{epoch}",tr_loss,"train")
        knowledge_metric(_GRU_NAME,f"val_loss_epoch_{epoch}",val_loss,"validation")
    knowledge_artifact(_GRU_NAME,"checkpoint",gru_ckpt,"Optional GRU Sign→Text checkpoint.")
else:
    print("GRU metadata skipped.")


GRU metadata skipped.


## 8. Sign → Text: spatial-temporal Transformer

The encoder sees the full landmark vector at every frame; temporal self-attention then relates frames across the sequence. This is the plain Transformer requested by the project decision log; GAT is intentionally not introduced here.

In [26]:
## 8. Primary model: spatial-temporal Transformer

class FrameProjection(nn.Module):
    """Project spatial landmarks plus explicit frame-to-frame motion features."""
    def __init__(self,input_dim,d_model):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(input_dim,384),
            nn.LayerNorm(384),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(384,d_model),
            nn.LayerNorm(d_model),
        )
        self.temporal=nn.Conv1d(d_model,d_model,kernel_size=5,padding=2,groups=d_model)
    def forward(self,x):
        # Explicit velocity is zero for the first frame. This makes the temporal
        # signal available before attention rather than asking the encoder to infer
        # all derivatives from absolute coordinates alone.
        delta = torch.zeros_like(x)
        delta[:,1:] = x[:,1:] - x[:,:-1]
        y=self.net(torch.cat([x, delta], dim=-1))
        y=self.temporal(y.transpose(1,2)).transpose(1,2)
        return y

class PositionalEncoding(nn.Module):
    def __init__(self,d_model,max_len):
        super().__init__()
        pe=torch.zeros(max_len,d_model)
        pos=torch.arange(max_len,dtype=torch.float32).unsqueeze(1)
        div=torch.exp(torch.arange(0,d_model,2,dtype=torch.float32)*(-np.log(10000.0)/d_model))
        pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer("pe",pe.unsqueeze(0))
    def forward(self,x): return x+self.pe[:,:x.size(1)]

class SignTransformer(nn.Module):
    def __init__(self,input_dim,vocab_size,d_model=192,nhead=4,layers=3,ff=512,dropout=0.15):
        super().__init__()
        self.input_proj=FrameProjection(input_dim,d_model)
        self.src_pos=PositionalEncoding(d_model,MAX_FRAMES+4)
        self.tok_embed=nn.Embedding(vocab_size,d_model,padding_idx=PAD_IDX)
        self.tgt_pos=PositionalEncoding(d_model,MAX_TOKENS+4)
        enc=nn.TransformerEncoderLayer(d_model,nhead,ff,dropout,batch_first=True,norm_first=False)
        dec=nn.TransformerDecoderLayer(d_model,nhead,ff,dropout,batch_first=True,norm_first=False)
        self.encoder=nn.TransformerEncoder(enc,layers)
        self.decoder=nn.TransformerDecoder(dec,layers)
        self.out=nn.Linear(d_model,vocab_size)
        self.scale=d_model**0.5
    def encode(self,frames,src_pad):
        return self.encoder(self.src_pos(self.input_proj(frames)),src_key_padding_mask=src_pad)
    def decode(self,tokens,memory,tgt_mask,src_pad,tgt_pad=None):
        x=self.tgt_pos(self.tok_embed(tokens)*self.scale)
        return self.out(self.decoder(
            x,memory,tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_pad,
            memory_key_padding_mask=src_pad,
        ))
    def forward(self,frames,src_pad,tokens,tgt_mask):
        return self.decode(
            tokens,self.encode(frames,src_pad),tgt_mask,src_pad,
            tgt_pad=tokens.eq(PAD_IDX)
        )

def padding_mask(lengths,max_len):
    return torch.arange(max_len,device=lengths.device).unsqueeze(0)>=lengths.unsqueeze(1)

def causal_mask(n):
    return torch.triu(torch.ones(n,n,device=DEVICE,dtype=torch.bool),diagonal=1)

def train_epoch_transformer(model,loader,opt,criterion,epoch=0,scheduler=None):
    model.train(); total=0.0; opt.zero_grad(set_to_none=True)
    if hasattr(getattr(loader, 'batch_sampler', None), 'set_epoch'):
        loader.batch_sampler.set_epoch(epoch)
    for i,(frames,lengths,tin,tout) in enumerate(tqdm(loader,desc="Transformer train",leave=False)):
        frames,lengths,tin,tout=(frames.to(DEVICE, non_blocking=True), lengths.to(DEVICE, non_blocking=True), tin.to(DEVICE, non_blocking=True), tout.to(DEVICE, non_blocking=True))
        src_pad=padding_mask(lengths,frames.size(1)); tgt_mask=causal_mask(tin.size(1))
        with autocast_ctx():
            logits=model(frames,src_pad,tin,tgt_mask)
            loss=criterion(logits.reshape(-1,logits.size(-1)),tout.reshape(-1))
        if not torch.isfinite(loss): raise RuntimeError("Non-finite Transformer loss.")
        scaler.scale(loss/GRAD_ACCUM_STEPS).backward()
        if (i+1)%GRAD_ACCUM_STEPS==0 or i==len(loader)-1:
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            # Step the LR schedule once per real optimizer step (not per micro-batch),
            # so warmup/decay progress matches actual gradient updates under GRAD_ACCUM_STEPS.
            if scheduler is not None: scheduler.step()
        total+=loss.item()
        if (i + 1) % 250 == 0:
            # Network-backed datasets can fail transiently. Keep the most recent
            # optimizer/scheduler/model state so a completed portion of an epoch is not lost.
            torch.save({
                "state_dict": model.state_dict(),
                "optimizer_state_dict": opt.state_dict(),
                "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
                "epoch": epoch,
                "batch": i + 1,
                "model_config": {
                    "input_dim": model.input_proj.net[0].in_features,
                    "raw_feature_dim": model.input_proj.net[0].in_features//2,
                    "vocab_size": len(vocab),
                    "d_model": 192, "nhead": 4, "layers": 3, "ff": 512,
                    "dropout": 0.15, "max_frames": MAX_FRAMES, "max_tokens": MAX_TOKENS,
                },
                "vocab": vocab,
            }, CHECKPOINT_DIR / "signsage_transformer_recovery.pt")
    return total/max(1,len(loader))

@torch.no_grad()
def eval_loss_transformer(model,loader,criterion):
    model.eval(); total=0.0
    for frames,lengths,tin,tout in tqdm(loader,desc="Transformer eval",leave=False):
        frames,lengths,tin,tout=(frames.to(DEVICE, non_blocking=True), lengths.to(DEVICE, non_blocking=True), tin.to(DEVICE, non_blocking=True), tout.to(DEVICE, non_blocking=True))
        with autocast_ctx():
            logits=model(frames,padding_mask(lengths,frames.size(1)),tin,causal_mask(tin.size(1)))
            total+=criterion(logits.reshape(-1,logits.size(-1)),tout.reshape(-1)).item()
    return total/max(1,len(loader))

def _ban_repeated_ngram(logits,ids,n):
    if n<=1 or len(ids)<n-1: return logits
    prefix=tuple(ids[-(n-1):])
    for j in range(len(ids)-n+1):
        ng=tuple(ids[j:j+n])
        if ng[:-1]==prefix: logits[ng[-1]]=-torch.inf
    return logits

@torch.no_grad()
def constrained_greedy_transformer(model,frames,length,return_confidence=False):
    model.eval(); frames=frames.unsqueeze(0).to(DEVICE)
    length=torch.tensor([int(length)],device=DEVICE)
    src_pad=padding_mask(length,frames.size(1)); memory=model.encode(frames,src_pad)
    ids=[BOS_IDX]; logps=[]
    for step in range(MAX_DECODE_TOKENS):
        tok=torch.tensor([ids],device=DEVICE)
        logits=model.decode(tok,memory,causal_mask(tok.size(1)),src_pad)[0,-1].float()
        logits[PAD_IDX]=-torch.inf; logits[BOS_IDX]=-torch.inf; logits[UNK_IDX]=-torch.inf
        if step < MIN_OUTPUT_TOKENS: logits[EOS_IDX]=-torch.inf
        logits=_ban_repeated_ngram(logits,ids,NO_REPEAT_NGRAM_SIZE)
        # Mild repetition penalty; do not use an aggressive penalty that can distort fluent English.
        for tid in set(ids[1:]):
            logits[tid]=logits[tid]/REPETITION_PENALTY if logits[tid]>=0 else logits[tid]*REPETITION_PENALTY
        lp=torch.log_softmax(logits,dim=-1)
        prob,tid=torch.max(lp,dim=-1)
        tid=int(tid); logps.append(float(prob))
        ids.append(tid)
        if tid==EOS_IDX: break
    text=decode_ids(ids[1:])
    conf=float(np.exp(np.mean(logps))) if logps else 0.0
    return (text,conf) if return_confidence else text

@torch.no_grad()
def batch_greedy_transformer(model,loader):
    """Fast full-test decoder with batch-safe constrained greedy decoding."""
    model.eval(); refs=[]; hyps=[]; confs=[]
    for frames,lengths,tin,tout in tqdm(loader,desc="Transformer test",leave=False):
        frames,lengths=frames.to(DEVICE,non_blocking=True),lengths.to(DEVICE,non_blocking=True)
        src_pad=padding_mask(lengths,frames.size(1))
        memory=model.encode(frames,src_pad)
        ids=torch.full((frames.size(0),1),BOS_IDX,dtype=torch.long,device=DEVICE)
        finished=torch.zeros(frames.size(0),dtype=torch.bool,device=DEVICE)
        sum_logp=torch.zeros(frames.size(0),dtype=torch.float32,device=DEVICE)
        active_steps=torch.zeros(frames.size(0),dtype=torch.long,device=DEVICE)
        generated=[]
        for step in range(MAX_DECODE_TOKENS):
            logits=model.decode(ids,memory,causal_mask(ids.size(1)),src_pad)[:,-1].float()
            logits[:,PAD_IDX]=-torch.inf; logits[:,BOS_IDX]=-torch.inf; logits[:,UNK_IDX]=-torch.inf
            if step < MIN_OUTPUT_TOKENS: logits[:,EOS_IDX]=-torch.inf
            lp=torch.log_softmax(logits,dim=-1)
            prob,nxt=torch.max(lp,dim=-1)
            # Only accumulate confidence for sequences that were still active.
            active=~finished
            sum_logp += torch.where(active,prob,torch.zeros_like(prob))
            active_steps += active.long()
            nxt=torch.where(finished,torch.full_like(nxt,EOS_IDX),nxt)
            generated.append(nxt.detach().cpu().numpy())
            finished |= nxt.eq(EOS_IDX)
            ids=torch.cat([ids,nxt[:,None]],dim=1)
            if finished.all(): break
        gen=np.stack(generated,axis=1)
        conf=torch.exp(sum_logp/active_steps.clamp(min=1)).detach().cpu().numpy()
        for r,row in enumerate(tout.cpu().tolist()):
            refs.append(decode_ids(row))
            hyps.append(decode_ids(gen[r].tolist()))
            confs.append(float(conf[r]))
    return refs,hyps,confs

def beam_transformer(model,frames,length,beam_width=BEAM_WIDTH):
    # Used only for qualitative inspection, not full-test metrics.
    model.eval(); frames=frames.unsqueeze(0).to(DEVICE); length=torch.tensor([int(length)],device=DEVICE)
    memory=model.encode(frames,padding_mask(length,frames.size(1)))
    beams=[(0.0,[BOS_IDX])]
    for step in range(MAX_DECODE_TOKENS):
        cand=[]
        for score,ids in beams:
            tok=torch.tensor([ids],device=DEVICE)
            logits=model.decode(tok,memory,causal_mask(tok.size(1)),padding_mask(length,frames.size(1)))[0,-1].float()
            logits[PAD_IDX]=-torch.inf; logits[BOS_IDX]=-torch.inf; logits[UNK_IDX]=-torch.inf
            if step<MIN_OUTPUT_TOKENS: logits[EOS_IDX]=-torch.inf
            logits=_ban_repeated_ngram(logits,ids,NO_REPEAT_NGRAM_SIZE)
            lp=torch.log_softmax(logits,dim=-1)
            vals,tids=torch.topk(lp,beam_width)
            for v,t in zip(vals.tolist(),tids.tolist()):
                new=ids+[int(t)]
                cand.append((score+float(v),new))
        cand.sort(key=lambda z:z[0]/max(1,len(z[1])-1),reverse=True)
        beams=cand[:beam_width]
        if all(x[1][-1]==EOS_IDX for x in beams): break
    best=max(beams,key=lambda z:z[0]/max(1,len(z[1])-1))
    return decode_ids(best[1][1:])

def build_transformer_lr_schedule(optimizer, total_steps):
    """Linear warmup followed by cosine decay to TRANSFORMER_MIN_LR_RATIO of peak LR.

    DIAGNOSIS: the previous fixed lr=2e-4 with no warmup is a known failure mode for
    a *post-norm* Transformer (norm_first=False, as used by SignTransformer) trained
    from scratch: large early gradients before self-/cross-attention has organized
    itself can push the model toward a degenerate solution (e.g. a decoder that
    ignores the encoder and falls back on the target-token prior) that it never
    fully recovers from within a small step budget. This is a targeted optimizer
    fix following the standard Transformer training recipe, not an architecture
    change.
    """
    warmup_steps = max(1, int(TRANSFORMER_WARMUP_FRACTION * total_steps))
    def lr_lambda(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = min(1.0, (step - warmup_steps) / max(1, total_steps - warmup_steps))
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        return TRANSFORMER_MIN_LR_RATIO + (1.0 - TRANSFORMER_MIN_LR_RATIO) * cosine
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def fit_transformer(train_loader,val_loader,input_dim,checkpoint):
    model=SignTransformer(input_dim*2,len(vocab)).to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=TRANSFORMER_PEAK_LR,weight_decay=1e-4)
    steps_per_epoch=math.ceil(len(train_loader)/GRAD_ACCUM_STEPS); total_steps=max(1,steps_per_epoch*N_EPOCHS_TRANSFORMER); scheduler=build_transformer_lr_schedule(opt,total_steps)
    crit=nn.CrossEntropyLoss(ignore_index=PAD_IDX); best=float("inf"); bad=0; history=[]; recovery_path=CHECKPOINT_DIR/"signsage_transformer_recovery.pt"
    if RESUME_FROM_RECOVERY and recovery_path.exists():
        rec=torch.load(recovery_path,map_location=DEVICE); model.load_state_dict(rec["state_dict"])
        if rec.get("optimizer_state_dict") is not None: opt.load_state_dict(rec["optimizer_state_dict"])
        if rec.get("scheduler_state_dict") is not None: scheduler.load_state_dict(rec["scheduler_state_dict"])
        print("Explicit recovery resume:",rec.get("epoch"),rec.get("batch"))
    print(f"Training {N_EPOCHS_TRANSFORMER} full-corpus epoch(s): {len(train_loader.dataset):,} samples/epoch")
    for epoch in range(1,N_EPOCHS_TRANSFORMER+1):
        tr=train_epoch_transformer(model,train_loader,opt,crit,epoch,scheduler); va=eval_loss_transformer(model,val_loader,crit); history.append((epoch,tr,va))
        print(f"[Transformer] epoch {epoch:02d}/{N_EPOCHS_TRANSFORMER}: train={tr:.4f} val={va:.4f} lr={scheduler.get_last_lr()[0]:.2e}")
        payload={"state_dict":model.state_dict(),"optimizer_state_dict":opt.state_dict(),"scheduler_state_dict":scheduler.state_dict(),"epoch":epoch,"val_loss":va,"model_config":{"input_dim":input_dim*2,"raw_feature_dim":input_dim,"vocab_size":len(vocab),"d_model":192,"nhead":4,"layers":3,"ff":512,"dropout":0.15,"max_frames":MAX_FRAMES,"max_tokens":MAX_TOKENS},"vocab":vocab}
        torch.save(payload,checkpoint.with_suffix(".latest.pt"))
        if va<best-1e-4: best=va; bad=0; torch.save(payload,checkpoint)
        else:
            bad+=1
            if bad>=2 and epoch>=MIN_EPOCHS_TRANSFORMER: print("Early stopping: validation loss stopped improving."); break
        gc.collect();
        if DEVICE.type=="cuda": torch.cuda.empty_cache()
    if not checkpoint.exists(): raise RuntimeError("No best Transformer checkpoint was produced.")
    model.load_state_dict(torch.load(checkpoint,map_location=DEVICE)["state_dict"]); model.eval(); return model,history

# Training now lives in its own cell below so a failure there does not
# discard these class/function definitions and does not bypass the
# one-batch diagnostic or the tiny-overfit gate (brief §47, §50).
if not RUN_PRIMARY_S2T:
    transformer_full = None
    transformer_full_history = []


In [27]:
# ============================================================
# 6b. Execute GPU benchmark now that the real model is defined
# ============================================================
_diag_model=SignTransformer(LANDMARK_DIM*2,len(vocab),dropout=0.0).to(DEVICE)
benchmark_training_step(_diag_model,train_loader_full)
del _diag_model; gc.collect()
if DEVICE.type=="cuda": torch.cuda.empty_cache()

# ============================================================
# 6c. Resolve epoch count from measured full-corpus throughput
# ============================================================
if DATA_MODE=="kaggle" and RUN_PRIMARY_S2T:
    measured_sec=float(DATA_LOADER_MEAN_SEC_PER_BATCH+GPU_MEAN_SEC_PER_BATCH)
    estimated_epoch_hours=len(train_loader_full)*measured_sec/3600.0
    usable_hours=max(0.25,TRAIN_TIME_BUDGET_HOURS-SESSION_SAFETY_MARGIN_MINUTES/60.0)
    max_fit=int(math.floor(usable_hours/max(estimated_epoch_hours,1e-9)))
    if max_fit<1:
        raise RuntimeError(f"Measured full-corpus epoch estimate is {estimated_epoch_hours:.2f} h, exceeding usable budget {usable_hours:.2f} h. Stop before full training; the archive-local loader did not become fast enough on this session.")
    if max_fit < MIN_EPOCHS_TRANSFORMER:
        raise RuntimeError(
            f"Measured safe training capacity is only {max_fit} epoch(s), "
            f"below MIN_EPOCHS_TRANSFORMER={MIN_EPOCHS_TRANSFORMER}. "
            "The Kaggle run is intentionally stopped rather than exceeding "
            "the safety-adjusted runtime budget."
        )
    N_EPOCHS_TRANSFORMER=min(MAX_EPOCHS_TRANSFORMER,max_fit)
else:
    estimated_epoch_hours=None; N_EPOCHS_TRANSFORMER=1
print(f"Resolved full-corpus epochs: {N_EPOCHS_TRANSFORMER}")
print(f"Estimated epoch wall time: {estimated_epoch_hours:.2f} h" if estimated_epoch_hours is not None else "Smoke/local mode")
knowledge_observation("epoch_budget_resolution",{"data_loader_sec_per_batch":float(DATA_LOADER_MEAN_SEC_PER_BATCH),"gpu_compute_sec_per_batch":float(GPU_MEAN_SEC_PER_BATCH),"estimated_epoch_hours":estimated_epoch_hours,"resolved_epochs":int(N_EPOCHS_TRANSFORMER)})


GPU batch 01: 0.070s
GPU batch 02: 0.068s
GPU batch 03: 0.067s
GPU batch 04: 0.067s
GPU batch 05: 0.071s
Mean GPU compute=0.069s/batch
Resolved full-corpus epochs: 3
Estimated epoch wall time: 1.91 h


In [28]:
# ============================================================
# 8-diag. One-batch forward+backward diagnostic (brief §50)
# ============================================================
# Proves, before ANY full-corpus GPU spend, that the model builds,
# a real batch flows through it, and backward() completes. If this
# raises, the traceback is the ONLY thing needed to fix the trainer.
import traceback

def _diag_one_batch(model, loader, n_batches=2):
    model.train()
    crit = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
    it = iter(loader)
    for i in range(n_batches):
        frames, lengths, tok_in, tok_out = next(it)
        frames  = frames.to(DEVICE, non_blocking=True)
        lengths = lengths.to(DEVICE, non_blocking=True)
        tok_in  = tok_in.to(DEVICE, non_blocking=True)
        tok_out = tok_out.to(DEVICE, non_blocking=True)

        # Shape assertions (brief §14 objective audit).
        B, T, D = frames.shape
        assert D == LANDMARK_DIM, f"frames last dim {D} != LANDMARK_DIM {LANDMARK_DIM}"
        assert T == MAX_FRAMES,  f"frames time dim {T} != MAX_FRAMES {MAX_FRAMES}"
        assert tok_in.shape == tok_out.shape, "tok_in/tok_out shape mismatch"

        src_pad  = padding_mask(lengths, frames.size(1))
        tgt_mask = causal_mask(tok_in.size(1))
        assert src_pad.shape == (B, T)
        assert tgt_mask.shape == (tok_in.size(1), tok_in.size(1))

        with autocast_ctx():
            logits = model(frames, src_pad, tok_in, tgt_mask)
            loss = crit(logits.reshape(-1, logits.size(-1)), tok_out.reshape(-1))
        assert logits.shape == (B, tok_in.size(1), len(vocab)), (
            f"logits {tuple(logits.shape)} != ({B},{tok_in.size(1)},{len(vocab)})")
        if not torch.isfinite(loss):
            raise RuntimeError(f"non-finite loss on batch {i+1}")
        loss.backward()
        print(f"[diag] batch {i+1}/{n_batches} OK -- loss={float(loss):.4f} "
              f"logits={tuple(logits.shape)} src_pad_true={int(src_pad.sum())}")
        model.zero_grad(set_to_none=True)
        del frames, lengths, tok_in, tok_out, src_pad, tgt_mask, logits, loss
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

try:
    _dm = SignTransformer(LANDMARK_DIM * 2, len(vocab), dropout=0.0).to(DEVICE)
    print(f"[diag] model built input_dim={LANDMARK_DIM*2} vocab={len(vocab)}")
    _diag_one_batch(_dm, train_loader_full, n_batches=2)
    print("[diag] ONE-BATCH DIAGNOSTIC PASSED.")
    del _dm; gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
except Exception:
    traceback.print_exc()
    raise


[diag] model built input_dim=3258 vocab=16000


/tmp/ipykernel_22/3809496179.py:39: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print(f"[diag] batch {i+1}/{n_batches} OK -- loss={float(loss):.4f} "


[diag] batch 1/2 OK -- loss=9.8121 logits=(64, 35, 16000) src_pad_true=1919
[diag] batch 2/2 OK -- loss=9.8181 logits=(64, 37, 16000) src_pad_true=1527
[diag] ONE-BATCH DIAGNOSTIC PASSED.


## 8a. Tiny-subset overfit test — mandatory correctness gate

The exact tokenizer, target shifting, masks, model and loss must prove that a tiny real subset can be memorized before full-corpus GPU spend. This is a correctness test, not a research result.


In [29]:
# ============================================================
# 8a. Tiny-subset overfit execution
# ============================================================
#
# Purpose:
#   Verify that the complete Sign→Text training path can
#   memorize a small subset before expensive full-corpus
#   training begins.
#
# This is a PIPELINE CORRECTNESS GATE.
# It is NOT a measure of final model quality.
#
# Feature contract:
#
#   Raw MediaPipe features : 1629
#   Temporal delta         : 1629
#   Concatenated input     : 3258
#
# The SignTransformer internally performs:
#
#   delta = x[t] - x[t-1]
#   concat([x, delta])
#
# ------------------------------------------------------------
# IMPORTANT:
# Exact-match evaluation below compares ONLY the valid target
# positions. Padding positions are not allowed to penalize the
# sequence-match metric.
# ============================================================


# ------------------------------------------------------------
# Tiny-overfit configuration
# ------------------------------------------------------------

RUN_TINY_OVERFIT_TEST = True

TINY_OVERFIT_N = 64
TINY_OVERFIT_STEPS = 600

TINY_OVERFIT_MAX_LOSS = 1.0
TINY_OVERFIT_MIN_EXACT_MATCH = 0.60


# ------------------------------------------------------------
# Verify primary DataLoader
# ------------------------------------------------------------

if not RUN_TINY_OVERFIT_TEST:
    raise RuntimeError(
        "Final notebook requires the tiny-overfit gate."
    )


if "train_loader_full" not in globals():
    raise RuntimeError(
        "train_loader_full is not defined. "
        "The primary Sign→Text DataLoader must be created "
        "before the tiny-overfit gate."
    )


if not isinstance(train_loader_full, DataLoader):
    raise RuntimeError(
        "train_loader_full exists but is not a PyTorch DataLoader."
    )


# ------------------------------------------------------------
# Use EXACTLY the Dataset used by primary training
# ------------------------------------------------------------

_base_train_ds = train_loader_full.dataset


if _base_train_ds is None:
    raise RuntimeError(
        "train_loader_full.dataset is None."
    )


if len(_base_train_ds) == 0:
    raise RuntimeError(
        "train_loader_full.dataset contains zero samples."
    )


print(
    f"Tiny-overfit source dataset: "
    f"train_loader_full.dataset "
    f"({len(_base_train_ds):,} samples)"
)


# ------------------------------------------------------------
# Build tiny subset
# ------------------------------------------------------------

n_tiny = min(
    int(TINY_OVERFIT_N),
    len(_base_train_ds),
)


if n_tiny < 2:
    raise RuntimeError(
        f"Tiny-overfit subset is too small: {n_tiny} samples."
    )


tiny_ds = Subset(
    _base_train_ds,
    range(n_tiny),
)


tiny_batch_size = min(
    16,
    n_tiny,
)


print("=" * 70)
print("TINY SUBSET OVERFIT GATE")
print("=" * 70)

print(
    f"Raw feature dimension   : {LANDMARK_DIM}"
)

print(
    f"Delta feature dimension : {LANDMARK_DIM}"
)

MODEL_INPUT_DIM = int(
    LANDMARK_DIM * 2
)

print(
    f"Model input dimension   : {MODEL_INPUT_DIM}"
)

print(
    f"Tiny samples            : {n_tiny}"
)

print(
    f"Batch size              : {tiny_batch_size}"
)

print(
    f"Optimization steps      : {TINY_OVERFIT_STEPS}"
)

print(
    f"Max loss threshold      : "
    f"{TINY_OVERFIT_MAX_LOSS:.4f}"
)

print(
    f"Exact-match minimum     : "
    f"{TINY_OVERFIT_MIN_EXACT_MATCH:.2%}"
)

print()


# ------------------------------------------------------------
# Tiny DataLoader
# ------------------------------------------------------------

tiny_loader = DataLoader(
    tiny_ds,
    batch_size=tiny_batch_size,
    shuffle=True,
    collate_fn=collate_sign_text,
    num_workers=0,
)


# ------------------------------------------------------------
# Tiny model
#
# The model receives 3258 because FrameProjection internally
# concatenates raw features and temporal deltas.
# ------------------------------------------------------------

tiny_model = SignTransformer(
    MODEL_INPUT_DIM,
    len(vocab),
    dropout=0.0,
).to(DEVICE)


# ------------------------------------------------------------
# Verify actual Linear projection dimensions
# ------------------------------------------------------------

linear_layers = [
    module
    for module in tiny_model.modules()
    if isinstance(module, nn.Linear)
]


if not linear_layers:
    raise RuntimeError(
        "Could not find any Linear projection layers "
        "inside SignTransformer."
    )


projection_candidates = [
    int(layer.in_features)
    for layer in linear_layers
]


if MODEL_INPUT_DIM not in projection_candidates:
    raise RuntimeError(
        "SignTransformer projection dimension mismatch. "
        f"Expected a Linear layer accepting "
        f"{MODEL_INPUT_DIM} features "
        f"({LANDMARK_DIM} raw + {LANDMARK_DIM} delta), "
        f"but found Linear input dimensions: "
        f"{projection_candidates}."
    )


print(
    "Motion-aware input projection verified: "
    f"{MODEL_INPUT_DIM} features."
)


# ------------------------------------------------------------
# Optimizer / loss
# ------------------------------------------------------------

tiny_opt = torch.optim.AdamW(
    tiny_model.parameters(),
    lr=1e-3,
    weight_decay=0.0,
)


tiny_criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_IDX,
)


# ------------------------------------------------------------
# AMP scaler
# ------------------------------------------------------------

if USE_AMP and DEVICE.type == "cuda":

    try:

        tiny_scaler = torch.amp.GradScaler(
            "cuda",
            enabled=True,
        )

    except (AttributeError, TypeError):

        tiny_scaler = torch.cuda.amp.GradScaler(
            enabled=True,
        )

else:

    try:

        tiny_scaler = torch.amp.GradScaler(
            "cuda",
            enabled=False,
        )

    except (AttributeError, TypeError):

        tiny_scaler = torch.cuda.amp.GradScaler(
            enabled=False,
        )


# ------------------------------------------------------------
# Tiny-subset optimization
# ------------------------------------------------------------

tiny_model.train()

last_loss = float("inf")

tiny_iter = iter(
    tiny_loader
)


for step in range(
    TINY_OVERFIT_STEPS
):

    try:

        (
            frames,
            lengths,
            tin,
            tout,
        ) = next(tiny_iter)

    except StopIteration:

        tiny_iter = iter(
            tiny_loader
        )

        (
            frames,
            lengths,
            tin,
            tout,
        ) = next(tiny_iter)


    frames = frames.to(
        DEVICE,
        non_blocking=PIN_MEMORY,
    )

    lengths = lengths.to(
        DEVICE,
        non_blocking=PIN_MEMORY,
    )

    tin = tin.to(
        DEVICE,
        non_blocking=PIN_MEMORY,
    )

    tout = tout.to(
        DEVICE,
        non_blocking=PIN_MEMORY,
    )


    tiny_opt.zero_grad(
        set_to_none=True
    )


    with autocast_ctx():

        logits = tiny_model(
            frames,
            padding_mask(
                lengths,
                frames.size(1),
            ),
            tin,
            causal_mask(
                tin.size(1),
            ),
        )

        loss = tiny_criterion(
            logits.reshape(
                -1,
                logits.size(-1),
            ),
            tout.reshape(-1),
        )


    if not torch.isfinite(loss):
        raise RuntimeError(
            "Tiny overfit produced a non-finite loss."
        )


    tiny_scaler.scale(
        loss
    ).backward()


    tiny_scaler.unscale_(
        tiny_opt
    )


    torch.nn.utils.clip_grad_norm_(
        tiny_model.parameters(),
        1.0,
    )


    tiny_scaler.step(
        tiny_opt
    )

    tiny_scaler.update()


    last_loss = float(
        loss.detach().item()
    )


    if (
        step == 0
        or (step + 1) % 50 == 0
        or step + 1 == TINY_OVERFIT_STEPS
    ):

        print(
            f"tiny step "
            f"{step + 1:03d}/{TINY_OVERFIT_STEPS}: "
            f"loss={last_loss:.4f}"
        )


    del (
        frames,
        lengths,
        tin,
        tout,
        logits,
        loss,
    )


# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

tiny_model.eval()

exact = []

token_correct = 0
token_total = 0


eval_tiny = DataLoader(
    tiny_ds,
    batch_size=tiny_batch_size,
    shuffle=False,
    collate_fn=collate_sign_text,
    num_workers=0,
)


with torch.no_grad():

    for (
        frames,
        lengths,
        tin,
        tout,
    ) in eval_tiny:

        frames = frames.to(
            DEVICE,
            non_blocking=PIN_MEMORY,
        )

        lengths = lengths.to(
            DEVICE,
            non_blocking=PIN_MEMORY,
        )

        tin = tin.to(
            DEVICE,
            non_blocking=PIN_MEMORY,
        )

        tout = tout.to(
            DEVICE,
            non_blocking=PIN_MEMORY,
        )


        with autocast_ctx():

            logits = tiny_model(
                frames,
                padding_mask(
                    lengths,
                    frames.size(1),
                ),
                tin,
                causal_mask(
                    tin.size(1),
                ),
            )


        pred = logits.argmax(
            dim=-1
        )


        # ----------------------------------------------------
        # Token accuracy
        # ----------------------------------------------------

        valid = (
            tout != PAD_IDX
        )


        token_correct += int(
            (
                pred[valid]
                == tout[valid]
            ).sum().item()
        )


        token_total += int(
            valid.sum().item()
        )


        # ----------------------------------------------------
        # Correct exact-sequence evaluation
        #
        # IMPORTANT:
        # Only compare positions belonging to the actual target
        # sequence. Padding positions must not count against
        # exact match.
        # ----------------------------------------------------

        pred_cpu = pred.cpu().tolist()

        target_cpu = tout.cpu().tolist()


        for p, t in zip(
            pred_cpu,
            target_cpu,
        ):

            # Remove target padding FIRST.
            target_tokens = [
                x
                for x in t
                if x != PAD_IDX
            ]


            target_len = len(
                target_tokens
            )


            # Compare exactly the same valid target positions.
            predicted_tokens = p[
                :target_len
            ]


            exact.append(
                predicted_tokens
                == target_tokens
            )


        del (
            frames,
            lengths,
            tin,
            tout,
            logits,
            pred,
            pred_cpu,
            target_cpu,
        )


# ------------------------------------------------------------
# Calculate metrics
# ------------------------------------------------------------

exact_match = (
    float(np.mean(exact))
    if exact
    else 0.0
)


token_accuracy = (
    token_correct
    / max(
        1,
        token_total,
    )
)


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print()

print("=" * 70)
print("TINY OVERFIT RESULTS")
print("=" * 70)

print(
    f"Tiny overfit final loss   : "
    f"{last_loss:.4f}"
)

print(
    f"Tiny token accuracy       : "
    f"{token_accuracy:.2%}"
)

print(
    f"Tiny exact sequence match : "
    f"{exact_match:.2%}"
)


# ------------------------------------------------------------
# Gate decision
# ------------------------------------------------------------

loss_pass = (
    last_loss
    <= TINY_OVERFIT_MAX_LOSS
)


exact_pass = (
    exact_match
    >= TINY_OVERFIT_MIN_EXACT_MATCH
)


print()

print(
    f"Loss criterion        : "
    f"{'PASS' if loss_pass else 'FAIL'}"
)

print(
    f"Exact-match criterion : "
    f"{'PASS' if exact_pass else 'FAIL'}"
)


if not loss_pass or not exact_pass:

    raise RuntimeError(
        "TINY OVERFIT GATE FAILED. "
        "Full-corpus training was intentionally not started. "
        f"Final loss={last_loss:.4f}, "
        f"exact_match={exact_match:.2%}, "
        f"token_accuracy={token_accuracy:.2%}."
    )


print()

print(
    "TINY OVERFIT GATE PASSED."
)

print(
    "The Sign→Text training path successfully memorized "
    "the tiny subset."
)


# ------------------------------------------------------------
# Cleanup
# ------------------------------------------------------------

del (
    tiny_model,
    tiny_opt,
    tiny_scaler,
    tiny_loader,
    tiny_ds,
    eval_tiny,
    tiny_iter,
    _base_train_ds,
    linear_layers,
    projection_candidates,
)

gc.collect()


if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

Tiny-overfit source dataset: train_loader_full.dataset (99,699 samples)
TINY SUBSET OVERFIT GATE
Raw feature dimension   : 1629
Delta feature dimension : 1629
Model input dimension   : 3258
Tiny samples            : 64
Batch size              : 16
Optimization steps      : 600
Max loss threshold      : 1.0000
Exact-match minimum     : 60.00%

Motion-aware input projection verified: 3258 features.
tiny step 001/600: loss=9.9313
tiny step 050/600: loss=1.7658
tiny step 100/600: loss=0.5677
tiny step 150/600: loss=0.2995
tiny step 200/600: loss=0.2668
tiny step 250/600: loss=0.1078
tiny step 300/600: loss=0.1559
tiny step 350/600: loss=0.0443
tiny step 400/600: loss=0.0032
tiny step 450/600: loss=0.0022
tiny step 500/600: loss=0.0015
tiny step 550/600: loss=0.0014
tiny step 600/600: loss=0.0013

TINY OVERFIT RESULTS
Tiny overfit final loss   : 0.0013
Tiny token accuracy       : 100.00%
Tiny exact sequence match : 100.00%

Loss criterion        : PASS
Exact-match criterion : PASS

TINY OVE

In [30]:
# ============================================================
# 8-pretrain. Final sanity gate
# ============================================================
required=["SignTransformer","padding_mask","causal_mask","load_pose","train_loader_full","val_loader_full","N_EPOCHS_TRANSFORMER","DATA_LOADER_MEAN_SEC_PER_BATCH","GPU_MEAN_SEC_PER_BATCH"]
missing=[x for x in required if x not in globals()]
if missing: raise RuntimeError(f"Missing before primary training: {missing}")
if RUN_MODE=="full": assert TRAIN_LIMIT is None and VAL_LIMIT is None and TEST_LIMIT is None
print("FINAL PRE-TRAINING SANITY PASSED")
print(f"Full split: train={len(df_train):,} val={len(df_val):,} test={len(df_test):,}")
print(f"Resolved epochs: {N_EPOCHS_TRANSFORMER}")


FINAL PRE-TRAINING SANITY PASSED
Full split: train=99,699 val=11,927 test=12,153
Resolved epochs: 3


In [31]:
# ============================================================
# 8-train. Primary Sign -> English training
# ============================================================
if RUN_PRIMARY_S2T:
    if RUN_MODE=="full": assert TRAIN_LIMIT is None and VAL_LIMIT is None and TEST_LIMIT is None
    transformer_full,transformer_full_history=fit_transformer(train_loader_full,val_loader_full,LANDMARK_DIM,CHECKPOINT_DIR/"signsage_transformer_full.pt")
else:
    transformer_full=None; transformer_full_history=[]
    print("Primary Transformer skipped.")
gc.collect()
if DEVICE.type=="cuda": torch.cuda.empty_cache()


Training 3 full-corpus epoch(s): 99,699 samples/epoch


Transformer train:   0%|          | 0/1558 [00:00<?, ?it/s]

Transformer eval:   0%|          | 0/187 [00:00<?, ?it/s]

[Transformer] epoch 01/3: train=6.1802 val=5.5520 lr=1.65e-04


Transformer train:   0%|          | 0/1558 [00:00<?, ?it/s]

Transformer eval:   0%|          | 0/187 [00:00<?, ?it/s]

[Transformer] epoch 02/3: train=5.3679 val=5.3288 lr=7.03e-05


Transformer train:   0%|          | 0/1558 [00:00<?, ?it/s]

Transformer eval:   0%|          | 0/187 [00:00<?, ?it/s]

[Transformer] epoch 03/3: train=5.1964 val=5.2611 lr=2.00e-05


## 9. Non-manual-marker ablation

This is meaningful only because the real pose header exposed `FACE_LANDMARKS` and the no-face mask is therefore verified rather than guessed.

In [32]:
## 9. Optional non-manual-marker ablation
if RUN_NOFACE_ABLATION:
    transformer_noface, transformer_noface_history = fit_transformer(
        train_loader_noface, val_loader_noface, NOFACE_DIM,
        CHECKPOINT_DIR/"signsage_transformer_noface.pt")
else:
    transformer_noface=None; transformer_noface_history=[]
    print("No-face ablation skipped in the primary run; enable RUN_NOFACE_ABLATION for the thesis ablation experiment.")


No-face ablation skipped in the primary run; enable RUN_NOFACE_ABLATION for the thesis ablation experiment.


In [33]:
# 9a. Record Transformer experiment metadata.
def record_training_history(name,family,input_features,history,checkpoint_path,epochs):
    best_val=float(min(v for _,_,v in history)) if history else float("nan")
    knowledge_insert("experiments",
        ["run_id","name","family","input_features","epochs","batch_size","learning_rate","weight_decay","checkpoint_path","best_validation_loss"],
        [RUN_ID,name,family,input_features,epochs,BATCH_SIZE,2e-4,1e-4,str(checkpoint_path),best_val],
        delete_keys={"run_id":RUN_ID,"name":name})
    for epoch,tr_loss,val_loss in history:
        knowledge_metric(name,f"train_loss_epoch_{epoch}",tr_loss,"train")
        knowledge_metric(name,f"val_loss_epoch_{epoch}",val_loss,"validation")
    knowledge_artifact(name,"checkpoint",checkpoint_path,f"{name} checkpoint.")

if RUN_PRIMARY_S2T:
    record_training_history("Transformer Sign→Text","Sign→Text",LANDMARK_DIM,
                            transformer_full_history,CHECKPOINT_DIR/"signsage_transformer_full.pt",
                            N_EPOCHS_TRANSFORMER)
if RUN_NOFACE_ABLATION:
    record_training_history("Transformer (no face)","Sign→Text",NOFACE_DIM,
                            transformer_noface_history,CHECKPOINT_DIR/"signsage_transformer_noface.pt",
                            N_EPOCHS_TRANSFORMER)
print("Training metadata recorded.")


Training metadata recorded.


## 10. Held-out Sign → Text evaluation

Model selection uses validation loss. The numbers below are computed on the **test split** that was never used for checkpoint selection.

The trivial baseline predicts the most common training sentence. It is deliberately weak but necessary as a reference point.

In [34]:
## 10. Held-out Sign → Text evaluation

# Full test split is evaluated. Fast batched greedy decoding is used for metrics;
# beam search is used only for qualitative inspection so evaluation does not become
# an unnecessary multi-hour bottleneck.
import jiwer, sacrebleu
from rouge_score import rouge_scorer
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def metrics(refs,hyps):
    return {
        "WER":float(jiwer.wer(refs,hyps)),
        "BLEU":float(sacrebleu.corpus_bleu(hyps,[refs]).score),
        "ROUGE-L":float(np.mean([rouge.score(r,h)["rougeL"].fmeasure for r,h in zip(refs,hyps)])),
        "exact_match":float(np.mean([normalize_text(r)==normalize_text(h) for r,h in zip(refs,hyps)])),
        "nonempty_output":float(np.mean([bool(normalize_text(h)) for h in hyps])),
    }

if transformer_full is None:
    raise RuntimeError("Primary Transformer was not trained.")

refs,tf_h,tf_conf=batch_greedy_transformer(transformer_full,test_loader_full)
results_s2t={"Transformer Sign→Text":metrics(refs,tf_h)}

if RUN_GRU_BASELINE:
    refs_gru,gru_h,gru_conf=batch_greedy_gru(gru_model,test_loader_full)
    results_s2t["GRU baseline"]=metrics(refs_gru,gru_h)
if RUN_NOFACE_ABLATION:
    refs_nf,nf_h,nf_conf=batch_greedy_transformer(transformer_noface,test_loader_noface)
    results_s2t["Transformer (no face)"]=metrics(refs_nf,nf_h)

baseline_pool=df_train.loc[df_train["n_words"]>=2, "text"].map(normalize_text)
if baseline_pool.empty:
    raise RuntimeError("No multi-word training targets remain for the trivial baseline.")
common_sentence=baseline_pool.value_counts().idxmax()
trivial_h=[common_sentence]*len(refs)
results_s2t["Trivial baseline"]=metrics(refs,trivial_h)

print(f"Evaluated {len(refs):,} held-out test examples.")
print(f"{'Model':<28}{'WER':>10}{'BLEU':>10}{'ROUGE-L':>12}{'Exact':>10}")
for name,m in results_s2t.items():
    print(f"{name:<28}{m['WER']:>10.3f}{m['BLEU']:>10.2f}{m['ROUGE-L']:>12.3f}{m['exact_match']:>10.2%}")

tf_unique=len(set(tf_h)); tf_unique_frac=tf_unique/max(1,len(tf_h))
print(f"\nTransformer output diversity: {tf_unique:,}/{len(tf_h):,} ({tf_unique_frac:.2%})")
print(f"Mean sequence confidence proxy: {np.mean(tf_conf):.4f}")
print(f"10th percentile confidence: {np.percentile(tf_conf,10):.4f}")
from collections import Counter as _Counter
_top_outputs=_Counter(tf_h).most_common(10)
print("Top generated outputs:", _top_outputs)
print("IMPORTANT: confidence is a diagnostic, not a calibrated probability.")

# --- Compact, memory-bounded collapse diagnostics (brief Section 12) -----------
# Everything the rest of the notebook needs from this evaluation is reduced to
# plain Python scalars/dicts HERE, before the large refs/hyps/confidence lists are
# released below. Downstream cells (GO/NO-GO report, deployment manifest) read
# these compact variables instead of re-indexing results_s2t by a hand-typed
# string key or reaching for the (soon to be deleted) raw lists -- which is
# exactly the class of bug this fix removes (see DIAGNOSIS note below).
primary_metrics = results_s2t["Transformer Sign→Text"]
tf_mean_conf = float(np.mean(tf_conf))
tf_p10_conf = float(np.percentile(tf_conf, 10))
top_prediction, top_prediction_count = _top_outputs[0]
top_prediction_freq = top_prediction_count / max(1, len(tf_h))
_gen_lengths = np.array([len(normalize_text(h).split()) for h in tf_h], dtype=np.int32)
_ref_lengths = np.array([len(normalize_text(r).split()) for r in refs], dtype=np.int32)
generated_length_stats = {
    "mean": float(_gen_lengths.mean()), "median": float(np.median(_gen_lengths)),
    "p90": float(np.percentile(_gen_lengths, 90)), "max": int(_gen_lengths.max()),
    "empty_fraction": float(np.mean(_gen_lengths == 0)),
}
reference_length_stats = {
    "mean": float(_ref_lengths.mean()), "median": float(np.median(_ref_lengths)),
    "p90": float(np.percentile(_ref_lengths, 90)), "max": int(_ref_lengths.max()),
}
repeated_output_rate = 1.0 - tf_unique_frac

print(f"\nTop prediction: {top_prediction!r} -- {top_prediction_count:,}/{len(tf_h):,} "
      f"({top_prediction_freq:.2%}) of all test outputs")
print(f"Repeated-output rate (1 - unique fraction): {repeated_output_rate:.2%}")
print(f"Generated length (words) -- mean={generated_length_stats['mean']:.2f}, "
      f"median={generated_length_stats['median']:.1f}, p90={generated_length_stats['p90']:.1f}, "
      f"empty={generated_length_stats['empty_fraction']:.2%}")
print(f"Reference length (words) -- mean={reference_length_stats['mean']:.2f}, "
      f"median={reference_length_stats['median']:.1f}, p90={reference_length_stats['p90']:.1f}")

knowledge_observation("sign_to_text_evaluation",{
    "n_test":len(refs),"metrics":results_s2t,
    "unique_outputs":tf_unique,"unique_fraction":tf_unique_frac,
    "repeated_output_rate":repeated_output_rate,
    "top_prediction":top_prediction,"top_prediction_freq":top_prediction_freq,
    "mean_confidence_proxy":tf_mean_conf,"p10_confidence_proxy":tf_p10_conf,
    "generated_length_stats":generated_length_stats,
    "reference_length_stats":reference_length_stats,
})


# Release large CPU-side evaluation objects before subsequent notebook phases.
# Keeping full test refs/hypotheses plus live DataLoader workers alive can accumulate
# host RAM on Kaggle. Everything needed downstream now lives in the compact scalar/
# dict variables computed above (primary_metrics, tf_unique_frac, tf_mean_conf,
# tf_p10_conf, top_prediction_freq, *_length_stats), so it is safe to drop the raw
# per-example lists here.
#
# DIAGNOSIS: the previous version of this cell deleted `tf_conf` here while a LATER
# cell (Section 18/Finalize) still referenced `np.mean(tf_conf)` directly, and a
# separate later cell (Section 17/GO-NO-GO) indexed `results_s2t["Transformer (full)"]`
# -- a key that was never actually written to results_s2t (it is
# "Transformer Sign→Text" above). Both would raise (NameError / KeyError) at the very
# end of an expensive full run, right when the deployment bundle is being produced.
# Those two cells now read the scalar variables defined above instead.
try:
    del refs, tf_h, tf_conf, trivial_h, _gen_lengths, _ref_lengths
except NameError:
    pass
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
print("Post-test memory cleanup completed.")


Transformer test:   0%|          | 0/190 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Evaluated 12,153 held-out test examples.
Model                              WER      BLEU     ROUGE-L     Exact
Transformer Sign→Text            1.005      0.24       0.089     0.00%
Trivial baseline                 1.031      0.40       0.023     0.02%

Transformer output diversity: 120/12,153 (0.99%)
Mean sequence confidence proxy: 0.1331
10th percentile confidence: 0.1016
Top generated outputs: [('the police said that the government has been arrested.', 5450), ('the police said that the government has been arrested for the government.', 1035), ('the police have been arrested.', 938), ('the picture of the words.', 814), ('page number', 534), ('the police have been arrested by the government.', 475), ('the police said that the government has been arrested in the world.', 435), ('the teacher is the same.', 371), ('the first time is a picture of the same.', 315), ('the police were very happy.', 214)]
IMPORTANT: confidence is a diagnostic, not a calibrated probability.

Top prediction: '

In [35]:
# 10a. Record Sign→Text evaluation metrics.
for model_name,metric_dict in results_s2t.items():
    for metric_name,metric_value in metric_dict.items():
        knowledge_metric(model_name,metric_name,metric_value,"test")
print("Sign→Text evaluation metrics recorded.")


Sign→Text evaluation metrics recorded.


### 10b. Validation-only confidence calibration

The runtime abstention threshold is calibrated on validation data, never on the held-out test set. It is a safety gate, not a claim that confidence is a true probability.

In [36]:
def token_overlap_score(ref,hyp):
    r=normalize_text(ref).split(); h=normalize_text(hyp).split()
    if not r or not h:return 0.0
    from collections import Counter
    common=sum((Counter(r)&Counter(h)).values())
    return common/max(1,len(r))

cal_n=min(500,len(df_val))
cal_df=df_val.sample(cal_n,random_state=SEED).reset_index(drop=True)
cal_ds=SignToTextDataset(cal_df,FULL_FEATURE_MASK)
cal_loader=DataLoader(cal_ds,batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate_sign_text,num_workers=EVAL_NUM_WORKERS,pin_memory=PIN_MEMORY and DEVICE.type=="cuda")
cal_refs,cal_hyps,cal_conf=batch_greedy_transformer(transformer_full,cal_loader)
cal_overlap=np.array([token_overlap_score(r,h) for r,h in zip(cal_refs,cal_hyps)])
good_conf=np.array(cal_conf)[cal_overlap>=0.5]
if len(good_conf)>=20:
    LIVE_CONFIDENCE_FLOOR=float(np.percentile(good_conf,10))
else:
    LIVE_CONFIDENCE_FLOOR=float(np.percentile(cal_conf,50)) if cal_conf else 0.0
    print("WARNING: too few validation examples met the overlap criterion; confidence gate is only a weak diagnostic.")
print(f"Validation calibration examples: {cal_n}")
print(f"LIVE_CONFIDENCE_FLOOR: {LIVE_CONFIDENCE_FLOOR:.4f}")
knowledge_observation("live_confidence_calibration",{
    "n_validation":cal_n,"good_overlap_examples":int(len(good_conf)),
    "threshold":LIVE_CONFIDENCE_FLOOR,
    "method":"10th percentile confidence among validation predictions with token overlap >= 0.5",
})


try:
    del cal_refs, cal_hyps, cal_conf, cal_overlap, cal_df, cal_ds, cal_loader
except NameError:
    pass
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


Pre-tokenizing 500 captions...


Tokenizing:   0%|          | 0/500 [00:00<?, ?it/s]

Building ZIP-member locality index from the central directory...


Indexing pose offsets:   0%|          | 0/500 [00:00<?, ?it/s]

Dataset ready: 500 samples


Transformer test:   0%|          | 0/8 [00:00<?, ?it/s]

Validation calibration examples: 500
LIVE_CONFIDENCE_FLOOR: 0.1389


In [37]:
# Qualitative inspection: use beam search on a small, deterministic test sample.
for i in range(min(5,len(test_ds_full))):
    frames,length,tokens=test_ds_full[i]
    ref=decode_ids(tokens.tolist())
    hyp=beam_transformer(transformer_full,frames,length)
    print(f"[{i:02d}] REF : {ref}")
    print(f"     PRED: {hyp}")
    print()


[00] REF : page number 105
     PRED: page number.

[01] REF : unit 7. reside this poem.
     PRED: page number.

[02] REF : on my blackboard i can draw.
     PRED: page number.

[03] REF : on my blackboard, i can draw.
     PRED: page number.

[04] REF : one little house with one green door.
     PRED: page number.



## 11. Text → Sign: gloss and retrieval baseline

The naive gloss function is intentionally deterministic. The optional Groq function is not required for the core experiment and is never allowed to break the notebook.

In [38]:
STOPWORDS = {
    "a","an","the","is","are","was","were","am","to","of","in","on","and","but",
    "it","this","that","for","with","as","at","by"
}

def text_to_naive_gloss(text):
    words = str(text).lower().split()
    kept = [w for w in words if w not in STOPWORDS]
    return " ".join(kept) if kept else " ".join(words)

def text_to_gloss_llm(text, api_key=None):
    api_key = api_key or os.environ.get("GROQ_API_KEY")
    if not api_key:
        return text_to_naive_gloss(text)
    try:
        from groq import Groq
        client = Groq(api_key=api_key)
        resp = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            reasoning_effort="low",
            messages=[
                {"role":"system","content":"Convert English to concise ISL gloss. Reply only with the gloss."},
                {"role":"user","content":str(text)}
            ],
        )
        return resp.choices[0].message.content.strip()
    except Exception as exc:
        print("Groq unavailable; using naive gloss:", type(exc).__name__)
        return text_to_naive_gloss(text)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# RETRIEVAL_MAX_SAMPLES is defined once in the config cell (Section 0) so it
# applies consistently across local/Kaggle runs and so the reproducibility-
# config cell can record its value regardless of execution order.
if len(df_train) > RETRIEVAL_MAX_SAMPLES:
    retrieval_df = df_train.sample(RETRIEVAL_MAX_SAMPLES, random_state=SEED)
    print(f"Using {len(retrieval_df)} samples for retrieval baseline (capped to bound memory/runtime).")
else:
    retrieval_df = df_train

train_gloss = retrieval_df["text"].map(text_to_naive_gloss).tolist()
tfidf = TfidfVectorizer().fit(train_gloss)
train_vecs = tfidf.transform(train_gloss)

def retrieve_pose_for_text(text):
    q = tfidf.transform([text_to_naive_gloss(text)])
    sims = cosine_similarity(q, train_vecs)[0]
    idx = int(np.argmax(sims))
    row = retrieval_df.iloc[idx]  # use the subset, not full df_train
    pose, length = load_pose(row["uid"], FULL_FEATURE_MASK)
    return pose[:length], row["text"], float(sims[idx])

print("Example gloss:", text_to_naive_gloss(df_test.iloc[0]["text"]))


Using 3000 samples for retrieval baseline (capped to bound memory/runtime).
Example gloss: page number 105


## 12. Text → Sign: autoregressive pose generator

The generator is deliberately small enough to run on CPU. Unlike the earlier notebook, it receives a learned text context and a learned time embedding at every step.

Training uses teacher forcing; evaluation is explicitly **free-running**, so the reported result is not contaminated by ground-truth previous frames.

In [39]:
# ============================================================
# Optional Text → Pose experiment
# English → ISL gloss → pose sequence
# ============================================================

if RUN_TEXT_TO_POSE:

    # --------------------------------------------------------
    # 1. Gloss generation
    # --------------------------------------------------------
    #
    # Default:
    #   deterministic naive gloss
    #
    # Optional:
    #   Groq-backed gloss generation when USE_LLM_GLOSS=True
    #
    # The retrieval baseline remains independent of this setting.
    # --------------------------------------------------------

    gloss_fn = (
        text_to_gloss_llm
        if USE_LLM_GLOSS
        else text_to_naive_gloss
    )

    total_rows = (
        len(df_train)
        + len(df_val)
        + len(df_test)
    )

    if USE_LLM_GLOSS:
        print(
            f"USE_LLM_GLOSS=True: generating glosses for "
            f"{total_rows:,} rows via Groq."
        )
        print(
            "This introduces network latency and non-determinism. "
            "The configured LLM gloss function must retain its "
            "per-row fallback behavior on API failure."
        )
    else:
        print(
            "USE_LLM_GLOSS=False: using deterministic naive gloss."
        )

    # Work on copies so this optional branch does not mutate the
    # primary Sign→Text dataframes used elsewhere.
    t2p_train = df_train[["uid", "text"]].copy()
    t2p_val = df_val[["uid", "text"]].copy()
    t2p_test = df_test[["uid", "text"]].copy()

    t2p_train["gloss"] = t2p_train["text"].map(gloss_fn)
    t2p_val["gloss"] = t2p_val["text"].map(gloss_fn)
    t2p_test["gloss"] = t2p_test["text"].map(gloss_fn)

    # --------------------------------------------------------
    # 2. Gloss vocabulary
    # --------------------------------------------------------

    gloss_vocab = build_vocab(t2p_train["gloss"])
    gloss_inv_vocab = {
        i: w
        for w, i in gloss_vocab.items()
    }

    G_PAD, G_BOS, G_EOS, G_UNK = [
        gloss_vocab[s]
        for s in SPECIALS
    ]

    def encode_gloss(text, max_len=MAX_TOKENS):
        """
        Encode whitespace-separated gloss tokens.

        Training vocabulary is built from the training split only.
        """
        words = str(text).lower().split()

        # Reserve BOS/EOS.
        words = words[: max(0, max_len - 2)]

        return (
            [G_BOS]
            + [gloss_vocab.get(w, G_UNK) for w in words]
            + [G_EOS]
        )

    # --------------------------------------------------------
    # 3. Text → Pose Dataset
    # --------------------------------------------------------
    #
    # IMPORTANT:
    # Do NOT pass the complete Pandas row into load_pose().
    # The pose cache is UID-based:
    #
    #     load_pose(uid, ...)
    #
    # We also avoid .iloc[idx] during every __getitem__ call.
    # --------------------------------------------------------

    class TextToPoseDataset(Dataset):

        def __init__(
            self,
            frame,
            output_frames=POSE_OUTPUT_FRAMES,
        ):
            frame = frame.reset_index(drop=True)

            self.uids = frame["uid"].astype(str).tolist()
            self.glosses = frame["gloss"].astype(str).tolist()
            self.output_frames = int(output_frames)

        def __len__(self):
            return len(self.uids)

        def __getitem__(self, idx):

            uid = self.uids[idx]
            gloss = self.glosses[idx]

            tokens = torch.tensor(
                encode_gloss(gloss),
                dtype=torch.long,
            )

            # UID-based pose loading.
            pose, _ = load_pose(
                uid,
                feature_mask=FULL_FEATURE_MASK,
                max_frames=self.output_frames,
            )

            pose = torch.from_numpy(
                np.ascontiguousarray(pose)
            ).float()

            return tokens, pose

    # --------------------------------------------------------
    # 4. Collate
    # --------------------------------------------------------

    def collate_text_pose(batch):

        tokens, poses = zip(*batch)

        max_len = max(
            len(tokens_i)
            for tokens_i in tokens
        )

        padded = torch.full(
            (
                len(tokens),
                max_len,
            ),
            G_PAD,
            dtype=torch.long,
        )

        for i, tokens_i in enumerate(tokens):
            padded[
                i,
                :len(tokens_i)
            ] = tokens_i

        poses = torch.stack(poses)

        return padded, poses

    # --------------------------------------------------------
    # 5. Datasets
    # --------------------------------------------------------

    train_pose_ds = TextToPoseDataset(
        t2p_train,
        output_frames=POSE_OUTPUT_FRAMES,
    )

    val_pose_ds = TextToPoseDataset(
        t2p_val,
        output_frames=POSE_OUTPUT_FRAMES,
    )

    test_pose_ds = TextToPoseDataset(
        t2p_test,
        output_frames=POSE_OUTPUT_FRAMES,
    )

    print()
    print("Text→Pose datasets:")
    print(f"  train: {len(train_pose_ds):,}")
    print(f"  val  : {len(val_pose_ds):,}")
    print(f"  test : {len(test_pose_ds):,}")
    print(f"  gloss vocabulary: {len(gloss_vocab):,}")
    print(f"  output frames: {POSE_OUTPUT_FRAMES}")
    print(f"  feature dimension: {LANDMARK_DIM}")

    # --------------------------------------------------------
    # 6. DataLoaders
    # --------------------------------------------------------

    train_pose_loader = DataLoader(
        train_pose_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_text_pose,
        num_workers=EVAL_NUM_WORKERS,
        pin_memory=PIN_MEMORY and DEVICE.type=="cuda",
    )

    val_pose_loader = DataLoader(
        val_pose_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_text_pose,
        num_workers=EVAL_NUM_WORKERS,
        pin_memory=PIN_MEMORY and DEVICE.type=="cuda",
    )

    test_pose_loader = DataLoader(
        test_pose_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_text_pose,
        num_workers=EVAL_NUM_WORKERS,
        pin_memory=PIN_MEMORY and DEVICE.type=="cuda",
    )

    # --------------------------------------------------------
    # 7. Text → Pose model
    # --------------------------------------------------------

    class TextToPoseGRU(nn.Module):

        def __init__(
            self,
            vocab_size,
            feature_dim,
            hidden=192,
            output_frames=POSE_OUTPUT_FRAMES,
        ):
            super().__init__()

            self.embed = nn.Embedding(
                vocab_size,
                hidden,
                padding_idx=G_PAD,
            )

            self.context = nn.Linear(
                hidden,
                hidden,
            )

            self.time = nn.Embedding(
                output_frames,
                hidden,
            )

            self.cell = nn.GRUCell(
                feature_dim + hidden,
                hidden,
            )

            self.out = nn.Linear(
                hidden,
                feature_dim,
            )

            self.output_frames = int(output_frames)
            self.feature_dim = int(feature_dim)

        def encode_text(self, tokens):

            mask = (
                tokens != G_PAD
            ).float().unsqueeze(-1)

            emb = self.embed(tokens) * mask

            pooled = (
                emb.sum(dim=1)
                / mask.sum(dim=1).clamp(min=1.0)
            )

            return torch.tanh(
                self.context(pooled)
            )

        def forward(
            self,
            tokens,
            teacher=None,
        ):

            hidden = self.encode_text(tokens)

            batch_size = tokens.size(0)

            prev = torch.zeros(
                batch_size,
                self.feature_dim,
                device=tokens.device,
                dtype=hidden.dtype,
            )

            outputs = []

            for t in range(self.output_frames):

                # Avoid constructing a new tensor with every
                # timestep from Python unnecessarily.
                time_index = torch.tensor(
                    t,
                    device=tokens.device,
                    dtype=torch.long,
                )

                ctx = self.time(
                    time_index
                ).expand(
                    batch_size,
                    -1,
                )

                hidden = self.cell(
                    torch.cat(
                        [prev, ctx],
                        dim=-1,
                    ),
                    hidden,
                )

                pred = self.out(hidden)

                outputs.append(pred)

                if teacher is not None:
                    prev = teacher[:, t]
                else:
                    prev = pred

            return torch.stack(
                outputs,
                dim=1,
            )

    # --------------------------------------------------------
    # 8. Motion-aware pose loss
    # --------------------------------------------------------

    def pose_loss(pred, target):

        position_loss = nn.functional.mse_loss(
            pred,
            target,
        )

        pred_velocity = (
            pred[:, 1:]
            - pred[:, :-1]
        )

        target_velocity = (
            target[:, 1:]
            - target[:, :-1]
        )

        velocity_loss = nn.functional.mse_loss(
            pred_velocity,
            target_velocity,
        )

        return (
            position_loss
            + 0.25 * velocity_loss
        )

    # --------------------------------------------------------
    # 9. Model + optimizer
    # --------------------------------------------------------

    pose_model = TextToPoseGRU(
        vocab_size=len(gloss_vocab),
        feature_dim=LANDMARK_DIM,
    ).to(DEVICE)

    pose_opt = torch.optim.AdamW(
        pose_model.parameters(),
        lr=1e-3,
        weight_decay=1e-4,
    )

    # Use a dedicated scaler for this optional experiment.
    # Do not reuse the primary Sign→Text optimizer/scaler state.
    if USE_AMP and DEVICE.type == "cuda":
        pose_scaler = torch.cuda.amp.GradScaler(
            enabled=True
        )
    else:
        pose_scaler = torch.cuda.amp.GradScaler(
            enabled=False
        )

    # --------------------------------------------------------
    # 10. Training
    # --------------------------------------------------------

    best = float("inf")
    pose_history = []

    n_train_batches = len(
        train_pose_loader
    )

    print()
    print("=" * 60)
    print("TEXT → POSE TRAINING")
    print("=" * 60)

    for epoch in range(
        1,
        N_EPOCHS_POSE + 1,
    ):

        pose_model.train()

        tr_total = 0.0

        pose_opt.zero_grad(
            set_to_none=True
        )

        pbar = tqdm(
            train_pose_loader,
            desc="TextToPose train",
            leave=False,
            total=n_train_batches,
        )

        for i, (tokens, poses) in enumerate(pbar):

            tokens = tokens.to(
                DEVICE,
                non_blocking=PIN_MEMORY,
            )

            poses = poses.to(
                DEVICE,
                non_blocking=PIN_MEMORY,
            )

            with autocast_ctx():

                pred = pose_model(
                    tokens,
                    teacher=poses,
                )

                loss = pose_loss(
                    pred,
                    poses,
                )

            if not torch.isfinite(loss):
                raise RuntimeError(
                    "Non-finite Text-to-Pose loss."
                )

            scaled_loss = (
                loss
                / max(1, GRAD_ACCUM_STEPS)
            )

            pose_scaler.scale(
                scaled_loss
            ).backward()

            is_last = (
                i == n_train_batches - 1
            )

            if (
                (i + 1) % GRAD_ACCUM_STEPS == 0
                or is_last
            ):

                pose_scaler.unscale_(
                    pose_opt
                )

                torch.nn.utils.clip_grad_norm_(
                    pose_model.parameters(),
                    1.0,
                )

                pose_scaler.step(
                    pose_opt
                )

                pose_scaler.update()

                pose_opt.zero_grad(
                    set_to_none=True
                )

            tr_total += float(
                loss.detach().item()
            )

            pbar.set_postfix(
                loss=f"{tr_total / (i + 1):.4f}"
            )

            del (
                tokens,
                poses,
                pred,
                loss,
                scaled_loss,
            )

        # ----------------------------------------------------
        # Validation
        # ----------------------------------------------------

        pose_model.eval()

        va_total = 0.0

        with torch.no_grad():

            for tokens, poses in tqdm(
                val_pose_loader,
                desc="TextToPose eval",
                leave=False,
            ):

                tokens = tokens.to(
                    DEVICE,
                    non_blocking=PIN_MEMORY,
                )

                poses = poses.to(
                    DEVICE,
                    non_blocking=PIN_MEMORY,
                )

                with autocast_ctx():

                    pred = pose_model(
                        tokens,
                        teacher=None,
                    )

                    batch_loss = pose_loss(
                        pred,
                        poses,
                    )

                va_total += float(
                    batch_loss.detach().item()
                )

                del (
                    tokens,
                    poses,
                    pred,
                    batch_loss,
                )

        tr = (
            tr_total
            / max(1, n_train_batches)
        )

        va = (
            va_total
            / max(1, len(val_pose_loader))
        )

        pose_history.append(
            (epoch, tr, va)
        )

        print(
            f"[TextToPose] "
            f"epoch {epoch:02d} "
            f"train={tr:.5f} "
            f"val={va:.5f}"
        )

        # ----------------------------------------------------
        # Best checkpoint
        # ----------------------------------------------------

        if va < best:

            best = va

            torch.save(
                pose_model.state_dict(),
                str(
                    CHECKPOINT_DIR
                    / "signsage_text_to_pose_best.pt"
                ),
            )

        gc.collect()
        maybe_check_memory()

    # --------------------------------------------------------
    # 11. Restore best checkpoint
    # --------------------------------------------------------

    best_path = (
        CHECKPOINT_DIR
        / "signsage_text_to_pose_best.pt"
    )

    if not best_path.is_file():
        raise FileNotFoundError(
            "Text→Pose training completed but the "
            f"best checkpoint was not found: {best_path}"
        )

    pose_model.load_state_dict(
        torch.load(
            str(best_path),
            map_location=DEVICE,
        )
    )

    pose_model.eval()

    # Release temporary dataframe references.
    del (
        t2p_train,
        t2p_val,
        t2p_test,
    )

    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    print(
        "\nText→Pose experiment completed. "
        f"Best validation loss: {best:.6f}"
    )

else:

    gloss_fn = None
    pose_model = None
    pose_history = []

    retrieval_dtw = []
    retrieval_mse = []
    generative_dtw = []
    generative_mse = []

    gt_motion = []
    retrieval_motion = []
    generative_motion = []

    t2p_metrics = {}

    print(
        "Text→Pose branch skipped. "
        "It remains available as an optional M.Tech research experiment."
    )

Text→Pose branch skipped. It remains available as an optional M.Tech research experiment.


In [40]:
if RUN_TEXT_TO_POSE:
    # 12a. Record Text→Pose experiment history and checkpoint.
    pose_ckpt = CHECKPOINT_DIR / "signsage_text_to_pose_best.pt"
    pose_best_val = float(min(v for _, _, v in pose_history)) if pose_history else float("nan")
    knowledge_insert(
        "experiments",
        ["run_id", "name", "family", "input_features", "epochs", "batch_size",
         "learning_rate", "weight_decay", "checkpoint_path", "best_validation_loss"],
        [RUN_ID, "Text→Pose GRU", "Text→Sign", "ISL gloss → pose", N_EPOCHS_POSE, BATCH_SIZE,
         1e-3, 1e-4, str(pose_ckpt), pose_best_val],
        delete_keys={"run_id": RUN_ID, "name": "Text→Pose GRU"},
    )
    for epoch, tr_loss, val_loss in pose_history:
        knowledge_metric("Text→Pose GRU", f"train_loss_epoch_{epoch}", tr_loss, "train")
        knowledge_metric("Text→Pose GRU", f"val_loss_epoch_{epoch}", val_loss, "validation")
    knowledge_artifact("text_to_pose_gru", "checkpoint", pose_ckpt, "Best Text→Pose GRU checkpoint.")
    print("Text→Pose experiment recorded.")
else:
    print("Text→Pose experiment record skipped.")


Text→Pose experiment record skipped.


## 13. Text → Sign evaluation

Retrieval and generation are compared on the same held-out test examples.

DTW is computed on normalized pose coordinates, and a simple normalized framewise MSE is also reported. Neither metric alone establishes sign-language quality; the static visualization and round-trip check are additional sanity checks.

In [41]:
if RUN_TEXT_TO_POSE:
    def dtw_distance(a, b):
        # Memory-safe O(nm) DP for short evaluation sequences.
        a = np.asarray(a, dtype=np.float32)
        b = np.asarray(b, dtype=np.float32)
        n, m = len(a), len(b)
        prev = np.full(m + 1, np.inf, dtype=np.float64)
        prev[0] = 0.0
        for i in range(1, n + 1):
            cur = np.full(m + 1, np.inf, dtype=np.float64)
            d = np.linalg.norm(a[i-1][None, :] - b, axis=1)
            cur[1:] = d + np.minimum(np.minimum(prev[1:], cur[:-1]), prev[:-1])
            prev = cur
        return float(prev[m])

    @torch.no_grad()
    def generate_pose(text):
        # BUG FIX: must use gloss_fn (Section 12), the same gloss function gloss_vocab and
        # pose_model were trained on. Hardcoding text_to_naive_gloss here meant that turning on
        # USE_LLM_GLOSS would silently break generation: tokens would be looked up against a
        # vocabulary built from a different gloss function and drift toward <unk>.
        tokens = torch.tensor([encode_gloss(gloss_fn(text))], dtype=torch.long, device=DEVICE)
        return pose_model(tokens, teacher=None)[0].cpu().numpy()

    def motion_energy(pose_seq):
        """Mean absolute frame-to-frame change -- a cheap proxy for 'how much does this
        sequence actually move'. A model that collapses to predicting a near-static/average
        pose can still score well on DTW/MSE against ground truth (regression-to-the-mean),
        so this is checked as an explicit, separate safeguard rather than trusting DTW/MSE
        alone to catch that failure mode."""
        if len(pose_seq) < 2:
            return 0.0
        diffs = pose_seq[1:] - pose_seq[:-1]
        return float(np.mean(np.abs(diffs)))

    eval_n = min(DTW_EVAL_N, len(df_test))
    retrieval_dtw, generative_dtw = [], []
    retrieval_mse, generative_mse = [], []
    gt_motion, retrieval_motion, generative_motion = [], [], []

    for _, row in df_test.iloc[:eval_n].iterrows():
        gt, _ = load_pose(row["uid"], FULL_FEATURE_MASK, max_frames=POSE_OUTPUT_FRAMES)
        gt = gt[:POSE_OUTPUT_FRAMES]

        retrieved, _, _ = retrieve_pose_for_text(row["text"])
        retrieved = resize_sequence(retrieved, POSE_OUTPUT_FRAMES)

        generated = generate_pose(row["text"])

        retrieval_dtw.append(dtw_distance(retrieved, gt))
        generative_dtw.append(dtw_distance(generated, gt))
        retrieval_mse.append(float(np.mean((retrieved - gt) ** 2)))
        generative_mse.append(float(np.mean((generated - gt) ** 2)))

        gt_motion.append(motion_energy(gt))
        retrieval_motion.append(motion_energy(retrieved))
        generative_motion.append(motion_energy(generated))

    print(f"Retrieval : DTW mean={np.mean(retrieval_dtw):.3f} ± {np.std(retrieval_dtw):.3f}; MSE={np.mean(retrieval_mse):.5f}")
    print(f"Generative: DTW mean={np.mean(generative_dtw):.3f} ± {np.std(generative_dtw):.3f}; MSE={np.mean(generative_mse):.5f}")

    print(f"\nMotion energy (mean |frame-to-frame change|) over {eval_n} test examples:")
    print(f"  Ground truth : {np.mean(gt_motion):.5f}")
    print(f"  Retrieval    : {np.mean(retrieval_motion):.5f}")
    print(f"  Generative   : {np.mean(generative_motion):.5f}")

    _gen_motion_ratio = (np.mean(generative_motion) / np.mean(gt_motion)) if np.mean(gt_motion) > 0 else float("nan")
    print(f"  Generative/Ground-truth motion ratio: {_gen_motion_ratio:.3f}")
    if np.isfinite(_gen_motion_ratio) and _gen_motion_ratio < 0.3:
        print("  WARNING: generated poses move far less than ground truth -- this suggests the\n"
              "  generator may have collapsed to a near-static/average pose rather than producing\n"
              "  real motion. A good DTW/MSE score alongside a low motion ratio should NOT be read\n"
              "  as evidence of successful pose generation on its own; inspect the visualization in\n"
              "  the next section before treating this as a positive result.")

    knowledge_observation("text_to_pose_motion_energy", {
        "eval_n": int(eval_n),
        "ground_truth_motion": float(np.mean(gt_motion)),
        "retrieval_motion": float(np.mean(retrieval_motion)),
        "generative_motion": float(np.mean(generative_motion)),
        "generative_to_gt_motion_ratio": float(_gen_motion_ratio) if np.isfinite(_gen_motion_ratio) else None,
    })
else:
    print("Text→Pose evaluation skipped.")


Text→Pose evaluation skipped.


In [42]:
if RUN_TEXT_TO_POSE:
    # 13a. Record Text→Pose evaluation metrics.
    t2p_metrics = {
        "retrieval_DTW_mean": float(np.mean(retrieval_dtw)),
        "retrieval_DTW_std": float(np.std(retrieval_dtw)),
        "retrieval_MSE_mean": float(np.mean(retrieval_mse)),
        "generative_DTW_mean": float(np.mean(generative_dtw)),
        "generative_DTW_std": float(np.std(generative_dtw)),
        "generative_MSE_mean": float(np.mean(generative_mse)),
        "ground_truth_motion_energy": float(np.mean(gt_motion)),
        "retrieval_motion_energy": float(np.mean(retrieval_motion)),
        "generative_motion_energy": float(np.mean(generative_motion)),
    }
    for metric_name, metric_value in t2p_metrics.items():
        knowledge_metric("Text→Pose", metric_name, metric_value, "test")
    knowledge_observation("text_to_pose_metrics", t2p_metrics)
    print("Text→Pose evaluation metrics recorded.")
else:
    print("Text→Pose metrics skipped.")


Text→Pose metrics skipped.


In [43]:
if RUN_TEXT_TO_POSE:
    # Qualitative Text -> Sign examples.
    for _, row in df_test.head(5).iterrows():
        retrieved, matched, sim = retrieve_pose_for_text(row["text"])
        generated = generate_pose(row["text"])
        print("TEXT      :", row["text"])
        print("GLOSS     :", row["gloss"], "(LLM)" if USE_LLM_GLOSS else "(naive)")
        print("RETRIEVED :", matched, f"(sim={sim:.3f}, frames={len(retrieved)})")
        print("GENERATED :", generated.shape)
        print()
else:
    print("Text→Pose qualitative examples skipped.")


Text→Pose qualitative examples skipped.


## 14. Pose visualization

In [44]:
if RUN_TEXT_TO_POSE:
    import matplotlib.pyplot as plt

    sample_text = df_test.iloc[0]["text"]
    generated = generate_pose(sample_text)

    fig, axes = plt.subplots(1, 6, figsize=(15, 3))
    for ax, fi in zip(axes, np.linspace(0, len(generated)-1, 6).astype(int)):
        pts = generated[fi].reshape(543, 3)
        # Show pose + hands; face is omitted from the static plot for readability.
        visible = np.concatenate([pts[:33], pts[501:543]], axis=0)
        ax.scatter(visible[:,0], -visible[:,1], s=7)
        ax.set_title(f"frame {fi}")
        ax.axis("equal")
        ax.axis("off")
    fig.suptitle(f'Text → generated pose: "{sample_text}"')
    plt.tight_layout()
    plt.show()
else:
    print("Pose visualization skipped in primary Sign→English run.")


Pose visualization skipped in primary Sign→English run.


## 15. Real round-trip sanity check

The previous notebook's round-trip cell was conceptually incomplete: it generated a pose, but then fed the **ground-truth pose from the original dataset row** into Sign → Text.

This cell fixes that. It generates a pose from text, then feeds the **generated pose** into the Sign → Text Transformer.

This is a model-level round trip, not a claim of live webcam performance.

In [45]:
if RUN_TEXT_TO_POSE:
    @torch.no_grad()
    def sign_text_from_pose_array(pose_array):
        # pose_array is normalized 543-landmark data.
        feature = pose_array[:, FULL_FEATURE_MASK]
        true_len = min(len(feature), MAX_FRAMES)
        feature = resize_sequence(feature, MAX_FRAMES)
        frames = torch.from_numpy(np.ascontiguousarray(feature)).float()
        return constrained_greedy_transformer(transformer_full, frames, true_len)

    roundtrip_text = df_test.iloc[0]["text"]
    roundtrip_gloss = text_to_naive_gloss(roundtrip_text)
    roundtrip_pose = generate_pose(roundtrip_text)
    roundtrip_recovered = sign_text_from_pose_array(roundtrip_pose)

    print("Original text :", roundtrip_text)
    print("Gloss         :", roundtrip_gloss)
    print("Generated pose:", roundtrip_pose.shape)
    print("Recovered text:", roundtrip_recovered)
else:
    roundtrip_text=None; roundtrip_gloss=None; roundtrip_pose=None; roundtrip_recovered=None
    print("Text→Pose round-trip skipped. Primary validation is direct held-out Sign→English inference.")


Text→Pose round-trip skipped. Primary validation is direct held-out Sign→English inference.


## 15. Live inference contract

The notebook's model input is **a sequence of 543 MediaPipe-compatible landmarks**, not a single webcam frame.
The web backend must collect a motion segment, apply the exact normalization below, then call this function.
No raw video is required by the trained model.

In [46]:
@torch.inference_mode()
def prepare_live_landmarks(landmarks_xyz):
    """Convert T×543×3 camera landmarks into the exact model representation."""
    arr=np.asarray(landmarks_xyz,dtype=np.float32)
    if arr.ndim!=3 or arr.shape[1:]!=(543,3):
        raise ValueError(f"Expected (T,543,3), received {arr.shape}")
    if len(arr)<2:
        raise ValueError("Need at least two frames.")
    flat=arr.reshape(len(arr),-1)
    norm=normalize_pose_sequence(flat)
    true_len=min(len(norm),MAX_FRAMES)
    return resize_sequence(norm,MAX_FRAMES),true_len

def live_motion_energy(landmarks_xyz):
    arr=np.asarray(landmarks_xyz,dtype=np.float32)
    if len(arr)<2:return 0.0
    # Activity gate is computed on root-centered, normalized landmarks so camera
    # translation/scale does not dominate it.
    norm=normalize_pose_sequence(arr.reshape(len(arr),-1))
    return float(np.mean(np.abs(np.diff(norm,axis=0))))

def predict_live_landmarks(landmarks_xyz, min_motion=0.002, min_confidence=None):
    processed,true_len=prepare_live_landmarks(landmarks_xyz)
    motion=live_motion_energy(landmarks_xyz)
    text,confidence=constrained_greedy_transformer(
        transformer_full,torch.from_numpy(processed).float(),true_len,return_confidence=True)
    if min_confidence is None:
        min_confidence=float(globals().get("LIVE_CONFIDENCE_FLOOR",0.0))
    accepted=bool(motion>=min_motion and confidence>=min_confidence and normalize_text(text))
    return {
        "text": text if accepted else "",
        "accepted": accepted,
        "motion_energy": motion,
        "confidence_proxy": confidence,
        "reason": "ok" if accepted else ("insufficient_motion" if motion<min_motion else "low_model_confidence"),
    }

print("Live inference contract defined: MediaPipe landmarks → shared normalization → Transformer → English.")


Live inference contract defined: MediaPipe landmarks → shared normalization → Transformer → English.


In [47]:
## 15a. Export a standalone backend inference package + immediate consistency check
#
# Fix (BLOCKING #2): the deployment bundle previously contained only the
# checkpoint + tokenizer + prose-description JSON. The backend had no actual
# code to run normalize_pose_sequence/resize_sequence/prepare_live_landmarks/
# SignTransformer/constrained_greedy_transformer -- only a paraphrased text
# description, which risks silent numerical drift if hand-reimplemented.
#
# This cell writes a small, self-contained Python package containing the
# ACTUAL logic used by this notebook (same formulas, same order of
# operations), refactored only so MAX_FRAMES/PAD_IDX/DEVICE/etc. are explicit
# parameters loaded from the deployment manifest instead of implicit notebook
# globals. It has no dependency on pose_format, the iSign dataset, or any
# /kaggle/* path.

# --------------------------------------------------------------------------
# Original in-memory contract check (unchanged): shapes + activity gate.
# --------------------------------------------------------------------------
sample_frames, sample_len, sample_tokens = test_ds_full[0]
assert sample_frames.shape == (MAX_FRAMES, LANDMARK_DIM)
assert 1 <= sample_len <= MAX_FRAMES
contract_input = np.zeros((max(2, sample_len), 543, 3), dtype=np.float32)
contract_processed, contract_len = prepare_live_landmarks(contract_input)
assert contract_processed.shape == (MAX_FRAMES, LANDMARK_DIM)
assert contract_len == min(max(2, sample_len), MAX_FRAMES)
zero_result = predict_live_landmarks(contract_input, min_motion=0.002, min_confidence=0.0)
assert not zero_result["accepted"], "Activity gate failed: constant landmarks were accepted."
print("Live inference contract self-test passed (shapes + activity gate).")

INFERENCE_PKG_DIR = REPORT_DIR / "signsage_inference"
INFERENCE_PKG_DIR.mkdir(parents=True, exist_ok=True)

_inference_package_files = {
    "__init__.py": "\"\"\"SignSage Sign->Text backend inference package.\n\nUsage:\n    from signsage_inference.loader import load_pipeline\n    pipeline = load_pipeline(\"/path/to/signsage_deployment_bundle\")\n    result = pipeline.transcribe(landmarks_xyz)  # (T, 543, 3) numpy array\n    print(result.text)\n\nThis package has no dependency on the iSign dataset, pose_format, Kaggle\npaths, or the training notebook. It only needs: torch, numpy, sentencepiece,\nand the four files shipped in the deployment bundle (checkpoint, tokenizer\nmodel, sign_to_text_deployment.json, landmark_contract.json).\n\"\"\"\nfrom .loader import load_pipeline, SignToTextPipeline, TranscriptionResult\nfrom .config import SignSageConfig\n\n__version__ = \"1.0.0\"\n\n__all__ = [\n    \"load_pipeline\",\n    \"SignToTextPipeline\",\n    \"TranscriptionResult\",\n    \"SignSageConfig\",\n]\n",
    "config.py": "\"\"\"\nSignSage deployment configuration loader.\n\nReads the JSON artifacts written by the training notebook\n(sign_to_text_deployment.json, landmark_contract.json) into a single\nexplicit config object. Nothing in this package reads a notebook-global\nvariable, a Kaggle path, or the iSign dataset -- everything needed at\ninference time comes from these two JSON files plus the checkpoint's own\nembedded model_config (read directly in loader.py).\n\"\"\"\nfrom __future__ import annotations\n\nimport json\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nfrom .preprocessing import TOTAL_LANDMARKS, COORDINATES_PER_LANDMARK, RAW_FEATURE_DIM, NORMALIZATION_VERSION\n\nEXPECTED_MODEL_INPUT_DIM = RAW_FEATURE_DIM * 2  # 3258: raw frame concatenated with its delta\n\n\n@dataclass\nclass SignSageConfig:\n    raw_feature_dim: int\n    model_input_dim: int\n    max_frames: int\n    max_tokens: int\n    max_decode_tokens: int\n    total_landmarks: int\n    coordinates_per_landmark: int\n    pad_id: int\n    unk_id: int\n    bos_id: int\n    eos_id: int\n    min_output_tokens: int\n    no_repeat_ngram_size: int\n    repetition_penalty: float\n    normalization_version: str\n    tokenizer_model_path: str\n    checkpoint_path: str\n    landmark_contract: dict\n    deployment_manifest: dict\n\n    @classmethod\n    def from_bundle(cls, bundle_dir) -> \"SignSageConfig\":\n        bundle_dir = Path(bundle_dir)\n        manifest_path = bundle_dir / \"sign_to_text_deployment.json\"\n        contract_path = bundle_dir / \"landmark_contract.json\"\n        for p in (manifest_path, contract_path):\n            if not p.is_file():\n                raise FileNotFoundError(f\"Required deployment artifact missing: {p}\")\n\n        with open(manifest_path, \"r\", encoding=\"utf-8\") as f:\n            manifest = json.load(f)\n        with open(contract_path, \"r\", encoding=\"utf-8\") as f:\n            contract = json.load(f)\n\n        special = manifest[\"special_token_ids\"]\n        decoding = manifest[\"decoding\"]\n\n        # --- Self-consistency validation (fail loudly, never silently load a\n        # mismatched bundle) -----------------------------------------------\n        errors = []\n\n        if int(contract.get(\"total_landmark_count\", -1)) != TOTAL_LANDMARKS:\n            errors.append(\n                f\"landmark_contract.json total_landmark_count={contract.get('total_landmark_count')} \"\n                f\"!= expected {TOTAL_LANDMARKS}\"\n            )\n        if int(contract.get(\"coordinates_per_landmark\", -1)) != COORDINATES_PER_LANDMARK:\n            errors.append(\n                f\"landmark_contract.json coordinates_per_landmark={contract.get('coordinates_per_landmark')} \"\n                f\"!= expected {COORDINATES_PER_LANDMARK}\"\n            )\n        if int(manifest.get(\"feature_dim\", -1)) != RAW_FEATURE_DIM:\n            errors.append(\n                f\"sign_to_text_deployment.json feature_dim={manifest.get('feature_dim')} \"\n                f\"!= expected {RAW_FEATURE_DIM}\"\n            )\n        if int(manifest.get(\"model_input_dim\", -1)) != EXPECTED_MODEL_INPUT_DIM:\n            errors.append(\n                f\"sign_to_text_deployment.json model_input_dim={manifest.get('model_input_dim')} \"\n                f\"!= expected {EXPECTED_MODEL_INPUT_DIM}\"\n            )\n        _full_mask = contract.get(\"full_feature_mask\")\n        if not isinstance(_full_mask, list) or len(_full_mask) != RAW_FEATURE_DIM:\n            errors.append(\n                f\"landmark_contract.json full_feature_mask has length \"\n                f\"{len(_full_mask) if isinstance(_full_mask, list) else 'N/A'}, expected {RAW_FEATURE_DIM}\"\n            )\n        manifest_norm_version = manifest.get(\"normalization_version\")\n        contract_norm_version = contract.get(\"normalization_version\")\n        if manifest_norm_version != NORMALIZATION_VERSION:\n            errors.append(\n                f\"sign_to_text_deployment.json normalization_version={manifest_norm_version!r} \"\n                f\"!= expected {NORMALIZATION_VERSION!r} (this package's preprocessing.py version)\"\n            )\n        if contract_norm_version != manifest_norm_version:\n            errors.append(\n                f\"landmark_contract.json normalization_version={contract_norm_version!r} \"\n                f\"!= sign_to_text_deployment.json normalization_version={manifest_norm_version!r}\"\n            )\n        if int(contract.get(\"raw_feature_dim\", -1)) != int(manifest.get(\"feature_dim\", -2)):\n            errors.append(\n                f\"landmark_contract.json raw_feature_dim={contract.get('raw_feature_dim')} \"\n                f\"!= sign_to_text_deployment.json feature_dim={manifest.get('feature_dim')}\"\n            )\n        if int(contract.get(\"final_model_input_dim\", -1)) != int(manifest.get(\"model_input_dim\", -2)):\n            errors.append(\n                f\"landmark_contract.json final_model_input_dim={contract.get('final_model_input_dim')} \"\n                f\"!= sign_to_text_deployment.json model_input_dim={manifest.get('model_input_dim')}\"\n            )\n        try:\n            _ids = (int(special[\"pad\"]), int(special[\"unk\"]), int(special[\"bos\"]), int(special[\"eos\"]))\n            if len(set(_ids)) != 4:\n                errors.append(f\"special_token_ids are not four distinct values: {special}\")\n        except (KeyError, TypeError, ValueError):\n            errors.append(f\"sign_to_text_deployment.json special_token_ids is malformed: {special}\")\n\n        if errors:\n            raise ValueError(\n                \"Deployment bundle failed self-consistency validation:\\n  - \" + \"\\n  - \".join(errors)\n            )\n\n        return cls(\n            raw_feature_dim=int(manifest[\"feature_dim\"]),\n            model_input_dim=int(manifest[\"model_input_dim\"]),\n            max_frames=int(manifest[\"max_frames\"]),\n            max_tokens=int(manifest[\"max_tokens\"]),\n            max_decode_tokens=int(decoding[\"max_decode_tokens\"]),\n            total_landmarks=int(contract[\"total_landmark_count\"]),\n            coordinates_per_landmark=int(contract[\"coordinates_per_landmark\"]),\n            pad_id=int(special[\"pad\"]),\n            unk_id=int(special[\"unk\"]),\n            bos_id=int(special[\"bos\"]),\n            eos_id=int(special[\"eos\"]),\n            min_output_tokens=int(decoding[\"min_output_tokens\"]),\n            no_repeat_ngram_size=int(decoding[\"no_repeat_ngram_size\"]),\n            repetition_penalty=float(decoding[\"repetition_penalty\"]),\n            normalization_version=str(manifest.get(\"normalization_version\", \"unknown\")),\n            tokenizer_model_path=str(manifest[\"tokenizer_model\"]),\n            checkpoint_path=str(manifest[\"checkpoint\"]),\n            landmark_contract=contract,\n            deployment_manifest=manifest,\n        )\n",
    "model.py": "\"\"\"\nSignSage Sign->Text model definition.\n\nVerbatim port of the training notebook's FrameProjection/PositionalEncoding/\nSignTransformer classes (Section 8). MAX_FRAMES/MAX_TOKENS/PAD_IDX are\nconverted from implicit notebook globals into explicit constructor\narguments so this class has no dependency on training-notebook state. The\nforward/encode/decode computation itself is byte-for-byte unchanged.\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\n\n\nclass FrameProjection(nn.Module):\n    \"\"\"Project spatial landmarks plus explicit frame-to-frame motion features.\"\"\"\n\n    def __init__(self, input_dim, d_model):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Linear(input_dim, 384),\n            nn.LayerNorm(384),\n            nn.GELU(),\n            nn.Dropout(0.10),\n            nn.Linear(384, d_model),\n            nn.LayerNorm(d_model),\n        )\n        self.temporal = nn.Conv1d(d_model, d_model, kernel_size=5, padding=2, groups=d_model)\n\n    def forward(self, x):\n        # Explicit velocity is zero for the first frame. This makes the temporal\n        # signal available before attention rather than asking the encoder to infer\n        # all derivatives from absolute coordinates alone.\n        delta = torch.zeros_like(x)\n        delta[:, 1:] = x[:, 1:] - x[:, :-1]\n        y = self.net(torch.cat([x, delta], dim=-1))\n        y = self.temporal(y.transpose(1, 2)).transpose(1, 2)\n        return y\n\n\nclass PositionalEncoding(nn.Module):\n    def __init__(self, d_model, max_len):\n        super().__init__()\n        pe = torch.zeros(max_len, d_model)\n        pos = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)\n        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-np.log(10000.0) / d_model))\n        pe[:, 0::2] = torch.sin(pos * div)\n        pe[:, 1::2] = torch.cos(pos * div)\n        self.register_buffer(\"pe\", pe.unsqueeze(0))\n\n    def forward(self, x):\n        return x + self.pe[:, : x.size(1)]\n\n\nclass SignTransformer(nn.Module):\n    \"\"\"Same architecture as training. max_frames/max_tokens/pad_idx are now\n    explicit constructor arguments (previously the notebook globals\n    MAX_FRAMES, MAX_TOKENS, PAD_IDX) so this class has no external state\n    dependency and can be reconstructed from a checkpoint's own model_config\n    in a completely fresh process.\n    \"\"\"\n\n    def __init__(\n        self,\n        input_dim,\n        vocab_size,\n        max_frames,\n        max_tokens,\n        pad_idx,\n        d_model=192,\n        nhead=4,\n        layers=3,\n        ff=512,\n        dropout=0.15,\n    ):\n        super().__init__()\n        self.pad_idx = pad_idx\n        self.input_proj = FrameProjection(input_dim, d_model)\n        self.src_pos = PositionalEncoding(d_model, max_frames + 4)\n        self.tok_embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)\n        self.tgt_pos = PositionalEncoding(d_model, max_tokens + 4)\n        enc = nn.TransformerEncoderLayer(d_model, nhead, ff, dropout, batch_first=True, norm_first=False)\n        dec = nn.TransformerDecoderLayer(d_model, nhead, ff, dropout, batch_first=True, norm_first=False)\n        self.encoder = nn.TransformerEncoder(enc, layers)\n        self.decoder = nn.TransformerDecoder(dec, layers)\n        self.out = nn.Linear(d_model, vocab_size)\n        self.scale = d_model ** 0.5\n\n    def encode(self, frames, src_pad):\n        return self.encoder(self.src_pos(self.input_proj(frames)), src_key_padding_mask=src_pad)\n\n    def decode(self, tokens, memory, tgt_mask, src_pad, tgt_pad=None):\n        x = self.tgt_pos(self.tok_embed(tokens) * self.scale)\n        return self.out(\n            self.decoder(\n                x, memory, tgt_mask=tgt_mask,\n                tgt_key_padding_mask=tgt_pad,\n                memory_key_padding_mask=src_pad,\n            )\n        )\n\n    def forward(self, frames, src_pad, tokens, tgt_mask):\n        return self.decode(\n            tokens, self.encode(frames, src_pad), tgt_mask, src_pad,\n            tgt_pad=tokens.eq(self.pad_idx),\n        )\n\n\ndef padding_mask(lengths, max_len):\n    return torch.arange(max_len, device=lengths.device).unsqueeze(0) >= lengths.unsqueeze(1)\n\n\ndef causal_mask(n, device):\n    return torch.triu(torch.ones(n, n, device=device, dtype=torch.bool), diagonal=1)\n",
    "preprocessing.py": "\"\"\"\nSignSage inference preprocessing.\n\nVerbatim port of the training notebook's pose normalization/resizing cell\n(Section 5a). This module has no dependency on the iSign dataset, on\npose_format, on Kaggle paths, or on any notebook-global variable -- the only\nexternal dependencies are numpy and the Python standard library.\n\nBackend usage:\n    from signsage_inference.preprocessing import prepare_live_landmarks\n    processed, true_len = prepare_live_landmarks(landmarks_xyz, max_frames=160)\n\"\"\"\nfrom __future__ import annotations\n\nimport warnings\n\nimport numpy as np\n\nTOTAL_LANDMARKS = 543\nCOORDINATES_PER_LANDMARK = 3\nRAW_FEATURE_DIM = TOTAL_LANDMARKS * COORDINATES_PER_LANDMARK  # 1629\n\n# Bump this whenever normalize_pose_sequence()'s algorithm changes. config.py\n# asserts the deployment manifest's declared normalization_version equals\n# this constant, so an old manifest paired with new preprocessing code (or\n# vice versa) is caught at load time instead of silently mismatching.\nNORMALIZATION_VERSION = \"pose_normalize_v1\"\n\n# MediaPipe pose landmarks: 23 = left hip, 24 = right hip. These are literal\n# anatomical-layout constants used by the training-time normalization itself\n# (not configuration) -- they are correct as long as the component ordering\n# in landmark_contract.json places POSE_LANDMARKS first with hips at local\n# indices 23/24, which loader.py cross-checks at load time.\n_HIP_INDICES = (23, 24)\n_BODY_LANDMARK_COUNT = 33\n\n\ndef normalize_pose_sequence(flat: np.ndarray) -> np.ndarray:\n    \"\"\"Root-center and scale one pose sequence.\n\n    Input:\n        flat: [frames, 543 * 3]\n    Output:\n        float32 array with shape [frames, 543 * 3]\n\n    This is a verbatim port of the training-time normalize_pose_sequence():\n    same hip-center reference, same body-landmark scale estimate, same\n    NaN/Inf handling, same order of operations.\n    \"\"\"\n    x = flat.reshape(flat.shape[0], TOTAL_LANDMARKS, COORDINATES_PER_LANDMARK).astype(np.float32)\n\n    hip = x[:, list(_HIP_INDICES), :]\n\n    with warnings.catch_warnings():\n        warnings.filterwarnings(\"ignore\", message=\"Mean of empty slice\")\n        center = np.nanmean(hip, axis=1, keepdims=True)\n\n    bad_center = ~np.isfinite(center).all(axis=(1, 2))\n\n    if bad_center.any():\n        with warnings.catch_warnings():\n            warnings.filterwarnings(\"ignore\", message=\"Mean of empty slice\")\n            fallback_center = np.nanmean(\n                x[bad_center, :_BODY_LANDMARK_COUNT, :],\n                axis=1,\n                keepdims=True,\n            )\n        center[bad_center] = fallback_center\n\n    x = x - center\n\n    body = x[:, :_BODY_LANDMARK_COUNT, :]\n\n    with warnings.catch_warnings():\n        warnings.filterwarnings(\"ignore\", message=\"Degrees of freedom\")\n        scale_per_frame = np.nanstd(body.reshape(body.shape[0], -1), axis=1)\n\n    valid_scale = scale_per_frame[np.isfinite(scale_per_frame) & (scale_per_frame > 1e-6)]\n\n    if valid_scale.size == 0:\n        scale = 1.0\n    else:\n        scale = np.nanmedian(valid_scale)\n        if not np.isfinite(scale) or scale <= 1e-6:\n            scale = 1.0\n\n    x = x / float(scale)\n    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)\n\n    return x.reshape(x.shape[0], -1).astype(np.float32)\n\n\ndef resize_sequence(seq: np.ndarray, target_len: int) -> np.ndarray:\n    \"\"\"Resize/pad a sequence to target_len (verbatim port).\n\n    Long sequences: uniformly sampled down to target_len.\n    Short sequences: zero-padded at the end.\n    \"\"\"\n    src_len = seq.shape[0]\n\n    if src_len <= 0:\n        raise ValueError(\"Empty pose sequence\")\n\n    if src_len == target_len:\n        return seq.astype(np.float32, copy=False)\n\n    if src_len > target_len:\n        idx = np.linspace(0, src_len - 1, target_len)\n        return seq[np.round(idx).astype(np.int64)].astype(np.float32, copy=False)\n\n    out = np.zeros((target_len, seq.shape[1]), dtype=np.float32)\n    out[:src_len] = seq\n    return out\n\n\ndef prepare_live_landmarks(landmarks_xyz, max_frames: int):\n    \"\"\"Convert T x 543 x 3 camera landmarks into the exact model representation.\n\n    Same logic/signature as the notebook's prepare_live_landmarks(); MAX_FRAMES\n    is now an explicit parameter (sourced from the deployment manifest by the\n    caller) instead of an implicit notebook global.\n    \"\"\"\n    arr = np.asarray(landmarks_xyz, dtype=np.float32)\n    if arr.ndim != 3 or arr.shape[1:] != (TOTAL_LANDMARKS, COORDINATES_PER_LANDMARK):\n        raise ValueError(\n            f\"Expected (T,{TOTAL_LANDMARKS},{COORDINATES_PER_LANDMARK}), received {arr.shape}\"\n        )\n    if len(arr) < 2:\n        raise ValueError(\"Need at least two frames.\")\n    flat = arr.reshape(len(arr), -1)\n    norm = normalize_pose_sequence(flat)\n    true_len = min(len(norm), max_frames)\n    return resize_sequence(norm, max_frames), true_len\n\n\ndef live_motion_energy(landmarks_xyz) -> float:\n    \"\"\"Mean absolute frame-to-frame delta on normalized landmarks (verbatim port).\"\"\"\n    arr = np.asarray(landmarks_xyz, dtype=np.float32)\n    if len(arr) < 2:\n        return 0.0\n    norm = normalize_pose_sequence(arr.reshape(len(arr), -1))\n    return float(np.mean(np.abs(np.diff(norm, axis=0))))\n",
    "decoder.py": "\"\"\"\nSignSage autoregressive greedy decoder.\n\nVerbatim port of the training notebook's constrained_greedy_transformer(),\n_ban_repeated_ngram(), and decode_ids() (Section 8 / Section 4). DEVICE,\nPAD_IDX/BOS_IDX/UNK_IDX/EOS_IDX, MIN_OUTPUT_TOKENS, NO_REPEAT_NGRAM_SIZE,\nREPETITION_PENALTY, and MAX_DECODE_TOKENS are converted from notebook\nglobals into an explicit DecodingConfig, and the tokenizer is an explicit\nargument instead of a module-level `sp`. The per-step token-selection\narithmetic (log-softmax, repetition penalty, n-gram ban, greedy argmax) is\nbyte-for-byte unchanged -- this performs REAL autoregressive decoding from\nBOS with no ground-truth tokens, exactly as during evaluation.\n\"\"\"\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\n\nimport numpy as np\nimport torch\n\nfrom .model import causal_mask, padding_mask\n\n\ndef normalize_text(text) -> str:\n    return \" \".join(str(text).lower().strip().split())\n\n\n@dataclass\nclass DecodingConfig:\n    pad_id: int\n    unk_id: int\n    bos_id: int\n    eos_id: int\n    max_decode_tokens: int\n    min_output_tokens: int = 1\n    no_repeat_ngram_size: int = 3\n    repetition_penalty: float = 1.15\n\n\ndef decode_ids(ids, sp, cfg: DecodingConfig) -> str:\n    \"\"\"Verbatim port of the training notebook's decode_ids(), parameterized by\n    an explicit tokenizer + special-token config instead of module globals.\n    \"\"\"\n    clean = []\n    for idx in ids:\n        idx = int(idx)\n        if idx == cfg.eos_id:\n            break\n        if idx not in (cfg.bos_id, cfg.pad_id):\n            clean.append(idx)\n    if not clean:\n        return \"\"\n    return normalize_text(sp.decode(clean))\n\n\ndef _ban_repeated_ngram(logits, ids, n):\n    if n <= 1 or len(ids) < n - 1:\n        return logits\n    prefix = tuple(ids[-(n - 1):])\n    for j in range(len(ids) - n + 1):\n        ng = tuple(ids[j:j + n])\n        if ng[:-1] == prefix:\n            logits[ng[-1]] = -torch.inf\n    return logits\n\n\n@torch.no_grad()\ndef greedy_decode(model, frames, length, sp, cfg: DecodingConfig, device, return_confidence=False):\n    \"\"\"Verbatim port of constrained_greedy_transformer(): real autoregressive\n    decoding from BOS with no access to ground-truth tokens, ending at EOS or\n    cfg.max_decode_tokens.\n    \"\"\"\n    model.eval()\n    frames = frames.unsqueeze(0).to(device)\n    length_t = torch.tensor([int(length)], device=device)\n    src_pad = padding_mask(length_t, frames.size(1))\n    memory = model.encode(frames, src_pad)\n\n    ids = [cfg.bos_id]\n    logps = []\n    for step in range(cfg.max_decode_tokens):\n        tok = torch.tensor([ids], device=device)\n        logits = model.decode(tok, memory, causal_mask(tok.size(1), device), src_pad)[0, -1].float()\n        logits[cfg.pad_id] = -torch.inf\n        logits[cfg.bos_id] = -torch.inf\n        logits[cfg.unk_id] = -torch.inf\n        if step < cfg.min_output_tokens:\n            logits[cfg.eos_id] = -torch.inf\n        logits = _ban_repeated_ngram(logits, ids, cfg.no_repeat_ngram_size)\n        # Mild repetition penalty; do not use an aggressive penalty that can\n        # distort fluent English.\n        for tid in set(ids[1:]):\n            logits[tid] = (\n                logits[tid] / cfg.repetition_penalty\n                if logits[tid] >= 0\n                else logits[tid] * cfg.repetition_penalty\n            )\n        lp = torch.log_softmax(logits, dim=-1)\n        prob, tid = torch.max(lp, dim=-1)\n        tid = int(tid)\n        logps.append(float(prob))\n        ids.append(tid)\n        if tid == cfg.eos_id:\n            break\n\n    text = decode_ids(ids[1:], sp, cfg)\n    conf = float(np.exp(np.mean(logps))) if logps else 0.0\n    return (text, conf) if return_confidence else text\n",
    "loader.py": "\"\"\"\nSignSage backend loader.\n\nTies the checkpoint, tokenizer, deployment manifest, and landmark contract\ntogether into one object (SignToTextPipeline) with a single\ntranscribe(landmarks_xyz) entrypoint. This is the only class the FastAPI\nbackend needs to import.\n\n    from signsage_inference.loader import load_pipeline\n    pipeline = load_pipeline(\"/path/to/signsage_deployment_bundle\")\n    result = pipeline.transcribe(landmarks_xyz)   # landmarks_xyz: (T, 543, 3) np.ndarray\n    print(result.text)\n\nNo function in this package touches the iSign dataset, pose_format, Kaggle\npaths, or any training-notebook global. Everything is derived from the\nfiles inside the deployment bundle: the checkpoint, the tokenizer model,\nthe deployment manifest, and the landmark contract. load_pipeline() also\ncross-checks the checkpoint's own architecture/feature dimensions and the\ntokenizer's own special-token IDs against the manifest, and fails loudly on\nany mismatch rather than silently loading something inconsistent.\n\"\"\"\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nfrom .config import SignSageConfig\nfrom .model import SignTransformer\nfrom .preprocessing import prepare_live_landmarks, live_motion_energy\nfrom .decoder import DecodingConfig, greedy_decode, normalize_text\n\n\n@dataclass\nclass TranscriptionResult:\n    text: str\n    accepted: bool\n    motion_energy: float\n    confidence_proxy: float\n    reason: str\n\n\nclass SignToTextPipeline:\n    def __init__(self, model, sp, config: SignSageConfig, device):\n        self.model = model\n        self.sp = sp\n        self.config = config\n        self.device = device\n        self._decoding_cfg = DecodingConfig(\n            pad_id=config.pad_id, unk_id=config.unk_id,\n            bos_id=config.bos_id, eos_id=config.eos_id,\n            max_decode_tokens=config.max_decode_tokens,\n            min_output_tokens=config.min_output_tokens,\n            no_repeat_ngram_size=config.no_repeat_ngram_size,\n            repetition_penalty=config.repetition_penalty,\n        )\n\n    def transcribe(self, landmarks_xyz, min_motion: float = 0.002, min_confidence: float = 0.0) -> TranscriptionResult:\n        \"\"\"landmarks_xyz: raw (T, 543, 3) array from MediaPipe Holistic, in the\n        exact component order documented in landmark_contract.json. No\n        normalization/delta computation is expected from the caller -- this\n        method performs the entire training-time preprocessing + model +\n        decode pipeline internally.\n        \"\"\"\n        processed, true_len = prepare_live_landmarks(landmarks_xyz, max_frames=self.config.max_frames)\n        motion = live_motion_energy(landmarks_xyz)\n        text, confidence = greedy_decode(\n            self.model,\n            torch.from_numpy(processed).float(),\n            true_len,\n            self.sp,\n            self._decoding_cfg,\n            self.device,\n            return_confidence=True,\n        )\n        accepted = bool(motion >= min_motion and confidence >= min_confidence and normalize_text(text))\n        reason = \"ok\" if accepted else (\"insufficient_motion\" if motion < min_motion else \"low_model_confidence\")\n        return TranscriptionResult(\n            text=text if accepted else \"\",\n            accepted=accepted,\n            motion_energy=motion,\n            confidence_proxy=confidence,\n            reason=reason,\n        )\n\n\ndef load_pipeline(bundle_dir, device: str | None = None) -> SignToTextPipeline:\n    \"\"\"Load checkpoint + tokenizer + manifest + landmark contract from a\n    deployment bundle directory and return a ready-to-use SignToTextPipeline.\n\n    Raises on any inconsistency between the checkpoint, the tokenizer, and\n    the deployment manifest -- it never silently loads a mismatched bundle.\n    \"\"\"\n    import sentencepiece as spm\n\n    bundle_dir = Path(bundle_dir)\n    config = SignSageConfig.from_bundle(bundle_dir)\n\n    torch_device = torch.device(device) if device else torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n\n    checkpoint_path = bundle_dir / config.checkpoint_path\n    if not checkpoint_path.is_file():\n        raise FileNotFoundError(f\"Checkpoint not found: {checkpoint_path}\")\n    payload = torch.load(checkpoint_path, map_location=torch_device)\n\n    if \"state_dict\" not in payload or \"model_config\" not in payload:\n        raise ValueError(\"Checkpoint is missing 'state_dict'/'model_config'; cannot reconstruct the model.\")\n    mc = payload[\"model_config\"]\n\n    if int(mc[\"input_dim\"]) != config.model_input_dim:\n        raise ValueError(\n            f\"Checkpoint model_input_dim={mc['input_dim']} does not match \"\n            f\"deployment manifest model_input_dim={config.model_input_dim}.\"\n        )\n    if int(mc[\"raw_feature_dim\"]) != config.raw_feature_dim:\n        raise ValueError(\n            f\"Checkpoint raw_feature_dim={mc['raw_feature_dim']} does not match \"\n            f\"deployment manifest feature_dim={config.raw_feature_dim}.\"\n        )\n\n    vocab = payload.get(\"vocab\")\n    vocab_size = int(mc[\"vocab_size\"]) if vocab is None else len(vocab)\n\n    model = SignTransformer(\n        input_dim=int(mc[\"input_dim\"]),\n        vocab_size=vocab_size,\n        max_frames=int(mc.get(\"max_frames\", config.max_frames)),\n        max_tokens=int(mc.get(\"max_tokens\", config.max_tokens)),\n        pad_idx=config.pad_id,\n        d_model=int(mc.get(\"d_model\", 192)),\n        nhead=int(mc.get(\"nhead\", 4)),\n        layers=int(mc.get(\"layers\", 3)),\n        ff=int(mc.get(\"ff\", 512)),\n        dropout=0.0,  # inference: disable dropout regardless of the training value\n    ).to(torch_device)\n    model.load_state_dict(payload[\"state_dict\"])\n    model.eval()\n\n    tokenizer_path = bundle_dir / config.tokenizer_model_path\n    if not tokenizer_path.is_file():\n        raise FileNotFoundError(f\"Tokenizer model not found: {tokenizer_path}\")\n    sp = spm.SentencePieceProcessor(model_file=str(tokenizer_path))\n\n    if (sp.pad_id(), sp.unk_id(), sp.bos_id(), sp.eos_id()) != (config.pad_id, config.unk_id, config.bos_id, config.eos_id):\n        raise ValueError(\n            \"Tokenizer special-token IDs do not match the deployment manifest's \"\n            f\"special_token_ids: tokenizer=({sp.pad_id()},{sp.unk_id()},{sp.bos_id()},{sp.eos_id()}) \"\n            f\"manifest=({config.pad_id},{config.unk_id},{config.bos_id},{config.eos_id}).\"\n        )\n\n    return SignToTextPipeline(model=model, sp=sp, config=config, device=torch_device)\n",
    "requirements.txt": "torch>=2.0\nnumpy>=1.23\nsentencepiece>=0.1.99\n",
}

for _fname, _fcontent in _inference_package_files.items():
    (INFERENCE_PKG_DIR / _fname).write_text(_fcontent, encoding="utf-8")

print(f"\nStandalone inference package written: {INFERENCE_PKG_DIR}")
for _p in sorted(INFERENCE_PKG_DIR.iterdir()):
    print(" -", _p.name)

# --------------------------------------------------------------------------
# Real (non-normalized) sample fetcher, used only to build realistic test
# inputs for the consistency checks below and in Section 20. Mirrors the
# shape-extraction lines of _read_and_normalize_raw() (Section 5a) but stops
# BEFORE normalization -- this is what a live MediaPipe capture looks like
# before any preprocessing. The backend itself never calls this; it has no
# pose_format/iSign-archive dependency at all.
# --------------------------------------------------------------------------
def fetch_raw_live_style_landmarks(uid):
    raw_bytes = read_pose_bytes_for_uid(str(uid))
    pose = Pose.read(raw_bytes)
    data = np.asarray(np.ma.filled(pose.body.data, 0.0), dtype=np.float32)
    data = data[:, 0, :543, :]
    return data

# --------------------------------------------------------------------------
# Immediate consistency check: import the JUST-WRITTEN package fresh (not the
# in-memory notebook functions) and prove its preprocessing is numerically
# IDENTICAL to the training-time implementation on a REAL raw held-out sample
# -- not just structurally similar, and not on already-normalized/padded
# tensor data. This runs now (cheap) so a transcription mistake is caught
# immediately, long before the full fresh-load verification in Section 20.
# --------------------------------------------------------------------------
import sys, importlib
if str(REPORT_DIR) not in sys.path:
    sys.path.insert(0, str(REPORT_DIR))
for _mod in list(sys.modules):
    if _mod == "signsage_inference" or _mod.startswith("signsage_inference."):
        del sys.modules[_mod]
importlib.invalidate_caches()
import signsage_inference.preprocessing as _pkg_pre

_consistency_uid = str(df_test.iloc[0]["uid"])
_consistency_raw = fetch_raw_live_style_landmarks(_consistency_uid)
print(f"Fetched real raw landmark sample for UID={_consistency_uid}, shape={_consistency_raw.shape}")

_notebook_processed, _notebook_len = prepare_live_landmarks(_consistency_raw)
_package_processed, _package_len = _pkg_pre.prepare_live_landmarks(_consistency_raw, max_frames=MAX_FRAMES)

assert _notebook_len == _package_len, "Package prepare_live_landmarks length mismatch."
assert np.allclose(_notebook_processed, _package_processed, atol=0, rtol=0), (
    "Exported package preprocessing does NOT numerically match the training-time "
    "implementation on a real sample -- do not ship this bundle."
)
print("Consistency check PASSED: exported preprocessing is numerically identical "
      "to the in-notebook implementation on a real held-out sample.")

knowledge_artifact("signsage_inference_package", "python_package", INFERENCE_PKG_DIR,
                   "Self-contained backend inference package: preprocessing, model, decoder, loader.")



Live inference contract self-test passed (shapes + activity gate).

Standalone inference package written: /kaggle/working/signsage_outputs/reports/signsage_inference
 - __init__.py
 - config.py
 - decoder.py
 - loader.py
 - model.py
 - preprocessing.py
 - requirements.txt
Fetched real raw landmark sample for UID=62e79450f3da-1, shape=(81, 543, 3)
Consistency check PASSED: exported preprocessing is numerically identical to the in-notebook implementation on a real held-out sample.


## 16. Latency harness

These are **model-only** timings. They are not end-to-end browser → WebSocket → backend → model timings and must not be presented as the project's final system latency.

In [48]:
## 16. Latency harness
def measure_ms(fn,*args,runs=5):
    if DEVICE.type=="cuda": torch.cuda.synchronize()
    vals=[]
    for _ in range(runs):
        if DEVICE.type=="cuda": torch.cuda.synchronize()
        t0=time.perf_counter(); fn(*args)
        if DEVICE.type=="cuda": torch.cuda.synchronize()
        vals.append((time.perf_counter()-t0)*1000)
    return float(np.mean(vals)),float(np.std(vals))

sample_frames,sample_len,_=test_ds_full[0]
s2t_mean,s2t_std=measure_ms(constrained_greedy_transformer,transformer_full,sample_frames,sample_len)
t2p_mean=t2p_std=float("nan")
print(f"Sign→Text model inference: {s2t_mean:.2f} ± {s2t_std:.2f} ms")
print("This excludes MediaPipe, WebSocket, network and frontend rendering.")


Sign→Text model inference: 31.00 ± 0.83 ms
This excludes MediaPipe, WebSocket, network and frontend rendering.


In [49]:
# 16a. Record runtime and deployment metadata.
knowledge_metric("Runtime","sign_to_text_latency_ms_mean",s2t_mean,"inference")
knowledge_metric("Runtime","sign_to_text_latency_ms_std",s2t_std,"inference")
knowledge_observation("deployment_contract",{
    "input":"normalized 543-landmark sequence",
    "max_frames":MAX_FRAMES,
    "feature_dim":LANDMARK_DIM,
    "output":"English text",
    "decoder":"constrained greedy for runtime; beam only for qualitative analysis",
    "raw_video_stored":False,
})


## 17. Automatic go / no-go report

This section does not pretend a failing experiment is a success. It tells you exactly what the current run established.

**Interpretation rules:**

- GRU training loss should decrease without exploding while validation remains finite.
- Transformer should beat the trivial baseline on at least the primary translation metric(s) before being described as successful.
- Face ablation is interpretable because the mask is verified from the actual pose header.
- Text → Pose generation should be compared against retrieval rather than judged by MSE alone.
- The round-trip result is qualitative evidence only; it is not an independent benchmark.

In [50]:
## 17. Automatic GO / NO-GO report for the actual objective

# DIAGNOSIS: this cell previously read results_s2t["Transformer (full)"], a key that
# is never written -- the evaluation cell (Section 10) stores the primary model's
# metrics under "Transformer Sign→Text". That is a guaranteed KeyError at the end of
# every full run, right after the expensive test-set decode. `primary_metrics` is now
# the scalar computed in the evaluation cell instead of being re-derived here from a
# hand-typed string key.
transformer_beats_trivial=(
    primary_metrics["WER"] < results_s2t["Trivial baseline"]["WER"]
    or primary_metrics["BLEU"] > results_s2t["Trivial baseline"]["BLEU"]
)
# A single diversity threshold can pass while nearly half the outputs are still one
# dominant phrase (e.g. 60% unique but the other 40% is all "page number"). Guard on
# BOTH the overall unique-output fraction AND the top single prediction's frequency.
translation_not_collapsed = (tf_unique_frac >= 0.30) and (top_prediction_freq <= 0.20)

print("=== SignSage Sign→English status ===")
print("Full usable training rows:", f"{len(df_train):,}")
print("Full usable validation rows:", f"{len(df_val):,}")
print("Full usable test rows:", f"{len(df_test):,}")
print("Video-level leakage-free:", leakage_free)
print("Transformer beats trivial baseline:", transformer_beats_trivial)
print("Output diversity >= 30% and top-prediction share <= 20%:", translation_not_collapsed)
print("Test WER:", round(primary_metrics["WER"],4))
print("Test BLEU:", round(primary_metrics["BLEU"],2))
print("Test ROUGE-L:", round(primary_metrics["ROUGE-L"],4))
print(f"Top prediction {top_prediction!r} covers {top_prediction_freq:.2%} of test outputs")
print(f"Generated length (median words): {generated_length_stats['median']:.1f} "
      f"vs reference median: {reference_length_stats['median']:.1f}")

if transformer_beats_trivial and translation_not_collapsed:
    print("\nRESULT: model shows measurable translation signal without obvious near-fixed-output collapse.")
else:
    print("\nRESULT: NO-GO for deployment. Do not connect this checkpoint to the webcam backend yet.")
    if not transformer_beats_trivial: print(" - Translation metrics do not beat the trivial baseline.")
    if not translation_not_collapsed: print(" - Output diversity/top-prediction share indicates collapse; inspect data/model behavior.")

knowledge_observation("go_no_go",{
    "transformer_beats_trivial":bool(transformer_beats_trivial),
    "output_diversity_ok":bool(translation_not_collapsed),
    "top_prediction_freq":top_prediction_freq,
    "test_metrics":primary_metrics,
    "phase_0_status":"GO-CANDIDATE" if transformer_beats_trivial and translation_not_collapsed else "NO-GO",
})


=== SignSage Sign→English status ===
Full usable training rows: 99,699
Full usable validation rows: 11,927
Full usable test rows: 12,153
Video-level leakage-free: True
Transformer beats trivial baseline: True
Output diversity >= 30% and top-prediction share <= 20%: False
Test WER: 1.0046
Test BLEU: 0.24
Test ROUGE-L: 0.0891
Top prediction 'the police said that the government has been arrested.' covers 44.84% of test outputs
Generated length (median words): 9.0 vs reference median: 11.0

RESULT: NO-GO for deployment. Do not connect this checkpoint to the webcam backend yet.
 - Output diversity/top-prediction share indicates collapse; inspect data/model behavior.


In [51]:
# 18. Finalize experiment + deployment artifacts

# The backend must never need the training notebook itself. Export the exact
# vocabulary/configuration used by the primary Sign→English model.
#
# FIX (deployment manifest completeness, per backend audit): special_tokens
# was previously an ORDERED LIST ["<pad>","<bos>","<eos>","<unk>"] whose
# position does NOT match the real token IDs (real: pad=0, unk=1, bos=2,
# eos=3). Anyone inferring IDs from list position would get BOS/UNK/EOS
# wrong. Replaced with an explicit id dict. Also added: model_architecture
# (read back from the just-saved checkpoint itself, so this can never drift
# from what is actually in the .pt file), an explicit decoding config,
# normalization_version, a reference to landmark_contract.json, and the
# inference-package entrypoint.
_ckpt_for_manifest = torch.load(CHECKPOINT_DIR / "signsage_transformer_full.pt", map_location="cpu")
_model_architecture = {
    k: _ckpt_for_manifest["model_config"][k]
    for k in ("input_dim", "raw_feature_dim", "vocab_size", "d_model", "nhead", "layers", "ff", "dropout", "max_frames", "max_tokens")
    if k in _ckpt_for_manifest["model_config"]
}
_checkpoint_metadata = {
    "epoch": _ckpt_for_manifest.get("epoch"),
    "val_loss": _ckpt_for_manifest.get("val_loss"),
}
del _ckpt_for_manifest

deployment_config={
    "project":"SignSage",
    "task":"ISL_to_English",
    "dataset":"iSign v1.1",
    "feature_dim":int(LANDMARK_DIM),
    "model_input_dim":int(LANDMARK_DIM*2),
    "motion_features":"per-frame delta of normalized landmarks; first frame zero",
    "landmarks":543,
    "coordinates":3,
    "coordinate_order":["x","y","z"],
    "max_frames":int(MAX_FRAMES),
    "max_tokens":int(MAX_TOKENS),
    "vocab_size":int(len(vocab)),
    "tokenizer":"SentencePiece unigram trained on training targets only",
    "tokenizer_model":"tokenizer/signsage_en_sp.model",
    # Explicit ID mapping (NOT an ordered list -- list position previously did
    # not match real IDs). token_vocabulary.json has the full piece<->id map.
    "special_token_ids":{"pad":int(PAD_IDX),"unk":int(UNK_IDX),"bos":int(BOS_IDX),"eos":int(EOS_IDX)},
    "normalization_version":"pose_normalize_v1",
    "normalization":"root-center on body hips (23,24), divide by per-sequence body landmark std; NaN/Inf -> 0",
    "landmark_contract_file":"landmark_contract.json",
    "model_architecture":_model_architecture,
    "decoding":{
        "decoder_runtime":"constrained_greedy",
        "max_decode_tokens":int(MAX_DECODE_TOKENS),
        "min_output_tokens":int(MIN_OUTPUT_TOKENS),
        "beam_width_qualitative":int(BEAM_WIDTH),
        "repetition_penalty":float(REPETITION_PENALTY),
        "no_repeat_ngram_size":int(NO_REPEAT_NGRAM_SIZE),
    },
    "inference_package":{
        "name":"signsage_inference",
        "version":"1.0.0",
        "entrypoint":"signsage_inference.loader.load_pipeline",
        "bundle_relative_path":"inference/",
    },
    "checkpoint_metadata":_checkpoint_metadata,
    "raw_video_required_by_model":False,
    "training_rows":int(len(df_train)),
    "validation_rows":int(len(df_val)),
    "test_rows":int(len(df_test)),
    "video_level_split":True,
    "checkpoint":"checkpoint/signsage_transformer_full.pt",  # FIXED: must include the "checkpoint/" prefix Section 19 actually uses
}
with open(REPORT_DIR/"sign_to_text_deployment.json","w",encoding="utf-8") as f:
    json.dump(deployment_config,f,indent=2,ensure_ascii=False)
with open(REPORT_DIR/"token_vocabulary.json","w",encoding="utf-8") as f:
    json.dump({"token_to_id":vocab,"special_token_ids":deployment_config["special_token_ids"],"tokenizer_type":"sentencepiece_unigram","tokenizer_model":str(TOKENIZER_MODEL)},f,indent=2,ensure_ascii=False)

# Dedicated evaluation_results.json (previously only embedded inside the large
# knowledge manifest below) so the bundle carries a standalone results file.
evaluation_results={
    "sign_to_text_metrics":results_s2t,
    "output_diversity_fraction":float(tf_unique_frac),
    "top_prediction_freq":float(top_prediction_freq),
    "generated_length_stats":generated_length_stats,
    "reference_length_stats":reference_length_stats,
    "mean_confidence_proxy":float(tf_mean_conf),
    "p10_confidence_proxy":float(tf_p10_conf),
    "transformer_beats_trivial":bool(transformer_beats_trivial),
    "output_diversity_ok":bool(translation_not_collapsed),
    "phase_0_status":"GO-CANDIDATE" if transformer_beats_trivial and translation_not_collapsed else "NO-GO",
}
with open(REPORT_DIR/"evaluation_results.json","w",encoding="utf-8") as f:
    json.dump(evaluation_results,f,indent=2,ensure_ascii=False)

knowledge_artifact("sign_to_text_deployment","json",REPORT_DIR/"sign_to_text_deployment.json",
                   "Exact Sign→English preprocessing/model/decoder contract for backend integration.")
knowledge_artifact("tokenizer_model","model",TOKENIZER_MODEL,"SentencePiece model used for English target tokenization.")
knowledge_artifact("token_vocabulary","json",REPORT_DIR/"token_vocabulary.json",
                   "Vocabulary used by the primary Sign→English checkpoint.")
knowledge_artifact("evaluation_results","json",REPORT_DIR/"evaluation_results.json",
                   "Standalone held-out evaluation metrics/diagnostics for the primary checkpoint.")
knowledge_artifact("transformer_checkpoint","pt",CHECKPOINT_DIR/"signsage_transformer_full.pt",
                   "Primary full-feature Sign→English Transformer checkpoint.")

manifest={
    "project":"SignSage","repository_version":17,"run_id":RUN_ID,
    "started_at":RUN_STARTED_AT,"completed_at":pd.Timestamp.utcnow().isoformat(),
    "task":"ISL_to_English",
    "environment":{"data_mode":DATA_MODE,"run_mode":RUN_MODE,"device":str(DEVICE),
                   "gpu_count":int(torch.cuda.device_count()),"seed":SEED},
    "tokenizer":{"type":"sentencepiece_unigram","vocab_size":int(sp.get_piece_size()),"model":str(TOKENIZER_MODEL)},
    "dataset":{"name":"iSign v1.1","csv_rows":int(len(df)),
               "unique_uids":int(df.uid.nunique()),"unique_video_ids":int(df.video_id.nunique()),
               "selected_rows":int(len(df_train)+len(df_val)+len(df_test))},
    "split":{"train_rows":int(len(df_train)),"val_rows":int(len(df_val)),
             "test_rows":int(len(df_test)),"leakage_free":bool(leakage_free)},
    "configuration":deployment_config,
    "evaluation":evaluation_results,
    "status":{"transformer_beats_trivial":bool(transformer_beats_trivial),
              "phase_0_status":evaluation_results["phase_0_status"]},
}
with open(KNOWLEDGE_JSON_PATH,"w",encoding="utf-8") as f:
    json.dump(manifest,f,indent=2,ensure_ascii=False)

conn=_sql_connect()
conn.execute("UPDATE runs SET completed_at=? WHERE run_id=?",(manifest["completed_at"],RUN_ID))
conn.commit(); conn.close()
print("Experiment repository finalized.")
print("Deployment config:",REPORT_DIR/"sign_to_text_deployment.json")
print("Landmark contract:",REPORT_DIR/"landmark_contract.json")
print("Evaluation results:",REPORT_DIR/"evaluation_results.json")
print("Checkpoint:",CHECKPOINT_DIR/"signsage_transformer_full.pt")


Experiment repository finalized.
Deployment config: /kaggle/working/signsage_outputs/reports/sign_to_text_deployment.json
Landmark contract: /kaggle/working/signsage_outputs/reports/landmark_contract.json
Evaluation results: /kaggle/working/signsage_outputs/reports/evaluation_results.json
Checkpoint: /kaggle/working/signsage_outputs/checkpoints/signsage_transformer_full.pt


## 19. Kaggle → SignSage hand-off

The ONLY artifact the backend needs is `signsage_outputs/signsage_deployment_bundle.zip`
(equivalently, the unzipped `signsage_deployment_bundle/` directory), built by the cell
below. Do not hand-wire paths from `checkpoints/`/`reports/` directly -- those are the
notebook's own working directories; the bundle is the deployment-correct repackaging of
the subset the backend actually needs:

```
signsage_deployment_bundle/
├── checkpoint/signsage_transformer_full.pt
├── tokenizer/signsage_en_sp.model
├── inference/            (importable Python package -- see INTEGRATION_NOTE.txt)
├── landmark_contract.json
├── sign_to_text_deployment.json
├── token_vocabulary.json
├── evaluation_results.json
└── INTEGRATION_NOTE.txt
```

The web backend should load the checkpoint once at startup (via
`inference.loader.load_pipeline(bundle_dir)`) and perform inference only. It must not
reimplement normalization/feature ordering/decoding by hand -- use the shipped
`inference/` package, which reads its own paths from `sign_to_text_deployment.json`
(never hardcode a checkpoint/tokenizer path in backend code).

IMPORTANT:
- The notebook trains the model; it is not executed for every webcam request.
- The backend should not send raw video to the model.
- If the webcam window is below the activity/confidence gate, return "no sign detected"
  rather than fabricating English.
- Final live webcam behavior is validated only after the backend uses this exact bundle,
  confirmed end to end by Section 20's fresh-load verification.


In [52]:
# 19. Create a compact deployment bundle for the web application
#
# This is the ONLY output bundle needed later for backend integration.
# It deliberately excludes dataset files, cache files, training logs, and
# per-example predictions. The large Kaggle output ZIP is therefore unnecessary.
#
# FIX (backend audit, blocking issues #1/#2): the bundle now also ships
# landmark_contract.json (the authoritative feature-ordering contract) and
# the full signsage_inference/ Python package (as inference/) so the backend
# needs NO code beyond what is in this ZIP -- no copying code out of the
# notebook, no reimplementation, no dependency on the iSign dataset or any
# /kaggle/* path.
import shutil, zipfile, json
BUNDLE_DIR = OUTPUT_DIR / "signsage_deployment_bundle"
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

required_artifacts = [
    (CHECKPOINT_DIR / "signsage_transformer_full.pt", "checkpoint/signsage_transformer_full.pt"),
    (REPORT_DIR / "tokenizer" / "signsage_en_sp.model", "tokenizer/signsage_en_sp.model"),
    (REPORT_DIR / "sign_to_text_deployment.json", "sign_to_text_deployment.json"),
    (REPORT_DIR / "token_vocabulary.json", "token_vocabulary.json"),
    (REPORT_DIR / "landmark_contract.json", "landmark_contract.json"),
    (REPORT_DIR / "evaluation_results.json", "evaluation_results.json"),
    (KNOWLEDGE_JSON_PATH, "signsage_knowledge_manifest.json"),
]
missing_artifacts = [str(src) for src, _ in required_artifacts if not src.exists()]
if missing_artifacts:
    raise RuntimeError("Deployment bundle is missing required artifacts: " + ", ".join(missing_artifacts))

for src, rel in required_artifacts:
    dst = BUNDLE_DIR / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

# Copy the standalone inference package (Section 15a) into the bundle as
# inference/. This is real, importable code -- not documentation -- and is
# what makes the bundle independently usable by the backend.
if not INFERENCE_PKG_DIR.exists():
    raise RuntimeError(
        f"Inference package not found at {INFERENCE_PKG_DIR}; re-run Section 15a "
        "before building the deployment bundle."
    )
_inference_dst = BUNDLE_DIR / "inference"
if _inference_dst.exists():
    shutil.rmtree(_inference_dst)
shutil.copytree(INFERENCE_PKG_DIR, _inference_dst)

integration_note = BUNDLE_DIR / "INTEGRATION_NOTE.txt"
integration_note.write_text(
    "SignSage Sign->Text deployment bundle.\n"
    "\n"
    "This bundle is self-contained: the backend does NOT need the training\n"
    "notebook, the iSign dataset, pose_format, or any /kaggle/* path.\n"
    "\n"
    "Quick start (Python, e.g. inside your FastAPI backend):\n"
    "\n"
    "    import sys\n"
    "    sys.path.insert(0, '<path-to-this-bundle>')\n"
    "    from inference.loader import load_pipeline\n"
    "    pipeline = load_pipeline('<path-to-this-bundle>')\n"
    "    result = pipeline.transcribe(landmarks_xyz)   # (T, 543, 3) numpy array\n"
    "    print(result.text)\n"
    "\n"
    "Dependencies (see inference/requirements.txt): torch, numpy, sentencepiece.\n"
    "\n"
    "landmarks_xyz must be raw MediaPipe Holistic output -- 543 landmarks in\n"
    "the EXACT order documented in landmark_contract.json (component_order /\n"
    "component_ranges), x/y/z per landmark, NOT pre-normalized and NOT\n"
    "pre-computed deltas. The model receives RAW 1629-D landmark frames\n"
    "(543 landmarks x 3 coordinates) after prepare_live_landmarks(); it\n"
    "internally concatenates each frame with its frame-to-frame delta, so the\n"
    "model input is 3258-D. Do not compute the delta yourself.\n"
    "\n"
    "Files:\n"
    "  checkpoint/signsage_transformer_full.pt   trained weights + architecture config\n"
    "  tokenizer/signsage_en_sp.model            SentencePiece model (PAD=0,UNK=1,BOS=2,EOS=3)\n"
    "  inference/                                importable Python package (see above)\n"
    "  landmark_contract.json                    authoritative 543-landmark ordering\n"
    "  sign_to_text_deployment.json               full deployment manifest\n"
    "  token_vocabulary.json                      piece<->id vocabulary\n"
    "  evaluation_results.json                    held-out test-set metrics\n"
    "\n"
    "Do not upload the iSign dataset or training cache to the web application.\n",
    encoding="utf-8",
)

bundle_zip = OUTPUT_DIR / "signsage_deployment_bundle.zip"
with zipfile.ZipFile(bundle_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in BUNDLE_DIR.rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(BUNDLE_DIR))

bundle_size_mb = bundle_zip.stat().st_size / (1024**2)
print("Compact deployment bundle:", bundle_zip)
print(f"Bundle size: {bundle_size_mb:.1f} MB")
print("Contents:")
for p in sorted(BUNDLE_DIR.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(BUNDLE_DIR))
print("\nDo NOT use the full Kaggle output ZIP for web-app integration; use this bundle.")


Compact deployment bundle: /kaggle/working/signsage_outputs/signsage_deployment_bundle.zip
Bundle size: 106.6 MB
Contents:
 - INTEGRATION_NOTE.txt
 - checkpoint/signsage_transformer_full.pt
 - evaluation_results.json
 - inference/__init__.py
 - inference/__pycache__/__init__.cpython-312.pyc
 - inference/__pycache__/config.cpython-312.pyc
 - inference/__pycache__/decoder.cpython-312.pyc
 - inference/__pycache__/loader.cpython-312.pyc
 - inference/__pycache__/model.cpython-312.pyc
 - inference/__pycache__/preprocessing.cpython-312.pyc
 - inference/config.py
 - inference/decoder.py
 - inference/loader.py
 - inference/model.py
 - inference/preprocessing.py
 - inference/requirements.txt
 - landmark_contract.json
 - sign_to_text_deployment.json
 - signsage_knowledge_manifest.json
 - token_vocabulary.json
 - tokenizer/signsage_en_sp.model

Do NOT use the full Kaggle output ZIP for web-app integration; use this bundle.


## 20. Backend artifact verification -- isolated exported-bundle reload, exported files only


In [53]:
## 20. Backend artifact verification -- fresh process, exported files only

# FIX (BLOCKING #3): the old self-test used transformer_full/vocab/sp/
# prepare_live_landmarks still resident in this Python process from training.
# That proves nothing about whether the SHIPPED BUNDLE works on its own. This
# cell discards that shortcut: it imports ONLY the "inference" package that
# was just copied into BUNDLE_DIR, loads ONLY the files inside BUNDLE_DIR, and
# runs a real held-out sample through it end to end.
#
# FIX (review pass): every path below is resolved FROM THE MANIFEST, exactly
# as inference/loader.py does (bundle_dir / config.checkpoint_path) -- nothing
# here hardcodes "checkpoint/signsage_transformer_full.pt" independently.
# A hardcoded path would silently mask the exact class of bug this cell exists
# to catch (a manifest value that doesn't match where a file actually is).
# Checks execute in dependency order (manifest/contract must be parsed before
# a checkpoint/tokenizer path can be resolved from them), but are PRINTED in
# the requested report order at the end -- printing order is a presentation
# choice; execution order is a correctness requirement.

import sys, importlib, traceback

_verify_root = str(BUNDLE_DIR)
if _verify_root not in sys.path:
    sys.path.insert(0, _verify_root)
for _mod in list(sys.modules):
    if _mod == "inference" or _mod.startswith("inference."):
        del sys.modules[_mod]
importlib.invalidate_caches()

_checks = {}   # name -> (bool_passed, detail_str)
_bundled = {}  # scratch space for objects produced by earlier checks

def _run_check(name, fn):
    try:
        result = fn()
        _checks[name] = (True, "")
        return result
    except Exception as e:
        _checks[name] = (False, f"{type(e).__name__}: {e}")
        return None

# --- Manifest loading (must happen before any path below can be resolved) ---
def _check_manifest():
    path = BUNDLE_DIR / "sign_to_text_deployment.json"
    if not path.is_file():
        raise FileNotFoundError(f"Manifest missing: {path}")
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

_bundled["manifest"] = _run_check("Manifest loading", _check_manifest)

# --- Landmark contract ---------------------------------------------------
def _check_landmark_contract():
    path = BUNDLE_DIR / "landmark_contract.json"
    if not path.is_file():
        raise FileNotFoundError(f"Landmark contract missing: {path}")
    with open(path, "r", encoding="utf-8") as f:
        contract = json.load(f)
    required_keys = ["total_landmark_count", "raw_feature_dim", "coordinate_order",
                      "component_order", "component_ranges", "final_model_input_dim"]
    missing = [k for k in required_keys if k not in contract]
    if missing:
        raise ValueError(f"landmark_contract.json missing keys: {missing}")
    return contract

_bundled["contract"] = _run_check("Landmark contract", _check_landmark_contract)

# --- Checkpoint loading: path resolved FROM THE MANIFEST, not hardcoded ----
def _check_checkpoint():
    manifest = _bundled["manifest"]
    if manifest is None:
        raise RuntimeError("Cannot resolve checkpoint path -- manifest failed to load.")
    rel_path = manifest["checkpoint"]
    path = BUNDLE_DIR / rel_path
    if not path.is_file():
        raise FileNotFoundError(
            f"Checkpoint not found at manifest-declared path: {path} "
            f"(manifest['checkpoint']={rel_path!r})"
        )
    payload = torch.load(path, map_location="cpu")
    if "state_dict" not in payload or "model_config" not in payload:
        raise ValueError("Checkpoint payload missing state_dict/model_config.")
    return payload

_bundled["checkpoint_payload"] = _run_check("Checkpoint loading", _check_checkpoint)

# --- Tokenizer loading: path resolved FROM THE MANIFEST, not hardcoded -----
def _check_tokenizer():
    import sentencepiece as spm
    manifest = _bundled["manifest"]
    if manifest is None:
        raise RuntimeError("Cannot resolve tokenizer path -- manifest failed to load.")
    rel_path = manifest["tokenizer_model"]
    path = BUNDLE_DIR / rel_path
    if not path.is_file():
        raise FileNotFoundError(
            f"Tokenizer not found at manifest-declared path: {path} "
            f"(manifest['tokenizer_model']={rel_path!r})"
        )
    return spm.SentencePieceProcessor(model_file=str(path))

_bundled["sp"] = _run_check("Tokenizer loading", _check_tokenizer)

# --- Feature dimension: exercises inference.config.SignSageConfig.from_bundle,
# which validates total_landmarks==543, coordinates_per_landmark==3,
# raw_feature_dim==1629, model_input_dim==3258, full_feature_mask length==1629,
# normalization_version against the package's own constant, special-token-id
# distinctness, and cross-file (manifest vs contract) consistency -- all in
# one call, using the REAL exported config module, not a re-derived check. ---
def _check_feature_dimension():
    import inference.config as bcfg
    config = bcfg.SignSageConfig.from_bundle(BUNDLE_DIR)
    ckpt = _bundled["checkpoint_payload"]
    mc = ckpt["model_config"]
    if int(mc["raw_feature_dim"]) != config.raw_feature_dim:
        raise ValueError(f"checkpoint raw_feature_dim={mc['raw_feature_dim']} != config.raw_feature_dim={config.raw_feature_dim}")
    if int(mc["input_dim"]) != config.model_input_dim:
        raise ValueError(f"checkpoint input_dim={mc['input_dim']} != config.model_input_dim={config.model_input_dim}")
    return config

_bundled["config"] = _run_check("Feature dimension", _check_feature_dimension)

# --- Full checkpoint ↔ manifest architecture consistency ------------------
def _check_architecture_consistency():
    manifest = _bundled["manifest"]
    ckpt = _bundled["checkpoint_payload"]
    config = _bundled["config"]
    manifest_arch = manifest.get("model_architecture", {})
    checkpoint_arch = ckpt.get("model_config", {})
    if not manifest_arch:
        raise ValueError("Manifest is missing model_architecture.")

    int_fields = [
        "input_dim", "raw_feature_dim", "vocab_size",
        "d_model", "nhead", "layers", "ff",
        "max_frames", "max_tokens",
    ]
    for key in int_fields:
        if key in manifest_arch and key in checkpoint_arch:
            if int(checkpoint_arch[key]) != int(manifest_arch[key]):
                raise ValueError(
                    f"Checkpoint model_config[{key}]={checkpoint_arch[key]} "
                    f"!= manifest model_architecture[{key}]={manifest_arch[key]}"
                )

    if "dropout" in manifest_arch and "dropout" in checkpoint_arch:
        if abs(float(checkpoint_arch["dropout"]) - float(manifest_arch["dropout"])) > 1e-8:
            raise ValueError(
                f"Checkpoint model_config[dropout]={checkpoint_arch['dropout']} "
                f"!= manifest model_architecture[dropout]={manifest_arch['dropout']}"
            )

    # Also ensure the config reconstructed from the bundle agrees with the
    # same checkpoint dimensions used above.
    if int(config.model_input_dim) != int(checkpoint_arch["input_dim"]):
        raise ValueError("Bundle config model_input_dim disagrees with checkpoint input_dim.")
    if int(config.raw_feature_dim) != int(checkpoint_arch["raw_feature_dim"]):
        raise ValueError("Bundle config raw_feature_dim disagrees with checkpoint raw_feature_dim.")

    return True

_bundled["architecture"] = _run_check(
    "Model architecture consistency",
    _check_architecture_consistency,
)

# Fetch one real held-out sample (raw, unnormalized) for the remaining checks.
_verify_uid = str(df_test.iloc[min(1, len(df_test) - 1)]["uid"])
_verify_reference_text = normalize_text(str(df_test.iloc[min(1, len(df_test) - 1)]["text"]))
_verify_raw_landmarks = fetch_raw_live_style_landmarks(_verify_uid)

# --- Preprocessing (imported from the BUNDLED package, not this notebook) ---
def _check_preprocessing():
    import inference.preprocessing as bpp
    config = _bundled["config"]
    processed, true_len = bpp.prepare_live_landmarks(_verify_raw_landmarks, max_frames=config.max_frames)
    expected_shape = (config.max_frames, config.raw_feature_dim)
    if processed.shape != expected_shape:
        raise ValueError(f"prepare_live_landmarks shape {processed.shape} != expected {expected_shape}")
    if not np.isfinite(processed).all():
        raise ValueError("Preprocessed output contains non-finite values.")
    return processed, true_len

_pre_result = _run_check("Preprocessing", _check_preprocessing)
_bundled["processed"], _bundled["true_len"] = _pre_result if _pre_result is not None else (None, None)

# --- Model reconstruction (imported from the BUNDLED package) --------------
def _check_model_reconstruction():
    import inference.model as bmodel
    ckpt = _bundled["checkpoint_payload"]; config = _bundled["config"]
    mc = ckpt["model_config"]
    vocab_ckpt = ckpt.get("vocab")
    vocab_size = len(vocab_ckpt) if vocab_ckpt is not None else int(mc["vocab_size"])
    model = bmodel.SignTransformer(
        input_dim=int(mc["input_dim"]), vocab_size=vocab_size,
        max_frames=int(mc.get("max_frames", config.max_frames)),
        max_tokens=int(mc.get("max_tokens", config.max_tokens)),
        pad_idx=config.pad_id,
        d_model=int(mc.get("d_model", 192)), nhead=int(mc.get("nhead", 4)),
        layers=int(mc.get("layers", 3)), ff=int(mc.get("ff", 512)), dropout=0.0,
    )
    model.load_state_dict(ckpt["state_dict"], strict=True)
    model.eval()
    return model

_bundled["model"] = _run_check("Model reconstruction", _check_model_reconstruction)

# --- Autoregressive decoding (raw, ungated -- proves BOS-to-EOS generation) --
def _check_autoregressive_decoding():
    import inference.decoder as bdec
    config = _bundled["config"]; sp = _bundled["sp"]; model = _bundled["model"]
    dcfg = bdec.DecodingConfig(
        pad_id=config.pad_id, unk_id=config.unk_id, bos_id=config.bos_id, eos_id=config.eos_id,
        max_decode_tokens=config.max_decode_tokens, min_output_tokens=config.min_output_tokens,
        no_repeat_ngram_size=config.no_repeat_ngram_size, repetition_penalty=config.repetition_penalty,
    )
    device = torch.device("cpu")
    text, conf = bdec.greedy_decode(
        model, torch.from_numpy(_bundled["processed"]).float(), _bundled["true_len"],
        sp, dcfg, device, return_confidence=True,
    )
    if not np.isfinite(conf):
        raise ValueError("Decoding produced a non-finite confidence value.")
    return text, conf

_decode_result = _run_check("Autoregressive decoding", _check_autoregressive_decoding)
_raw_generated_text, _raw_generated_conf = _decode_result if _decode_result is not None else ("", 0.0)

# --- Held-out sample inference: the full public pipeline object, gated -----
def _check_full_pipeline():
    import inference.loader as bloader
    pipeline = bloader.load_pipeline(BUNDLE_DIR, device="cpu")
    return pipeline.transcribe(_verify_raw_landmarks)

_pipeline_result = _run_check("Held-out sample inference", _check_full_pipeline)

# --- English text generation -------------------------------------------------
def _check_text_generation():
    if not isinstance(_raw_generated_text, str):
        raise ValueError(f"Decoded output is not a string: {type(_raw_generated_text)}")
    return _raw_generated_text

_run_check("English text generation", _check_text_generation)

# --------------------------------------------------------------------------
# Report (printed in the requested display order; execution order above was
# dependency-correct, which is what matters for whether these checks mean
# anything).
# --------------------------------------------------------------------------
_report_order = [
    "Checkpoint loading", "Tokenizer loading", "Manifest loading", "Landmark contract",
    "Feature dimension", "Preprocessing", "Model reconstruction",
    "Autoregressive decoding", "Held-out sample inference", "English text generation",
]
print("=" * 60)
print("BACKEND ARTIFACT VERIFICATION")
print("=" * 60)
_all_passed = True
for name in _report_order:
    passed, detail = _checks.get(name, (False, "check did not run"))
    status = "PASS" if passed else "FAIL"
    if not passed:
        _all_passed = False
    print(f"{name:<28} : {status}" + (f"  ({detail})" if detail else ""))
print("=" * 60)
print(f"BACKEND ARTIFACT VERIFICATION: {'PASSED' if _all_passed else 'FAILED'}")
print("=" * 60)

print(f"\nHeld-out UID: {_verify_uid}")
print(f"Reference caption : {_verify_reference_text!r}")
print(f"Generated (raw, ungated) : {_raw_generated_text!r}  (confidence={_raw_generated_conf:.4f})")
if _pipeline_result is not None:
    print(f"Generated (gated pipeline.transcribe) : {_pipeline_result.text!r} "
          f"(accepted={_pipeline_result.accepted}, reason={_pipeline_result.reason})")

knowledge_observation("backend_artifact_verification", {
    "all_passed": bool(_all_passed),
    "checks": {k: v[0] for k, v in _checks.items()},
    "held_out_uid": _verify_uid,
    "reference_text": _verify_reference_text,
    "generated_text": _raw_generated_text,
    "generated_confidence": float(_raw_generated_conf),
})

if not _all_passed:
    raise RuntimeError(
        "BACKEND ARTIFACT VERIFICATION FAILED -- see the report above for which "
        "stage failed. Do NOT hand this bundle to the backend until every check "
        "passes."
    )


# Empty output is a model-quality diagnostic, not an artifact-integrity failure.
if _checks.get("English text generation", (False, ""))[0]:
    if isinstance(_raw_generated_text, str) and not _raw_generated_text.strip():
        print("WARNING: bundle loaded and decoded successfully, but generated text is empty.")


BACKEND ARTIFACT VERIFICATION
Checkpoint loading           : PASS
Tokenizer loading            : PASS
Manifest loading             : PASS
Landmark contract            : PASS
Feature dimension            : PASS
Preprocessing                : PASS
Model reconstruction         : PASS
Autoregressive decoding      : PASS
Held-out sample inference    : PASS
English text generation      : PASS
BACKEND ARTIFACT VERIFICATION: PASSED

Held-out UID: 62e79450f3da-2
Reference caption : 'unit 7. reside this poem.'
Generated (raw, ungated) : 'the picture of the words.'  (confidence=0.0961)
Generated (gated pipeline.transcribe) : 'the picture of the words.' (accepted=True, reason=ok)


In [54]:
# ============================================================
# 20. Final artifact inventory
# ============================================================
artifact_paths=sorted([p for p in OUTPUT_DIR.rglob("*") if p.is_file()])
print(f"Artifacts written: {len(artifact_paths)}")
for p in artifact_paths: print(f"{p.relative_to(OUTPUT_DIR)} — {p.stat().st_size/(1024**2):.2f} MB")
print("Notebook revision: FULL-CORPUS KAGGLE READY — archive-local batching, dynamic epoch budget, mandatory tiny-overfit gate, memory-safe evaluation/export.")


Artifacts written: 47
checkpoints/signsage_transformer_full.latest.pt — 115.22 MB
checkpoints/signsage_transformer_full.pt — 115.22 MB
checkpoints/signsage_transformer_recovery.pt — 115.22 MB
reports/evaluation_results.json — 0.00 MB
reports/landmark_contract.json — 0.02 MB
reports/sign_to_text_deployment.json — 0.00 MB
reports/signsage_inference/__init__.py — 0.00 MB
reports/signsage_inference/__pycache__/__init__.cpython-312.pyc — 0.00 MB
reports/signsage_inference/__pycache__/config.cpython-312.pyc — 0.01 MB
reports/signsage_inference/__pycache__/decoder.cpython-312.pyc — 0.01 MB
reports/signsage_inference/__pycache__/loader.cpython-312.pyc — 0.01 MB
reports/signsage_inference/__pycache__/model.cpython-312.pyc — 0.01 MB
reports/signsage_inference/__pycache__/preprocessing.cpython-312.pyc — 0.01 MB
reports/signsage_inference/config.py — 0.01 MB
reports/signsage_inference/decoder.py — 0.00 MB
reports/signsage_inference/loader.py — 0.01 MB
reports/signsage_inference/model.py — 0.00 MB
